In [1]:
deps_path = '/kaggle/input/datasets/nhhsag12/colpali-dependency'
!pip install --no-index --find-links {deps_path} --requirement {deps_path}/requirements.txt

Looking in links: /kaggle/input/datasets/nhhsag12/colpali-dependency
Processing /kaggle/input/datasets/nhhsag12/colpali-dependency/colpali_engine-0.3.15-py3-none-any.whl (from -r /kaggle/input/datasets/nhhsag12/colpali-dependency/requirements.txt (line 1))
Processing /kaggle/input/datasets/nhhsag12/colpali-dependency/colbert_ai-0.2.21-py3-none-any.whl (from -r /kaggle/input/datasets/nhhsag12/colpali-dependency/requirements.txt (line 5))
Processing /kaggle/input/datasets/nhhsag12/colpali-dependency/bitarray-3.8.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (from colbert-ai==0.2.21->-r /kaggle/input/datasets/nhhsag12/colpali-dependency/requirements.txt (line 5))
Processing /kaggle/input/datasets/nhhsag12/colpali-dependency/git_python-1.0.3-py2.py3-none-any.whl (from colbert-ai==0.2.21->-r /kaggle/input/datasets/nhhsag12/colpali-dependency/requirements.txt (line 5))
Processing /kaggle/input/datasets/nhhsag12/colpali-dependency/transformers-5.4.0-py3-no

In [2]:
!cat /kaggle/input/datasets/nhhsag12/colpali-dependency/requirements.txt


colpali_engine
torch
torchvision
transformers
colbert-ai==0.2.21

In [3]:
import torch

print(torch.__version__)
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))

2.10.0+cu128
NVIDIA RTX PRO 6000 Blackwell Server Edition
(12, 0)


In [4]:
# --- BƯỚC 1: SETUP DATA ---
import glob
import os
import pandas as pd
import numpy as np
import json
from tqdm.notebook import tqdm

# ==============================================================================
# CONFIG
# ==============================================================================
COLSMOL_DIR = "/kaggle/input/datasets/nguyenducdung1107/colqwen2layout/colqwen layout"
SEARCH_ROOT = "/kaggle/input"
ANNOTATIONS_PATH = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_annotations.jsonl"
PARQUET_PATH = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_layouts.parquet"

CURRENT_BATCH_IDX = 0  # ← Chỉnh batch ở đây

# ==============================================================================
# STEP 1: LOAD PKL FILES & BUILD BATCH RANGES
# ==============================================================================
pkl_files = sorted(glob.glob(os.path.join(COLSMOL_DIR, "colqwen2_fused_index *.pkl")))

print(f"Found {len(pkl_files)} PKL files")
for p in pkl_files:
    print(f"  {os.path.basename(p)}")

if len(pkl_files) == 0:
    raise ValueError("❌ No PKL files found. Check COLSMOL_DIR path!")

BATCH_RANGES = []

for pkl_file in pkl_files:
    base = os.path.basename(pkl_file).replace('.pkl', '')
    # base = "colsmol_fused_index 100-125"
    try:
        nums_part = base.split(' ')[-1]   # "100-125"
        start, end = map(int, nums_part.split('-'))
        BATCH_RANGES.append((start, end))
    except Exception as e:
        print(f"⚠️ Could not parse range from '{base}': {e}")

# Fallback nếu parse hoàn toàn thất bại
if len(BATCH_RANGES) == 0:
    print("⚠️ Using fallback batch indexing (sequential)")
    BATCH_RANGES = [(i, i + 1) for i in range(len(pkl_files))]

BATCH_RANGES = sorted(BATCH_RANGES, key=lambda x: x[0])

print(f"\n✅ Parsed {len(BATCH_RANGES)} batch ranges:")
for idx, (start, end) in enumerate(BATCH_RANGES):
    print(f"  Batch {idx:3d}: [{start:4d}:{end:4d}]")

# Validate index
if CURRENT_BATCH_IDX >= len(BATCH_RANGES):
    raise IndexError(
        f"❌ CURRENT_BATCH_IDX={CURRENT_BATCH_IDX} out of range. "
        f"Max is {len(BATCH_RANGES) - 1}"
    )

START_IDX, END_IDX = BATCH_RANGES[CURRENT_BATCH_IDX]
print(f"\n>>> Processing batch {CURRENT_BATCH_IDX}: [{START_IDX}:{END_IDX}]")

# ==============================================================================
# STEP 2: AUTO DETECT PATHS
# ==============================================================================
ENHANCED_JSONL_DIR = None
ENHANCED_IMG_DIR = None

for root, dirs, files in os.walk(SEARCH_ROOT):
    if "LAYOUT_CONTENT_FINAL" in root:
        ENHANCED_JSONL_DIR = root
    if "IMAGE ENHACED" in root or (sum(1 for f in files if f.endswith(".jpg")) > 1000):
        ENHANCED_IMG_DIR = root

# Fallback
if not ENHANCED_JSONL_DIR:
    ENHANCED_JSONL_DIR = "/kaggle/input/siglip-qwen-enhaced/SIGLIP_QWEN_ENHACED/LAYOUT_CONTENT_FINAL"

if not ENHANCED_IMG_DIR:
    ENHANCED_IMG_DIR = "/kaggle/input/siglip-qwen-enhaced/SIGLIP_QWEN_ENHACED/IMAGE ENHACED"

print(f"\nEnhanced JSONL DIR : {ENHANCED_JSONL_DIR}")
print(f"Enhanced IMG  DIR  : {ENHANCED_IMG_DIR}")

# ==============================================================================
# STEP 3: BUILD DOC MAPPING
# ==============================================================================
print("\nBuilding document mapping...")

valid_docs_in_qa = set()
with open(ANNOTATIONS_PATH, 'r') as f:
    for line in f:
        try:
            d = json.loads(line)
            valid_docs_in_qa.add(d['doc_name'].replace('.pdf', ''))
        except:
            pass

print(f"  QA docs loaded      : {len(valid_docs_in_qa)}")

available_jsonls = glob.glob(os.path.join(ENHANCED_JSONL_DIR, "*.jsonl"))
print(f"  JSONL files found   : {len(available_jsonls)}")

jsonl_map = {}
for p in available_jsonls:
    fname = os.path.basename(p).replace('_layout.jsonl', '')
    jsonl_map[fname] = p

intersection_docs = sorted(list(valid_docs_in_qa.intersection(jsonl_map.keys())))
print(f"  Intersection docs   : {len(intersection_docs)}")

if len(intersection_docs) == 0:
    print("\n⚠️  DEBUG - Sample QA doc names:")
    for x in list(valid_docs_in_qa)[:5]:
        print(f"    '{x}'")
    print("  Sample JSONL keys:")
    for x in list(jsonl_map.keys())[:5]:
        print(f"    '{x}'")
    raise ValueError(
        "❌ No intersection between QA docs and JSONL files. "
        "Check naming convention (strip suffix, prefix, etc.)"
    )

# ==============================================================================
# STEP 4: SELECT BATCH DOCS
# ==============================================================================
# Dùng START_IDX/END_IDX từ PKL filename để slice intersection_docs
target_doc_names = intersection_docs[START_IDX:END_IDX]
target_files = [jsonl_map[d] for d in target_doc_names if d in jsonl_map]

print(f"\nProcessing batch [{START_IDX}:{END_IDX}]")
print(f"  Target docs : {len(target_doc_names)}")
print(f"  Target files: {len(target_files)}")

if len(target_doc_names) == 0:
    raise ValueError(
        f"❌ Batch [{START_IDX}:{END_IDX}] is empty! "
        f"Total intersection docs = {len(intersection_docs)}. "
        f"Check CURRENT_BATCH_IDX."
    )

# ==============================================================================
# STEP 5: LOAD DATA
# ==============================================================================
print("\nLoading Parquet...")
df_orig = pd.read_parquet(PARQUET_PATH)
df_orig['join_doc_name'] = df_orig['doc_name'].str.replace('.pdf', '', regex=False)
df_orig = df_orig[df_orig['join_doc_name'].isin(target_doc_names)]
print(f"  Parquet rows (filtered): {len(df_orig)}")

print("Loading JSONL enrichment...")
dfs = []

for f in tqdm(target_files, desc="Reading JSONLs"):
    try:
        temp = pd.read_json(f, lines=True)
        temp['join_doc_name'] = os.path.basename(f).replace('_layout.jsonl', '')

        if 'layout' in temp.columns:
            temp = temp.rename(columns={'layout': 'layout_id'})

        cols = ['join_doc_name', 'layout_id', 'vlm_text', 'img_enhanced_path']
        if 'text_level' in temp.columns:
            cols.append('text_level')

        temp = temp[[c for c in cols if c in temp.columns]]
        dfs.append(temp)
    except Exception as e:
        print(f"  Skip {os.path.basename(f)}: {e}")

if len(dfs) > 0:
    df_enh = pd.concat(dfs, ignore_index=True)
    df_enh = df_enh.rename(columns={
        'vlm_text': 'vlm_text_enhanced',
        'text_level': 'text_level_enhanced'
    })
    print(f"  Enrichment rows: {len(df_enh)}")
else:
    df_enh = pd.DataFrame()
    print("  ⚠️ No enrichment data loaded")

# ==============================================================================
# STEP 6: MERGE
# ==============================================================================
print("\nMerging data...")
df_final = pd.merge(df_orig, df_enh, on=['join_doc_name', 'layout_id'], how='left')
df_final = df_final.sort_values(by=['join_doc_name', 'page_id', 'layout_id'])
print(f"  Merged rows: {len(df_final)}")

# ==============================================================================
# STEP 7: CONTEXT BUILDING
# ==============================================================================
def identify_header(row):
    if row.get('type') in ['title', 'section_header', 'header']:
        return str(row.get('text', ''))
    if pd.notna(row.get('text_level_enhanced')):
        return str(row.get('text', ''))
    return np.nan

df_final['temp_header'] = df_final.apply(identify_header, axis=1)
df_final['current_section'] = (
    df_final.groupby('join_doc_name')['temp_header']
    .ffill()
    .fillna("General Content")
)

# ==============================================================================
# STEP 8: IMAGE MAP
# ==============================================================================
enh_image_map = {}

if os.path.exists(ENHANCED_IMG_DIR):
    for f in glob.glob(os.path.join(ENHANCED_IMG_DIR, "*")):
        enh_image_map[os.path.basename(f)] = f
    print(f"\nEnhanced images found: {len(enh_image_map)}")
else:
    print(f"\n⚠️ Enhanced image dir not found: {ENHANCED_IMG_DIR}")

# ==============================================================================
# STEP 9: FINAL SOURCE SELECTION
# ==============================================================================
def get_best_sources(row):
    img_type, img_data = None, None

    if pd.notna(row.get('img_enhanced_path')):
        fname = os.path.basename(str(row['img_enhanced_path']))
        if fname in enh_image_map:
            img_type, img_data = 'path', enh_image_map[fname]

    if img_data is None and pd.notna(row.get('image_binary')):
        img_type, img_data = 'binary', row['image_binary']

    raw_content = ""
    if pd.notna(row.get('vlm_text_enhanced')) and len(str(row['vlm_text_enhanced'])) > 5:
        raw_content = str(row['vlm_text_enhanced'])
    elif pd.notna(row.get('text')) and len(str(row['text'])) > 5:
        raw_content = str(row['text'])
    elif pd.notna(row.get('ocr_text')) and len(str(row['ocr_text'])) > 5:
        raw_content = str(row['ocr_text'])
    elif pd.notna(row.get('vlm_text')):
        raw_content = str(row['vlm_text'])

    section = row.get('current_section', '')
    final_text_prompt = f"Section: {section}\nContent: {raw_content}"

    if len(final_text_prompt) < 10:
        final_text_prompt = "Document layout."

    return pd.Series(
        [img_type, img_data, final_text_prompt],
        index=['img_type', 'img_data', 'final_text']
    )

print("\nBuilding final dataset...")
processed = df_final.apply(get_best_sources, axis=1)

sample_layouts_df = (
    pd.concat([df_final, processed], axis=1)
    .dropna(subset=['img_type'])
    .reset_index(drop=True)
)

# ==============================================================================
# DONE
# ==============================================================================
print("=" * 60)
print(f"✅ BATCH {CURRENT_BATCH_IDX} [{START_IDX}-{END_IDX}] READY!")

print(f"   Total Layouts: {len(sample_layouts_df)}")
print("=" * 60)

Found 13 PKL files
  colqwen2_fused_index 0-25.pkl
  colqwen2_fused_index 100-125.pkl
  colqwen2_fused_index 125-150.pkl
  colqwen2_fused_index 150-175.pkl
  colqwen2_fused_index 175-200.pkl
  colqwen2_fused_index 200-225.pkl
  colqwen2_fused_index 225-250.pkl
  colqwen2_fused_index 25-50.pkl
  colqwen2_fused_index 250-275.pkl
  colqwen2_fused_index 275-300.pkl
  colqwen2_fused_index 300-313.pkl
  colqwen2_fused_index 50-75.pkl
  colqwen2_fused_index 75-100.pkl

✅ Parsed 13 batch ranges:
  Batch   0: [   0:  25]
  Batch   1: [  25:  50]
  Batch   2: [  50:  75]
  Batch   3: [  75: 100]
  Batch   4: [ 100: 125]
  Batch   5: [ 125: 150]
  Batch   6: [ 150: 175]
  Batch   7: [ 175: 200]
  Batch   8: [ 200: 225]
  Batch   9: [ 225: 250]
  Batch  10: [ 250: 275]
  Batch  11: [ 275: 300]
  Batch  12: [ 300: 313]

>>> Processing batch 0: [0:25]

Enhanced JSONL DIR : /kaggle/input/datasets/cdnghnam/siglip-qwen-enhaced/SIGLIP_QWEN_ENHACED/LAYOUT_CONTENT_FINAL
Enhanced IMG  DIR  : /kaggle/input/

Reading JSONLs:   0%|          | 0/25 [00:00<?, ?it/s]

  Enrichment rows: 6850

Merging data...
  Merged rows: 6850

Enhanced images found: 14901

Building final dataset...
✅ BATCH 0 [0-25] READY!
   Total Layouts: 6850


In [5]:
import transformers
print(transformers.__version__)

5.4.0


In [6]:

COLQWEN2_BASE   = "/kaggle/input/models/nhhsag12/colqwen2-v1-0-base/pytorch/default/1"
COLQWEN2_LORA   = "/kaggle/input/models/nhhsag12/colqwen2-v1-0/pytorch/default/2"


In [7]:
# ==============================================================================
# ColQwen2 & ColQwen2Processor — defined from scratch (no colpali_engine import)
# Sources provided by user.
# ==============================================================================
import sys
import os, sys

fake_path = "/tmp/colqwen2_notebook_fix.py"

if not os.path.exists(fake_path):
    with open(fake_path, "w") as f:
        f.write("# dummy file for transformers\n")

sys.modules["__main__"].__file__ = fake_path
import importlib
from abc import ABC, abstractmethod
from typing import ClassVar, List, Optional, Tuple, Union

import torch
from torch import nn
from PIL import Image
from transformers import BatchEncoding, BatchFeature
from transformers.models.qwen2_vl import Qwen2VLConfig, Qwen2VLModel, Qwen2VLProcessor
from transformers.models.qwen2_vl.image_processing_qwen2_vl import smart_resize


# ── Minimal device helper (replaces colpali_engine.utils.get_torch_device) ────
def _get_torch_device(device: str = "auto") -> torch.device:
    if device == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device)


# ==============================================================================
# BaseVisualRetrieverProcessor
# ==============================================================================

class BaseVisualRetrieverProcessor(ABC):
    """Base class for visual retriever processors."""

    query_prefix: ClassVar[str] = ""

    @abstractmethod
    def process_images(self, images: List[Image.Image]) -> Union[BatchFeature, BatchEncoding]:
        pass

    @abstractmethod
    def process_texts(self, texts: List[str]) -> Union[BatchFeature, BatchEncoding]:
        pass

    def process_queries(
        self,
        texts: Optional[List[str]] = None,
        queries: Optional[List[str]] = None,
        max_length: int = 50,
        contexts: Optional[List[str]] = None,
        suffix: Optional[str] = None,
    ) -> Union[BatchFeature, BatchEncoding]:
        if texts and queries:
            raise ValueError("Only one of 'texts' or 'queries' should be provided.")
        if queries is not None:
            texts = queries
        elif texts is None:
            raise ValueError("No texts or queries provided.")

        if suffix is None:
            suffix = self.query_augmentation_token * 10

        texts = [self.query_prefix + text + suffix for text in texts]
        return self.process_texts(texts=texts)

    @abstractmethod
    def score(
        self,
        qs: Union[torch.Tensor, List[torch.Tensor]],
        ps: Union[torch.Tensor, List[torch.Tensor]],
        device: Optional[Union[str, torch.device]] = None,
        **kwargs,
    ) -> torch.Tensor:
        pass

    @staticmethod
    def score_single_vector(
        qs: Union[torch.Tensor, List[torch.Tensor]],
        ps: Union[torch.Tensor, List[torch.Tensor]],
        device: Optional[Union[str, torch.device]] = None,
    ) -> torch.Tensor:
        device = device or _get_torch_device("auto")
        if isinstance(qs, list):
            qs = torch.stack(qs).to(device)
        else:
            qs = qs.to(device)
        if isinstance(ps, list):
            ps = torch.stack(ps).to(device)
        else:
            ps = ps.to(device)
        scores = torch.einsum("bd,cd->bc", qs, ps).to(torch.float32)
        return scores

    @staticmethod
    def score_multi_vector(
        qs: Union[torch.Tensor, List[torch.Tensor]],
        ps: Union[torch.Tensor, List[torch.Tensor]],
        batch_size: int = 128,
        device: Optional[Union[str, torch.device]] = None,
    ) -> torch.Tensor:
        device = device or _get_torch_device("auto")
        if len(qs) == 0:
            raise ValueError("No queries provided")
        if len(ps) == 0:
            raise ValueError("No passages provided")

        scores_list: List[torch.Tensor] = []
        for i in range(0, len(qs), batch_size):
            qs_batch = torch.nn.utils.rnn.pad_sequence(
                qs[i : i + batch_size], batch_first=True, padding_value=0
            ).to(device)
            scores_batch = []
            for j in range(0, len(ps), batch_size):
                ps_batch = torch.nn.utils.rnn.pad_sequence(
                    ps[j : j + batch_size], batch_first=True, padding_value=0
                ).to(device)
                scores_batch.append(
                    torch.einsum("bnd,csd->bcns", qs_batch, ps_batch).max(dim=3)[0].sum(dim=2)
                )
            scores_list.append(torch.cat(scores_batch, dim=1).cpu())

        return torch.cat(scores_list, dim=0).to(torch.float32)

    @abstractmethod
    def get_n_patches(
        self,
        image_size: Tuple[int, int],
        *args,
        **kwargs,
    ) -> Tuple[int, int]:
        pass


# ==============================================================================
# ColQwen2
# ==============================================================================

class ColQwen2(Qwen2VLModel):
    """
    ColQwen2 model implementation from the "ColPali: Efficient Document Retrieval with Vision
    Language Models" paper.

    Args:
        config (Qwen2VLConfig): The model configuration.
        mask_non_image_embeddings (bool): Whether to ignore all token embeddings except those
            of the image at inference. Defaults to False.
    """

    main_input_name: ClassVar[str] = "doc_input_ids"
    _checkpoint_conversion_mapping = {
        r"^base_model\.model\.custom_text_proj": "custom_text_proj",
        r"^model\.layers": "language_model.layers",
    }

    def __init__(self, config: Qwen2VLConfig, mask_non_image_embeddings: bool = False):
        super().__init__(config=config)

        hidden_size = getattr(self.config, "hidden_size", None)
        if hidden_size is None and hasattr(self.config, "text_config"):
            hidden_size = getattr(self.config.text_config, "hidden_size", None)
        if hidden_size is None:
            raise ValueError(
                f"Unable to determine text hidden size for {type(self.config).__name__}."
            )

        self.dim = 128
        self.custom_text_proj = nn.Linear(hidden_size, self.dim)
        self.padding_side = "left"
        self.mask_non_image_embeddings = mask_non_image_embeddings
        self.post_init()

    @classmethod
    def from_pretrained(cls, *args, **kwargs):
        key_mapping = kwargs.pop("key_mapping", None)
        if key_mapping is None:
            key_mapping = dict(getattr(super(), "_checkpoint_conversion_mapping", {}))
            key_mapping.update(cls._checkpoint_conversion_mapping)
        return super().from_pretrained(*args, **kwargs, key_mapping=key_mapping)

    def forward(self, *args, **kwargs) -> torch.Tensor:
        # Unpad pixel_values produced by ColQwen2Processor before passing to backbone
        if "pixel_values" in kwargs:
            offsets = kwargs["image_grid_thw"][:, 1] * kwargs["image_grid_thw"][:, 2]
            kwargs["pixel_values"] = torch.cat(
                [pv[:off] for pv, off in zip(kwargs["pixel_values"], offsets)],
                dim=0,
            )

        kwargs.pop("return_dict", True)
        kwargs.pop("output_hidden_states", None)
        kwargs.pop("use_cache", None)

        hidden_states = (
            super()
            .forward(*args, **kwargs, use_cache=False, output_hidden_states=True, return_dict=True)
            .last_hidden_state
        )  # (batch_size, sequence_length, hidden_size)

        proj = self.custom_text_proj(hidden_states)          # (B, S, dim)
        proj = proj / proj.norm(dim=-1, keepdim=True)        # L2 normalise
        proj = proj * kwargs["attention_mask"].unsqueeze(-1)  # zero-out padding

        if "pixel_values" in kwargs and self.mask_non_image_embeddings:
            image_mask = (kwargs["input_ids"] == self.config.image_token_id).unsqueeze(-1)
            proj = proj * image_mask

        return proj

    @property
    def patch_size(self) -> int:
        return self.visual.config.patch_size

    @property
    def spatial_merge_size(self) -> int:
        return self.visual.config.spatial_merge_size


# ==============================================================================
# ColQwen2Processor
# ==============================================================================

class ColQwen2Processor(BaseVisualRetrieverProcessor, Qwen2VLProcessor):
    """Processor for ColQwen2."""

    visual_prompt_prefix: ClassVar[str] = (
        "<|im_start|>user\n<|vision_start|><|image_pad|><|vision_end|>"
        "Describe the image.<|im_end|><|endoftext|>"
    )
    query_augmentation_token: ClassVar[str] = "<|endoftext|>"
    image_token: ClassVar[str] = "<|image_pad|>"

    def __init__(
        self,
        image_processor=None,
        tokenizer=None,
        video_processor=None,
        chat_template=None,
        **kwargs,
    ):
        super().__init__(
            image_processor=image_processor,
            tokenizer=tokenizer,
            video_processor=video_processor,
            chat_template=chat_template,
            **kwargs,
        )
        self.tokenizer.padding_side = "left"

    @classmethod
    def from_pretrained(cls, *args, device_map: Optional[str] = None, **kwargs):
        instance = super().from_pretrained(*args, device_map=device_map, **kwargs)
        if "max_num_visual_tokens" in kwargs:
            instance.image_processor.max_pixels = kwargs["max_num_visual_tokens"] * 28 * 28
            instance.image_processor.size["longest_edge"] = instance.image_processor.max_pixels
        return instance

    def process_images(self, images: List[Image.Image]) -> Union[BatchFeature, BatchEncoding]:
        images = [image.convert("RGB") for image in images]
        batch_doc = self(
            text=[self.visual_prompt_prefix] * len(images),
            images=images,
            padding="longest",
            return_tensors="pt",
        )
        # Unpad pixel_values so that DDP / multi-GPU works correctly
        offsets = batch_doc["image_grid_thw"][:, 1] * batch_doc["image_grid_thw"][:, 2]
        pixel_values = list(torch.split(batch_doc["pixel_values"], offsets.tolist()))
        batch_doc["pixel_values"] = torch.nn.utils.rnn.pad_sequence(
            pixel_values, batch_first=True
        )
        return batch_doc

    def process_texts(self, texts: List[str]) -> Union[BatchFeature, BatchEncoding]:
        return self(
            text=texts,
            return_tensors="pt",
            padding="longest",
        )

    def score(
        self,
        qs: List[torch.Tensor],
        ps: List[torch.Tensor],
        device: Optional[Union[str, torch.device]] = None,
        **kwargs,
    ) -> torch.Tensor:
        return self.score_multi_vector(qs, ps, device=device, **kwargs)

    def get_n_patches(
        self,
        image_size: Tuple[int, int],
        spatial_merge_size: int,
    ) -> Tuple[int, int]:
        patch_size = self.image_processor.patch_size
        height_new, width_new = smart_resize(
            width=image_size[0],
            height=image_size[1],
            factor=patch_size * self.image_processor.merge_size,
            min_pixels=self.image_processor.size["shortest_edge"],
            max_pixels=self.image_processor.size["longest_edge"],
        )
        n_patches_x = width_new // patch_size // spatial_merge_size
        n_patches_y = height_new // patch_size // spatial_merge_size
        return n_patches_x, n_patches_y

    def get_image_mask(self, batch_images: BatchFeature) -> torch.Tensor:
        return batch_images.input_ids == self.image_token_id


print("✅ ColQwen2 and ColQwen2Processor class definitions ready.")
# ==============================================================================
# Load ColQwen2 model & processor directly from a single local checkpoint
# (no LoRA / PeftModel needed — ColQwen2 is a single merged checkpoint)
# ==============================================================================

import gc
import torch
from peft import PeftModel

gc.collect()
torch.cuda.empty_cache()

print(">>> Loading ColQwen2 model...")
query_model = ColQwen2.from_pretrained(
    COLQWEN2_BASE,                    # path to the ColQwen2 checkpoint directory
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    attn_implementation="eager",     # required to expose attention weights
                                     # for attention-based methods (2 & 4)
)
query_model = PeftModel.from_pretrained(
    query_model,
    COLQWEN2_LORA
)
query_model.eval()

print(">>> Loading ColQwen2Processor...")
query_processor = ColQwen2Processor.from_pretrained(COLQWEN2_LORA)

print("✅ ColQwen2 ready")
print(f"   proj dim : {query_model.dim}")
print(f"   patch sz : {query_model.patch_size}")
print(f"   merge sz : {query_model.spatial_merge_size}")

✅ ColQwen2 and ColQwen2Processor class definitions ready.
>>> Loading ColQwen2 model...


Loading weights:   0%|          | 0/731 [00:00<?, ?it/s]

>>> Loading ColQwen2Processor...
✅ ColQwen2 ready
   proj dim : 128
   patch sz : 14
   merge sz : 2


In [8]:
import sys

# Create the dummy file so transformers can open and read it
_DUMMY_FILE = "/tmp/colqwen2_notebook_fix.py"
with open(_DUMMY_FILE, "w") as f:
    f.write("# dummy file for transformers experts-implementation check\n")

if not hasattr(sys.modules["__main__"], "__file__"):
    sys.modules["__main__"].__file__ = _DUMMY_FILE

In [9]:
BATCH_RANGES

[(0, 25),
 (25, 50),
 (50, 75),
 (75, 100),
 (100, 125),
 (125, 150),
 (150, 175),
 (175, 200),
 (200, 225),
 (225, 250),
 (250, 275),
 (275, 300),
 (300, 313)]

In [10]:

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    major, _ = torch.cuda.get_device_capability(0)
    RUNTIME_DTYPE = torch.bfloat16 if major >= 8 else torch.float16
else:
    RUNTIME_DTYPE = torch.float32


**1.OURS**

In [11]:
processor=query_processor
model=query_model

In [12]:
# ==============================================================================
# BƯỚC 3 v13 — ColQwen2 Full Ablation [FIXED]
#
# Pipeline:
#   - ColQwen2-specific extraction (Qwen2VLModel inner forward cho attention)
#   - SVD sink removal + softplus importance (giống v12 ColSMoL)
#   - ClusterPool cho discarded tokens
#   - Ablation: N_LAST_LAYERS ∈ {4,8,16,28} × norm ∈ {pre,post} × scoring ∈ {trad,weighted}
#   - TOPK_RATIOS: 0.1 … 0.9
#
# Fix từ v13 ColSMoL:
#   - pos_bbox_list lookup theo position 0..N-1 (fix Recall@10 = 0.0%)
#   - area-based Recall@K (recall_layout_area)
#   - hit_metrics trả thêm recall1/recall5/recall10
# ==============================================================================
'''
print(">>> BƯỚC 3 v13-ColQwen2 [Full Ablation — SVD+ClusterPool+AreaRecall] STARTING")

import torch
import torch.nn.functional as F
from transformers.models.qwen2_vl import Qwen2VLModel

# ==============================================================================
# CONFIG
# ==============================================================================
IMPORTANCE_VARIANT   = 'svd_sink_softplus_clusterPool_v13_ablation_colqwen2'
TEMPERATURE          = 0.5
MIN_IMPORTANCE_CLAMP = 1e-6
SVD_RANK_REMOVE      = 1
TOPK_RATIOS          = [round(i * 0.1, 1) for i in range(1, 10)]   # 0.1 … 0.9

# Ablation axes
N_LAST_LAYERS_LIST = [4, 8, 16, 28]
NORMALIZE_MODES    = ['pre', 'post']

MAX_N_LAYERS = max(N_LAST_LAYERS_LIST)

# PKL path override (giữ nguyên từ bản gốc ColQwen2)
BATCH_RANGE_PKL_OVERRIDE = {
    (0, 25): "/kaggle/input/datasets/nguyenducdung1107/vvvvvvv/colqwen2_fused_index (2).pkl",
}

def _make_ablation_keys():
    keys = ['traditional', 'trad_weighted']
    for n in N_LAST_LAYERS_LIST:
        for mode in NORMALIZE_MODES:
            for r in TOPK_RATIOS:
                tag = f"L{n}_norm{mode}_r{int(r*100)}"
                keys.append(f"{tag}_trad")
                keys.append(f"{tag}_weighted")
    return keys

ABLATION_KEYS = _make_ablation_keys()
print(f"  Ablation keys ({len(ABLATION_KEYS)}): {ABLATION_KEYS[:6]} ...")


# ==============================================================================
# UTILS
# ==============================================================================
def build_content_mask(inputs, processor):
    attn_mask = inputs["attention_mask"]
    input_ids = inputs.get("input_ids", None)
    if input_ids is None:
        return attn_mask.float()

    special_ids = set()
    tok = getattr(processor, 'tokenizer', processor)
    for attr in ['pad_token_id', 'bos_token_id', 'eos_token_id',
                 'unk_token_id', 'sep_token_id', 'cls_token_id']:
        tid = getattr(tok, attr, None)
        if tid is not None:
            special_ids.add(int(tid))
    if hasattr(tok, 'added_tokens_encoder'):
        for _, tid in tok.added_tokens_encoder.items():
            special_ids.add(int(tid))
    if not special_ids:
        return attn_mask.float()

    special_tensor = torch.tensor(list(special_ids), device=input_ids.device)
    is_special     = (input_ids.unsqueeze(-1) == special_tensor).any(dim=-1)
    return attn_mask.float() * (~is_special).float()


# ==============================================================================
# BBOX UTILS
# ==============================================================================
def _parse_bbox(raw):
    """Parse bbox → [v0,v1,v2,v3] float hoặc None."""
    if raw is None:
        return None
    try:
        import numpy as np
        if isinstance(raw, np.ndarray):
            flat = raw.flatten()
            if len(flat) == 4:
                return [float(x) for x in flat]
            return None
    except Exception:
        pass
    if isinstance(raw, (list, tuple)) and len(raw) == 4:
        try:
            return [float(x) for x in raw]
        except Exception:
            return None
    if isinstance(raw, dict):
        for keys in [('x1','y1','x2','y2'), ('top','left','bottom','right'),
                     ('left','top','right','bottom')]:
            if all(k in raw for k in keys):
                try:
                    return [float(raw[k]) for k in keys]
                except Exception:
                    pass
    return None


# ==============================================================================
# AREA-BASED RECALL
# ==============================================================================
def calculate_overlap_area(bbox1, bbox2):
    t1, l1, b1, r1 = bbox1
    t2, l2, b2, r2 = bbox2
    inter_t = max(t1, t2); inter_l = max(l1, l2)
    inter_b = min(b1, b2); inter_r = min(r1, r2)
    if inter_t < inter_b and inter_l < inter_r:
        return (inter_b - inter_t) * (inter_r - inter_l)
    return 0.0


def recall_layout_area(top_k_positions, pos_bbox_list, layout_mapping_raw):
    """
    top_k_positions : list[int]  — vị trí 0..n_docs-1 từ torch.topk
    pos_bbox_list   : list       — mỗi phần tử là (page_id, t, l, b, r) hoặc None
    layout_mapping_raw: list[dict] — annotation gốc [{"page":p,"bbox":[t,l,b,r]},...]
    """
    recall_area = 0.0
    for pos in top_k_positions:
        info = pos_bbox_list[pos] if 0 <= pos < len(pos_bbox_list) else None
        if info is None:
            continue
        page_id, t, l, b, r = info
        for gt in layout_mapping_raw:
            if page_id != gt["page"]:
                continue
            gt_bbox = _parse_bbox(gt.get("bbox"))
            if gt_bbox is None:
                continue
            recall_area += calculate_overlap_area([t, l, b, r], gt_bbox)

    gt_area = 0.0
    for gt in layout_mapping_raw:
        gt_bbox = _parse_bbox(gt.get("bbox"))
        if gt_bbox is None:
            continue
        t2, l2, b2, r2 = gt_bbox
        gt_area += max(0.0, b2 - t2) * max(0.0, r2 - l2)

    if gt_area <= 0.0:
        return 0.0
    return min(recall_area / gt_area, 1.0)


# ==============================================================================
# SVD CORE
# ==============================================================================
def remove_sink_components_batch(attn_heads, k):
    try:
        U, S, Vh = torch.linalg.svd(attn_heads, full_matrices=False)
        k    = min(k, S.shape[-1])
        sink = (U[..., :k] * S[:, :k].unsqueeze(1)) @ Vh[:, :k, :]
        return (attn_heads - sink).clamp(min=0.0)
    except Exception:
        return attn_heads


# ==============================================================================
# IMPORTANCE COMPUTATION — SOFTPLUS
# ==============================================================================
def compute_svd_importance_softplus(attentions, content_mask, layer_weights, k):
    device     = content_mask.device
    B, S       = content_mask.shape
    importance = torch.zeros(B, S, device=device)

    for i, attn in enumerate(attentions):
        attn = attn.float().to(device)
        if attn.dim() == 4:
            B_, H, Sq, Sk = attn.shape
        else:
            attn = attn.unsqueeze(1)
            B_, H, Sq, Sk = attn.shape

        attn_flat = attn.view(B_ * H, Sq, Sk)
        cleaned   = remove_sink_components_batch(attn_flat, k)
        cleaned   = cleaned.view(B_, H, Sq, Sk)

        layer_imp = cleaned.sum(dim=2).mean(dim=1)
        # handle sequence length mismatch (vision tokens, etc.)
        if layer_imp.shape[1] != S:
            if layer_imp.shape[1] > S:
                layer_imp = layer_imp[:, :S]
            else:
                pad = torch.zeros(B_, S - layer_imp.shape[1], device=device)
                layer_imp = torch.cat([layer_imp, pad], dim=1)

        layer_imp = layer_imp * content_mask
        layer_imp = layer_imp / layer_imp.max(dim=-1, keepdim=True).values.clamp(min=1e-8)
        importance = importance + layer_weights[i] * layer_imp

    importance = importance * content_mask
    importance = F.softplus(importance / TEMPERATURE)
    importance = importance * content_mask
    return importance


# ==============================================================================
# DOC MATRIX BUILDER
# ==============================================================================
def _coerce_to_tensor(item):
    if isinstance(item, torch.Tensor):
        return item
    if isinstance(item, dict):
        for key in ('embedding', 'embeddings', 'vector', 'vectors', 'token_embeddings'):
            if key in item:
                v = item[key]
                if isinstance(v, torch.Tensor):
                    return v
        tensor_vals = [(k, v) for k, v in item.items() if isinstance(v, torch.Tensor)]
        if len(tensor_vals) == 1:
            return tensor_vals[0][1]
        if tensor_vals:
            return tensor_vals[0][1]
    raise ValueError(
        f"Cannot extract Tensor from fused_embeddings item of type {type(item)}. "
        f"Keys: {list(item.keys()) if isinstance(item, dict) else 'N/A'}"
    )


def build_doc_matrix(docs_emb, device):
    tensors = []
    for i, item in enumerate(docs_emb):
        try:
            t = _coerce_to_tensor(item)
        except ValueError as e:
            raise ValueError(f"docs_emb[{i}]: {e}") from e
        if t.dim() == 1:
            t = t.unsqueeze(0)
        elif t.dim() > 2:
            t = t.squeeze(0)
        tensors.append(t)

    n_docs  = len(tensors)
    max_len = max(t.shape[0] for t in tensors)
    D       = tensors[0].shape[1]

    doc_matrix = torch.zeros(n_docs, max_len, D, device=device)
    doc_mask   = torch.zeros(n_docs, max_len,    device=device, dtype=torch.bool)

    for i, t in enumerate(tensors):
        L = t.shape[0]
        doc_matrix[i, :L] = F.normalize(t.float().to(device), dim=-1)
        doc_mask[i, :L]   = True

    return doc_matrix, doc_mask


# ==============================================================================
# FAST MAXSIM
# ==============================================================================
@torch.no_grad()
def fast_maxsim(q_norm, doc_matrix, doc_mask):
    sim = torch.einsum('qd,nld->qnl', q_norm, doc_matrix)
    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
    return sim.max(dim=-1).values   # (S_q, n_docs)


# ==============================================================================
# CLUSTER-POOL BUILDER
# ==============================================================================
def build_cluster_pool_scores(
    q_norm_kept, q_norm_disc,
    imp_kept, imp_disc,
    doc_matrix, doc_mask,
    normalize_mode='pre',
):
    n_docs = doc_matrix.shape[0]

    if q_norm_disc.shape[0] == 0:
        return torch.zeros(n_docs, device=doc_matrix.device), 0.0

    sim_assign  = torch.mm(q_norm_disc, q_norm_kept.t())   # [P, K]
    cluster_ids = sim_assign.argmax(dim=-1)                 # [P]

    cluster_scores = torch.zeros(n_docs, device=doc_matrix.device)
    total_disc_imp = imp_disc.sum().item()

    for c in cluster_ids.unique():
        members = (cluster_ids == c).nonzero(as_tuple=True)[0]
        w       = imp_disc[members]
        w_sum   = w.sum().clamp(min=1e-8)
        w_norm  = w / w_sum
        pool_vec = (q_norm_disc[members] * w_norm.unsqueeze(-1)).sum(dim=0)  # [D]

        if normalize_mode == 'pre':
            pool_vec = F.normalize(pool_vec.unsqueeze(0), dim=-1)
            sim = torch.einsum('qd,nld->qnl', pool_vec, doc_matrix)
            sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
            ms = sim.max(dim=-1).values.squeeze(0)
            cluster_scores += ms * w_sum.item()
        else:  # post
            pool_vec_u = pool_vec.unsqueeze(0)
            sim = torch.einsum('qd,nld->qnl', pool_vec_u, doc_matrix)
            sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
            ms = sim.max(dim=-1).values.squeeze(0)
            cluster_scores += ms * w_sum.item()

    if total_disc_imp > 1e-8:
        cluster_scores = cluster_scores / total_disc_imp

    return cluster_scores, total_disc_imp


# ==============================================================================
# EXTRACTION FUNCTION — ColQwen2 specific
# (giữ nguyên cách lấy attention qua Qwen2VLModel inner forward)
# ==============================================================================
_QWEN2VL_FORWARD_ALLOWED = {
    'input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw',
    'video_grid_thw', 'pixel_values_videos', 'position_ids',
    'past_key_values', 'inputs_embeds', 'use_cache',
    'output_attentions', 'output_hidden_states', 'return_dict',
}
_COLQWEN2_FORWARD_STRIP = {
    'output_hidden_states', 'return_dict', 'use_cache', 'output_attentions'
}


def extract_embeddings_and_importance_colqwen2(model, inputs, processor, n_last_layers=16):
    colqwen2_inputs = {k: v for k, v in inputs.items()
                       if k not in _COLQWEN2_FORWARD_STRIP}
    inner_inputs = {k: v for k, v in inputs.items()
                    if k in _QWEN2VL_FORWARD_ALLOWED
                    and k not in _COLQWEN2_FORWARD_STRIP}

    try:
        with torch.no_grad():
            proj = model(**colqwen2_inputs)
            inner_out_attn = Qwen2VLModel.forward(
                model,
                **inner_inputs,
                output_attentions=True,
                output_hidden_states=False,
                use_cache=False,
                return_dict=True,
                attn_implementation="eager",
            )
            attn_list_full = list(inner_out_attn.attentions)

        content_mask = build_content_mask(inputs, processor).to(proj.device)

        attn_list = attn_list_full[-n_last_layers:] if len(attn_list_full) >= n_last_layers \
                    else attn_list_full
        n_actual  = len(attn_list)

        layer_weights = torch.exp(
            torch.linspace(0, 1, max(n_actual, 1), device=proj.device)
        )
        layer_weights /= layer_weights.sum()

        if n_actual > 0:
            importance = compute_svd_importance_softplus(
                attn_list, content_mask, layer_weights, SVD_RANK_REMOVE
            )
        else:
            importance = content_mask.clone()

        importance = importance * content_mask
        return proj, importance, (n_actual > 0)

    except Exception as e:
        print(f"❌ ColQwen2 extraction error: {e}")
        import traceback; traceback.print_exc()
        with torch.no_grad():
            proj = model(**colqwen2_inputs)
        content_mask = build_content_mask(inputs, processor).to(proj.device)
        importance   = content_mask.clone()
        return proj, importance, False


print(">>> BƯỚC 3 v13-ColQwen2 core functions READY")


# ==============================================================================
# METRIC FUNCTIONS
# ==============================================================================
import numpy as np

def compute_ndcg(ranked_indices, gt_set, k):
    dcg  = sum(1.0 / np.log2(r + 2) for r, idx in enumerate(ranked_indices[:k]) if idx in gt_set)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt_set), k)))
    return dcg / idcg if idcg > 0 else 0.0

def first_hit(top_k, gt_set):
    for r, idx in enumerate(top_k):
        if idx in gt_set:
            return r + 1
    return -1

def pretty_token(t):
    t = str(t).replace("\n", "\\n").replace(" ", "_")
    return t if len(t) <= 24 else t[:22] + ".."

def hit_metrics(top10_pos, gt_pos_set, pos_bbox_list, layout_mapping_raw):
    """
    top10_pos        : list[int]   — positions 0..n_docs-1 từ torch.topk
    gt_pos_set       : set[int]    — positions của GT layouts (IoU>0.5)
    pos_bbox_list    : list        — (page_id,t,l,b,r) hoặc None theo đúng thứ tự fused_embeddings
    layout_mapping_raw: list[dict] — annotation gốc → area-based Recall@K
    """
    if not layout_mapping_raw:
        return None
    h = first_hit(top10_pos, gt_pos_set)
    return {
        'r1':  int(h != -1 and h <= 1),
        'r5':  int(h != -1 and h <= 5),
        'r10': int(h != -1 and h <= 10),
        # Area-based Recall@K
        'recall1':  recall_layout_area(top10_pos[:1],  pos_bbox_list, layout_mapping_raw),
        'recall5':  recall_layout_area(top10_pos[:5],  pos_bbox_list, layout_mapping_raw),
        'recall10': recall_layout_area(top10_pos[:10], pos_bbox_list, layout_mapping_raw),
        'n1':  float(compute_ndcg(top10_pos, gt_pos_set, 1)),
        'n5':  float(compute_ndcg(top10_pos, gt_pos_set, 5)),
        'n10': float(compute_ndcg(top10_pos, gt_pos_set, 10)),
    }


# ==============================================================================
# METRIC STORE
# ==============================================================================
def _init_metric():
    return {'r1':0,'r5':0,'r10':0,'n1':0.,'n5':0.,'n10':0.,
            'recall1':0.,'recall5':0.,'recall10':0.,'count':0}

def _add_metric(dst, src):
    for f in ('r1','r5','r10'):
        dst[f] += int(src[f])
    for f in ('n1','n5','n10','recall1','recall5','recall10'):
        dst[f] += float(src[f])
    dst['count'] += 1

def _ensure(store, key):
    if key not in store: store[key] = _init_metric()
    return store[key]

def record(key, m, domain):
    _add_metric(_ensure(all_metrics, key), m)
    _add_metric(_ensure(all_domain_metrics[domain], key), m)


# ==============================================================================
# LATENCY TRACKER
# ==============================================================================
class LatencyTracker:
    def __init__(self, method_name: str):
        self.name = method_name
        self.ratio_ms = {}

    def add_ratio(self, ratio: float, score_ms: float):
        if ratio not in self.ratio_ms:
            self.ratio_ms[ratio] = []
        self.ratio_ms[ratio].append(float(score_ms))

    def report(self):
        if not self.ratio_ms:
            print(f"[{self.name}] No latency data collected.")
            return
        print(f"\n{'='*70}")
        print(f"Latency Report — {self.name}")
        print(f"  {'Ratio':<10} {'n':>6} {'avg ms':>10} {'p50 ms':>10} {'p95 ms':>10}")
        print(f"  {'-'*50}")
        for ratio in sorted(self.ratio_ms.keys()):
            vals = self.ratio_ms[ratio]
            n    = len(vals)
            avg  = np.mean(vals)
            p50  = np.percentile(vals, 50)
            p95  = np.percentile(vals, 95)
            print(f"  {ratio:<10.0%} {n:>6} {avg:>10.2f} {p50:>10.2f} {p95:>10.2f}")

    def to_dict(self):
        rows = []
        for ratio in sorted(self.ratio_ms.keys()):
            vals = self.ratio_ms[ratio]
            n    = len(vals)
            rows.append({
                'method':        self.name,
                'ratio':         ratio,
                'n_queries':     n,
                'avg_score_ms':  round(np.mean(vals), 3) if n else 0,
                'p50_score_ms':  round(np.percentile(vals, 50), 3) if n else 0,
                'p95_score_ms':  round(np.percentile(vals, 95), 3) if n else 0,
            })
        return rows


# ==============================================================================
# BƯỚC 4: BATCH EVALUATION
# ==============================================================================
print(">>> BƯỚC 4: Process All Batches [ColQwen2 v13-Full Ablation]")
print(f"  {len(ABLATION_KEYS)} ablation keys × {len(BATCH_RANGES)} batches")

import json, os, pickle, gc, glob
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm
import time

WORKING_DIR      = "/kaggle/working"
ANNOTATIONS_PATH = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_annotations.jsonl"
QUERY_BATCH_SIZE = 50

all_query_results  = []
all_token_results  = []
all_metrics        = {}
all_domain_metrics = {}
all_batch_stats    = []
latency_trackers   = {k: LatencyTracker(k) for k in ABLATION_KEYS}

print("\n" + "=" * 80)

# ==============================================================================
# BATCH LOOP
# ==============================================================================
for batch_num, (START_IDX, END_IDX) in enumerate(BATCH_RANGES, 1):
    print(f"\n[{batch_num}/{len(BATCH_RANGES)}] Batch [{START_IDX}:{END_IDX}]")

    # ── Resolve PKL path (giữ nguyên logic override từ bản gốc) ───────────────
    batch_key  = (START_IDX, END_IDX)
    index_path = BATCH_RANGE_PKL_OVERRIDE.get(batch_key, None)

    if index_path is not None:
        print(f"  ℹ️  PKL override: {index_path}")
    else:
        for _pkl in pkl_files:
            try:
                _nums = os.path.basename(_pkl).replace('.pkl', '').split(' ')[-1]
                _s, _e = map(int, _nums.split('-'))
                if _s == START_IDX and _e == END_IDX:
                    index_path = _pkl
                    break
            except Exception:
                continue

    if index_path is None:
        print(f"  ⚠️  No PKL found for [{START_IDX}:{END_IDX}]")
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}", 'status': 'index_not_found'})
        continue

    if not os.path.exists(index_path):
        print(f"  ⚠️  Index file missing: {index_path}")
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}", 'status': 'index_not_found'})
        continue

    # ── Load index ─────────────────────────────────────────────────────────────
    try:
        with open(index_path, 'rb') as f:
            saved = pickle.load(f)
        if isinstance(saved, list):
            fused_embeddings = saved
        elif isinstance(saved, dict):
            fused_embeddings = saved.get('embeddings', [])
        else:
            fused_embeddings = saved
        print(f"  ✅ Index: {len(fused_embeddings)} layouts")
    except Exception as e:
        print(f"  ❌ Load error: {e}"); continue

    # ── Load batch data ─────────────────────────────────────────────────────────
    print("  Loading batch data...")
    try:
        batch_doc_names    = intersection_docs[START_IDX:END_IDX]
        batch_target_files = [jsonl_map[d] for d in batch_doc_names]

        batch_df_orig = pd.read_parquet(PARQUET_PATH)
        batch_df_orig['join_doc_name'] = batch_df_orig['doc_name'].str.replace('.pdf','',regex=False)
        batch_df_orig = batch_df_orig[batch_df_orig['join_doc_name'].isin(batch_doc_names)]

        dfs = []
        for f in tqdm(batch_target_files, desc="    Reading JSONLs", leave=False):
            try:
                temp = pd.read_json(f, lines=True)
                temp['join_doc_name'] = os.path.basename(f).replace('_layout.jsonl','')
                if 'layout' in temp.columns:
                    temp = temp.rename(columns={'layout':'layout_id'})
                cols = ['join_doc_name','layout_id','vlm_text','img_enhanced_path']
                if 'text_level' in temp.columns: cols.append('text_level')
                temp = temp[[c for c in cols if c in temp.columns]]
                dfs.append(temp)
            except Exception: pass

        batch_df_enh = pd.concat(dfs, ignore_index=True).rename(
            columns={'vlm_text':'vlm_text_enhanced','text_level':'text_level_enhanced'}
        ) if dfs else pd.DataFrame()

        batch_df_final = pd.merge(batch_df_orig, batch_df_enh,
                                  on=['join_doc_name','layout_id'], how='left')
        batch_df_final = batch_df_final.sort_values(['join_doc_name','page_id','layout_id'])

        is_header_type = batch_df_final['type'].isin(['title','section_header','header'])
        has_text_level = batch_df_final.get('text_level_enhanced',
                                            pd.Series(dtype=object)).notna()
        batch_df_final['temp_header'] = batch_df_final['text'].where(is_header_type | has_text_level)
        batch_df_final['current_section'] = (
            batch_df_final.groupby('join_doc_name')['temp_header'].ffill().fillna("General Content")
        )

        enh_image_map = {}
        if os.path.exists(ENHANCED_IMG_DIR):
            for fn in glob.glob(os.path.join(ENHANCED_IMG_DIR, "*")):
                enh_image_map[os.path.basename(fn)] = fn

        df = batch_df_final.copy()
        if 'img_enhanced_path' in df.columns:
            df['img_data'] = df['img_enhanced_path'].dropna().map(
                lambda p: enh_image_map.get(os.path.basename(str(p))))
            df['img_type'] = df['img_data'].notna().map(lambda x: 'path' if x else None)
        else:
            df['img_data'] = None; df['img_type'] = None
        if 'image_binary' in df.columns:
            mask = df['img_type'].isna() & df['image_binary'].notna()
            df.loc[mask, 'img_type'] = 'binary'
            df.loc[mask, 'img_data'] = df.loc[mask, 'image_binary']

        def _pick_text(row):
            for col in ['vlm_text_enhanced','text','ocr_text','vlm_text']:
                v = row.get(col)
                if pd.notna(v) and len(str(v)) > 5: return str(v)
            return "Document layout."

        df['final_text'] = ("Section: " + df['current_section'].fillna('')
                            + " \n Content: " + df.apply(_pick_text, axis=1))

        # reset_index để position 0..N-1 khớp với fused_embeddings
        batch_sample_layouts_df = df.dropna(subset=['img_type']).reset_index(drop=True)
        print(f"  ✅ Batch data: {len(batch_sample_layouts_df)} layouts")

        # ── Build pos_bbox_list: list theo position 0..N-1 ──────────────────────
        col_page_global = 'page_idx' if 'page_idx' in batch_sample_layouts_df.columns else 'page_id'
        pos_bbox_list = []
        for i in range(len(batch_sample_layouts_df)):
            row = batch_sample_layouts_df.iloc[i]
            bbox = _parse_bbox(row.get('bbox'))
            if bbox is None:
                pos_bbox_list.append(None)
                continue
            try:
                page_id = int(pd.to_numeric(row[col_page_global], errors='coerce'))
                pos_bbox_list.append((page_id, *bbox))
            except Exception:
                pos_bbox_list.append(None)

        print(f"  ✅ pos_bbox_list: {sum(x is not None for x in pos_bbox_list)}"
              f"/{len(pos_bbox_list)} with valid bbox")

    except Exception as e:
        print(f"  ❌ Batch data error: {e}")
        import traceback; traceback.print_exc(); continue

    # ── Build QA pairs ──────────────────────────────────────────────────────────
    print("  Building QA pairs...")

    def calculate_iou(box1, box2):
        b1 = list(box1) if not isinstance(box1, list) else box1
        b2 = list(box2) if not isinstance(box2, list) else box2
        x1,y1 = max(b1[0],b2[0]), max(b1[1],b2[1])
        x2,y2 = min(b1[2],b2[2]), min(b1[3],b2[3])
        inter  = max(0,x2-x1)*max(0,y2-y1)
        union  = ((b1[2]-b1[0])*(b1[3]-b1[1])) + ((b2[2]-b2[0])*(b2[3]-b2[1])) - inter
        return inter/union if union > 0 else 0.0

    batch_qa_pairs   = []
    col_page_qa = 'page_idx' if 'page_idx' in batch_sample_layouts_df.columns else 'page_id'
    batch_doc_lookup = {k: v for k, v in batch_sample_layouts_df.groupby('join_doc_name')}
    batch_avail_docs = set(batch_doc_lookup.keys())

    with open(ANNOTATIONS_PATH, 'r') as f:
        for line in f:
            try: doc_data = json.loads(line)
            except Exception: continue

            target_doc = doc_data['doc_name'].replace('.pdf', '')
            if target_doc not in batch_avail_docs: continue

            doc_df   = batch_doc_lookup[target_doc]
            domain   = doc_data.get('domain', 'General')

            doc_df = doc_df.copy()
            doc_df['safe_page'] = (
                pd.to_numeric(doc_df[col_page_qa], errors='coerce').fillna(-999).astype(int)
            )

            for q_item in doc_data.get('questions', []):
                layout_mapping_raw = q_item.get('layout_mapping', [])
                gt_positions = []

                for target in layout_mapping_raw:
                    t_page = target['page']
                    t_bbox = _parse_bbox(target.get('bbox'))
                    if t_bbox is None: continue
                    try:
                        t_page_int = int(t_page)
                        cands = pd.concat([
                            doc_df[doc_df['safe_page'] == t_page_int],
                            doc_df[doc_df['safe_page'] == t_page_int - 1],
                        ])
                        for pos, row in cands.iterrows():
                            # pos = index của batch_sample_layouts_df (reset → 0..N-1)
                            row_bbox = _parse_bbox(row.get('bbox'))
                            if row_bbox and calculate_iou(row_bbox, t_bbox) > 0.5:
                                gt_positions.append(int(pos))
                    except Exception:
                        continue

                if layout_mapping_raw:
                    batch_qa_pairs.append({
                        'question':           q_item['Q'],
                        'gt_pos_set':         set(gt_positions),
                        'layout_mapping_raw': layout_mapping_raw,
                        'doc_name':           target_doc,
                        'domain':             domain,
                    })

    print(f"  ✅ {len(batch_qa_pairs)} QA pairs")
    if not batch_qa_pairs:
        print("  ⚠️  No QA pairs, skipping")
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}",
                                'n_layouts': len(fused_embeddings), 'n_qa': 0, 'status': 'no_qa'})
        continue

    # ── Evaluate ────────────────────────────────────────────────────────────────
    print(f"  Evaluating {len(batch_qa_pairs)} queries ...")
    doc_matrix = doc_mask = None
    batch_query_rows = []; batch_token_rows = []

    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        n_docs = len(fused_embeddings)
        total  = len(batch_qa_pairs)

        print("  Building doc matrix...")
        doc_matrix, doc_mask = build_doc_matrix(fused_embeddings, device)
        print(f"  ✅ Doc matrix: {doc_matrix.shape}")

        batch_metrics = {k: _init_metric() for k in ABLATION_KEYS}

        for qb_start in range(0, total, QUERY_BATCH_SIZE):
            qb_end = min(qb_start + QUERY_BATCH_SIZE, total)
            pbar = tqdm(enumerate(batch_qa_pairs[qb_start:qb_end]),
                        total=qb_end-qb_start, desc=f"    Q[{qb_start}:{qb_end}]", leave=False)

            for local_q_idx, item in pbar:
                global_q_idx       = qb_start + local_q_idx
                question           = item['question']
                gt_pos_set         = item['gt_pos_set']
                layout_mapping_raw = item['layout_mapping_raw']
                domain             = item['domain']

                if domain not in all_domain_metrics:
                    all_domain_metrics[domain] = {}
                if not layout_mapping_raw:
                    continue

                with torch.no_grad():
                    q_inputs = processor.process_queries([question]).to(device)

                attn_mask_1d    = q_inputs['attention_mask'][0].float()
                content_mask_1d = build_content_mask(q_inputs, processor)[0].float()

                trad_idx   = torch.where(attn_mask_1d > 0)[0]
                method_idx = torch.where(content_mask_1d > 0)[0]
                if trad_idx.numel() == 0: continue
                if method_idx.numel() == 0: method_idx = trad_idx

                # ── Run model once per n_last_layers — cache kết quả ─────────
                embed_cache = {}
                for n_layers in N_LAST_LAYERS_LIST:
                    proj, imp, _ = extract_embeddings_and_importance_colqwen2(
                        model, q_inputs, processor, n_last_layers=n_layers
                    )
                    embed_cache[n_layers] = (proj[0].float(), imp[0].float())

                # ── BASELINE: Traditional MaxSim ─────────────────────────────
                q_embed_base = embed_cache[N_LAST_LAYERS_LIST[0]][0]
                q_trad_norm  = F.normalize(q_embed_base[trad_idx].float(), dim=-1)

                t0 = time.perf_counter()
                M_trad      = fast_maxsim(q_trad_norm, doc_matrix, doc_mask)
                trad_scores = M_trad.sum(dim=0)
                trad_top10  = torch.topk(trad_scores, k=min(10, n_docs)).indices.cpu().tolist()
                trad_lat_ms = (time.perf_counter() - t0) * 1000.0

                trad_m = hit_metrics(trad_top10, gt_pos_set, pos_bbox_list, layout_mapping_raw)
                if trad_m:
                    latency_trackers['traditional'].add_ratio(1.0, trad_lat_ms)
                    _add_metric(batch_metrics['traditional'], trad_m)
                    record('traditional', trad_m, domain)

                # ── BASELINE: Traditional importance-weighted ─────────────────
                imp_base = embed_cache[N_LAST_LAYERS_LIST[0]][1]
                imp_trad = imp_base[trad_idx]

                t0 = time.perf_counter()
                imp_trad_n    = imp_trad / imp_trad.sum().clamp(min=1e-8)
                trad_w_scores = (M_trad * imp_trad_n.unsqueeze(-1)).sum(dim=0)
                trad_w_top10  = torch.topk(trad_w_scores, k=min(10, n_docs)).indices.cpu().tolist()
                trad_w_lat_ms = (time.perf_counter() - t0) * 1000.0

                trad_w_m = hit_metrics(trad_w_top10, gt_pos_set, pos_bbox_list, layout_mapping_raw)
                if trad_w_m:
                    latency_trackers['trad_weighted'].add_ratio(1.0, trad_w_lat_ms)
                    _add_metric(batch_metrics['trad_weighted'], trad_w_m)
                    record('trad_weighted', trad_w_m, domain)

                query_row = {
                    'batch':    f"{START_IDX}-{END_IDX}",
                    'query_id': global_q_idx,
                    'doc_name': item['doc_name'],
                    'domain':   domain,
                    'question': question,
                    'gt_count': len(gt_pos_set),
                }
                if trad_m:
                    query_row.update({
                        'trad_r@1':       trad_m['recall1'],   'trad_r@5':      trad_m['recall5'],
                        'trad_r@10':      trad_m['recall10'],
                        'trad_recall@1':  round(trad_m['recall1'],  4),
                        'trad_recall@5':  round(trad_m['recall5'],  4),
                        'trad_recall@10': round(trad_m['recall10'], 4),
                        'trad_ndcg@1':    round(trad_m['n1'],  4),
                        'trad_ndcg@5':    round(trad_m['n5'],  4),
                        'trad_ndcg@10':   round(trad_m['n10'], 4),
                        'trad_lat_ms':    round(trad_lat_ms,   4),
                    })
                if trad_w_m:
                    query_row.update({
                        'tradW_r@1':       trad_w_m['r1'],   'tradW_r@5':     trad_w_m['r5'],
                        'tradW_r@10':      trad_w_m['r10'],
                        'tradW_recall@1':  round(trad_w_m['recall1'],  4),
                        'tradW_recall@5':  round(trad_w_m['recall5'],  4),
                        'tradW_recall@10': round(trad_w_m['recall10'], 4),
                        'tradW_ndcg@1':    round(trad_w_m['n1'],  4),
                        'tradW_ndcg@5':    round(trad_w_m['n5'],  4),
                        'tradW_ndcg@10':   round(trad_w_m['n10'], 4),
                        'tradW_lat_ms':    round(trad_w_lat_ms,   4),
                    })

                # ── ABLATION LOOP: n_last_layers × normalize_mode × ratio × scoring ──
                for n_layers in N_LAST_LAYERS_LIST:
                    q_embed, q_importance = embed_cache[n_layers]

                    q_method_norm  = F.normalize(q_embed[method_idx].float(), dim=-1)
                    imp_valid      = q_importance[method_idx].float()
                    n_method       = method_idx.numel()

                    # argsort ngoài latency window (setup cost)
                    sorted_imp_idx = torch.argsort(imp_valid, descending=True)

                    for mode in NORMALIZE_MODES:
                        for r in TOPK_RATIOS:
                            tag    = f"L{n_layers}_norm{mode}_r{int(r*100)}"
                            n_keep = max(1, int(n_method * r))

                            kept_idx = sorted_imp_idx[:n_keep]
                            disc_idx = sorted_imp_idx[n_keep:]

                            q_kept   = q_method_norm[kept_idx]
                            q_disc   = q_method_norm[disc_idx]
                            imp_kept = imp_valid[kept_idx]
                            imp_disc = imp_valid[disc_idx]

                            # ---- SCORING MODE 1: Traditional sum ----
                            t0 = time.perf_counter()
                            M_kept = fast_maxsim(q_kept, doc_matrix, doc_mask)
                            cluster_scores, total_disc_imp = build_cluster_pool_scores(
                                q_kept, q_disc, imp_kept, imp_disc,
                                doc_matrix, doc_mask, normalize_mode=mode,
                            )
                            pool_frac   = total_disc_imp / imp_valid.sum().clamp(min=1e-8).item()
                            scores_trad = M_kept.sum(dim=0) + cluster_scores * pool_frac
                            top10_trad  = torch.topk(scores_trad, k=min(10, n_docs)).indices.cpu().tolist()
                            lat_trad_ms = (time.perf_counter() - t0) * 1000.0

                            m_trad   = hit_metrics(top10_trad, gt_pos_set, pos_bbox_list, layout_mapping_raw)
                            key_trad = f"{tag}_trad"
                            if m_trad:
                                latency_trackers[key_trad].add_ratio(r, lat_trad_ms)
                                _add_metric(batch_metrics[key_trad], m_trad)
                                record(key_trad, m_trad, domain)
                                query_row.update({
                                    f'{key_trad}_r@1':        m_trad['recall1'],
                                    f'{key_trad}_r@5':        m_trad['recall5'],
                                    f'{key_trad}_r@10':       m_trad['recall10'],
                                    f'{key_trad}_recall@1':   round(m_trad['recall1'],  4),
                                    f'{key_trad}_recall@5':   round(m_trad['recall5'],  4),
                                    f'{key_trad}_recall@10':  round(m_trad['recall10'], 4),
                                    f'{key_trad}_ndcg@1':     round(m_trad['n1'],  4),
                                    f'{key_trad}_ndcg@5':     round(m_trad['n5'],  4),
                                    f'{key_trad}_ndcg@10':    round(m_trad['n10'], 4),
                                    f'{key_trad}_lat_ms':     round(lat_trad_ms, 4),
                                })

                            # ---- SCORING MODE 2: Importance-weighted ----
                            t0 = time.perf_counter()
                            M_kept_w = fast_maxsim(q_kept, doc_matrix, doc_mask)
                            cluster_scores_w, total_disc_imp_w = build_cluster_pool_scores(
                                q_kept, q_disc, imp_kept, imp_disc,
                                doc_matrix, doc_mask, normalize_mode=mode,
                            )
                            pool_frac_w = total_disc_imp_w / imp_valid.sum().clamp(min=1e-8).item()
                            imp_kept_n  = imp_kept / imp_kept.sum().clamp(min=1e-8)
                            scores_wt   = (M_kept_w * imp_kept_n.unsqueeze(-1)).sum(dim=0)
                            scores_wt   = scores_wt + cluster_scores_w * pool_frac_w
                            top10_wt    = torch.topk(scores_wt, k=min(10, n_docs)).indices.cpu().tolist()
                            lat_wt_ms   = (time.perf_counter() - t0) * 1000.0

                            m_wt   = hit_metrics(top10_wt, gt_pos_set, pos_bbox_list, layout_mapping_raw)
                            key_wt = f"{tag}_weighted"
                            if m_wt:
                                latency_trackers[key_wt].add_ratio(r, lat_wt_ms)
                                _add_metric(batch_metrics[key_wt], m_wt)
                                record(key_wt, m_wt, domain)
                                query_row.update({
                                    f'{key_wt}_r@1':        m_wt['recall1'],
                                    f'{key_wt}_r@5':        m_wt['recall5'],
                                    f'{key_wt}_r@10':       m_wt['recall10'],
                                    f'{key_wt}_recall@1':   round(m_wt['recall1'],  4),
                                    f'{key_wt}_recall@5':   round(m_wt['recall5'],  4),
                                    f'{key_wt}_recall@10':  round(m_wt['recall10'], 4),
                                    f'{key_wt}_ndcg@1':     round(m_wt['n1'],  4),
                                    f'{key_wt}_ndcg@5':     round(m_wt['n5'],  4),
                                    f'{key_wt}_ndcg@10':    round(m_wt['n10'], 4),
                                    f'{key_wt}_lat_ms':     round(lat_wt_ms, 4),
                                })

                batch_query_rows.append(query_row)

                # ── Token-level rows (reference: L=N_LAST_LAYERS_LIST[0]) ──────
                q_imp_ref  = embed_cache[N_LAST_LAYERS_LIST[0]][1]
                q_emb_ref  = embed_cache[N_LAST_LAYERS_LIST[0]][0]
                q_ref_norm = F.normalize(q_emb_ref[method_idx].float(), dim=-1)
                M_ref      = fast_maxsim(q_ref_norm, doc_matrix, doc_mask)

                method_idx_np = method_idx.cpu().numpy()
                imp_np        = q_imp_ref[method_idx].cpu().numpy()
                input_ids     = q_inputs['input_ids'][0].cpu().tolist()
                tok_str_all   = processor.tokenizer.convert_ids_to_tokens(input_ids)
                best_score_np = M_ref.max(dim=1).values.cpu().numpy()

                for local_t, global_t in enumerate(method_idx_np):
                    tok_raw = tok_str_all[int(global_t)] if int(global_t) < len(tok_str_all) else "<UNK>"
                    batch_token_rows.append({
                        'batch':       f"{START_IDX}-{END_IDX}",
                        'query_id':    global_q_idx,
                        'token_idx':   int(global_t),
                        'token':       pretty_token(tok_raw),
                        'importance':  round(float(imp_np[local_t]),  8),
                        'maxsim_best': round(float(best_score_np[local_t]), 8),
                    })

            gc.collect(); torch.cuda.empty_cache()
            print(f"      ✓ Q[{qb_start}:{qb_end}] done")

        all_query_results.extend(batch_query_rows)
        all_token_results.extend(batch_token_rows)

        ref = batch_metrics.get('traditional', _init_metric())
        cnt = ref['count'] if ref['count'] > 0 else 1
        print(f"  ✅ {len(batch_query_rows)} Q | [traditional] "
              f"Hit@10={ref['r10']/cnt*100:.1f}%  "
              f"Recall@10={ref['recall10']/cnt*100:.1f}%  "
              f"nDCG@10={ref['n10']/cnt:.4f}")

        all_batch_stats.append({
            'batch':           f"{START_IDX}-{END_IDX}",
            'n_layouts':       len(fused_embeddings),
            'n_queries':       total,
            'trad_hit@10':     round(ref['r10']      / cnt * 100, 2),
            'trad_recall@10':  round(ref['recall10'] / cnt * 100, 2),
            'trad_ndcg@10':    round(ref['n10']      / cnt,       4),
            'status':          'ok',
        })

    except Exception as e:
        print(f"  ❌ Evaluation error: {e}")
        import traceback; traceback.print_exc()
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}", 'status': 'error', 'error': str(e)})

    # ── Save ────────────────────────────────────────────────────────────────────
    try:
        if batch_query_rows:
            pd.DataFrame(batch_query_rows).to_csv(
                os.path.join(WORKING_DIR, f"batch_{START_IDX}_{END_IDX}_queries.csv"), index=False)
        if batch_token_rows:
            pd.DataFrame(batch_token_rows).to_csv(
                os.path.join(WORKING_DIR, f"batch_{START_IDX}_{END_IDX}_tokens.csv"), index=False)
        print("  ✅ Batch files saved")
    except Exception as e:
        print(f"  ❌ Save error: {e}")

    # ── Cleanup ─────────────────────────────────────────────────────────────────
    if doc_matrix is not None: del doc_matrix
    if doc_mask   is not None: del doc_mask
    del fused_embeddings, batch_sample_layouts_df, batch_qa_pairs, pos_bbox_list
    del batch_df_orig, batch_df_enh, batch_df_final
    gc.collect(); torch.cuda.empty_cache()
    print("  ✓ Memory cleaned\n")


# ==============================================================================
# FINAL: CONSOLIDATE & PRINT
# ==============================================================================
print("=" * 130)
print("FINAL RESULTS — ColQwen2 v13 | SVD+ClusterPool | L×norm×scoring | AreaRecall")
print("=" * 130)

def _lat_mean_from_tracker(method_key):
    tracker = latency_trackers.get(method_key)
    if tracker is None or not tracker.ratio_ms:
        return float('nan')
    all_vals = [v for vals in tracker.ratio_ms.values() for v in vals]
    return float(np.mean(all_vals)) if all_vals else float('nan')

try:
    if all_query_results:
        df_all_q = pd.DataFrame(all_query_results)
        df_all_q.to_csv(os.path.join(WORKING_DIR, "MASTER_all_batches_queries.csv"), index=False)
        print(f"✅ {len(df_all_q)} query rows saved")
    if all_token_results:
        df_all_t = pd.DataFrame(all_token_results)
        df_all_t.to_csv(os.path.join(WORKING_DIR, "MASTER_all_batches_tokens.csv"), index=False)
        print(f"✅ {len(df_all_t)} token rows saved")

    SEP = "─" * 145
    HDR = (f"{'Method':<42} {'Hit@1':>7} {'Hit@5':>7} {'Hit@10':>7}  "
           f"{'Rec@1':>8} {'Rec@5':>8} {'Rec@10':>9}  "
           f"{'nDCG@1':>8} {'nDCG@5':>8} {'nDCG@10':>9}  {'Lat(ms)':>9}")
    print(f"\n{HDR}\n{SEP}")

    summary_rows = []

    for method in ['traditional', 'trad_weighted']:
        if method not in all_metrics: continue
        m   = all_metrics[method]
        cnt = m['count'] if m['count'] > 0 else 1
        lat = _lat_mean_from_tracker(method)
        lat_s = f"{lat:8.3f}" if not np.isnan(lat) else "     N/A"
        print(f"{method:<42} "
              f"{m['r1']/cnt*100:6.2f}%  {m['r5']/cnt*100:6.2f}%  {m['r10']/cnt*100:6.2f}%   "
              f"{m['recall1']/cnt*100:7.2f}%  {m['recall5']/cnt*100:7.2f}%  {m['recall10']/cnt*100:8.2f}%  "
              f"{m['n1']/cnt:7.4f}  {m['n5']/cnt:7.4f}  {m['n10']/cnt:8.4f}  {lat_s}")
        summary_rows.append({
            'method': method, 'n_layers': None, 'norm_mode': None, 'ratio': None, 'scoring': method,
            'hit@1':     round(m['r1']       / cnt * 100, 4),
            'hit@5':     round(m['r5']       / cnt * 100, 4),
            'hit@10':    round(m['r10']      / cnt * 100, 4),
            'recall@1':  round(m['recall1']  / cnt * 100, 4),
            'recall@5':  round(m['recall5']  / cnt * 100, 4),
            'recall@10': round(m['recall10'] / cnt * 100, 4),
            'ndcg@1':    round(m['n1']  / cnt, 6),
            'ndcg@5':    round(m['n5']  / cnt, 6),
            'ndcg@10':   round(m['n10'] / cnt, 6),
            'avg_lat_ms': round(lat, 4) if not np.isnan(lat) else None,
            'n_queries': cnt,
        })
    print(SEP)

    for n_layers in N_LAST_LAYERS_LIST:
        for mode in NORMALIZE_MODES:
            for scoring in ['trad', 'weighted']:
                for r in TOPK_RATIOS:
                    mk = f"L{n_layers}_norm{mode}_r{int(r*100)}_{scoring}"
                    if mk not in all_metrics: continue
                    m   = all_metrics[mk]
                    cnt = m['count'] if m['count'] > 0 else 1
                    lat = _lat_mean_from_tracker(mk)
                    lat_s = f"{lat:8.3f}" if not np.isnan(lat) else "     N/A"
                    print(f"{mk:<42} "
                          f"{m['r1']/cnt*100:6.2f}%  {m['r5']/cnt*100:6.2f}%  {m['r10']/cnt*100:6.2f}%   "
                          f"{m['recall1']/cnt*100:7.2f}%  {m['recall5']/cnt*100:7.2f}%  {m['recall10']/cnt*100:8.2f}%  "
                          f"{m['n1']/cnt:7.4f}  {m['n5']/cnt:7.4f}  {m['n10']/cnt:8.4f}  {lat_s}")
                    summary_rows.append({
                        'method': mk, 'n_layers': n_layers, 'norm_mode': mode,
                        'ratio': r, 'scoring': scoring,
                        'hit@1':     round(m['r1']       / cnt * 100, 4),
                        'hit@5':     round(m['r5']       / cnt * 100, 4),
                        'hit@10':    round(m['r10']      / cnt * 100, 4),
                        'recall@1':  round(m['recall1']  / cnt * 100, 4),
                        'recall@5':  round(m['recall5']  / cnt * 100, 4),
                        'recall@10': round(m['recall10'] / cnt * 100, 4),
                        'ndcg@1':    round(m['n1']  / cnt, 6),
                        'ndcg@5':    round(m['n5']  / cnt, 6),
                        'ndcg@10':   round(m['n10'] / cnt, 6),
                        'avg_lat_ms': round(lat, 4) if not np.isnan(lat) else None,
                        'n_queries': cnt,
                    })
            print(SEP)

    # Delta: weighted vs trad
    print(f"\n{'='*100}\nDELTA: weighted − trad  (nDCG@10 | Recall@10)")
    for n_layers in N_LAST_LAYERS_LIST:
        for mode in NORMALIZE_MODES:
            for r in TOPK_RATIOS:
                kt = f"L{n_layers}_norm{mode}_r{int(r*100)}_trad"
                kw = f"L{n_layers}_norm{mode}_r{int(r*100)}_weighted"
                if kt not in all_metrics or kw not in all_metrics: continue
                mt = all_metrics[kt]; ct = mt['count'] or 1
                mw = all_metrics[kw]; cw = mw['count'] or 1
                vt_n = mt['n10']/ct;          vw_n = mw['n10']/cw
                vt_r = mt['recall10']/ct*100; vw_r = mw['recall10']/cw*100
                dn = vw_n-vt_n; dr = vw_r-vt_r
                sn = "▲" if dn>0 else "▼" if dn<0 else "="
                sr = "▲" if dr>0 else "▼" if dr<0 else "="
                print(f"  L{n_layers}_{mode}_r{int(r*100):<10} "
                      f"nDCG {vt_n:.4f}→{vw_n:.4f} {sn}{abs(dn):.4f}  "
                      f"Rec {vt_r:.2f}%→{vw_r:.2f}% {sr}{abs(dr):.2f}%")

    # Delta: post vs pre
    print(f"\n{'='*100}\nDELTA: post − pre  (nDCG@10 | Recall@10)")
    for n_layers in N_LAST_LAYERS_LIST:
        for sc in ['trad', 'weighted']:
            for r in TOPK_RATIOS:
                kp = f"L{n_layers}_normpre_r{int(r*100)}_{sc}"
                ko = f"L{n_layers}_normpost_r{int(r*100)}_{sc}"
                if kp not in all_metrics or ko not in all_metrics: continue
                mp = all_metrics[kp]; cp = mp['count'] or 1
                mo = all_metrics[ko]; co = mo['count'] or 1
                vp_n = mp['n10']/cp;          vo_n = mo['n10']/co
                vp_r = mp['recall10']/cp*100; vo_r = mo['recall10']/co*100
                dn = vo_n-vp_n; dr = vo_r-vp_r
                sn = "▲" if dn>0 else "▼" if dn<0 else "="
                sr = "▲" if dr>0 else "▼" if dr<0 else "="
                print(f"  L{n_layers}_{sc}_r{int(r*100):<10} "
                      f"nDCG {vp_n:.4f}→{vo_n:.4f} {sn}{abs(dn):.4f}  "
                      f"Rec {vp_r:.2f}%→{vo_r:.2f}% {sr}{abs(dr):.2f}%")

    # Per-domain
    print(f"\n{'='*100}\nPER-DOMAIN — reference: traditional")
    print(f"  {'Domain':<30} {'N':>5} {'nDCG@10':>10} {'Recall@10':>12}")
    print("  " + "-" * 62)
    domain_summary_rows = []
    for domain in sorted(all_domain_metrics.keys()):
        dm    = all_domain_metrics[domain]
        ref_m = dm.get('traditional', _init_metric()); ref_c = ref_m['count'] or 1
        print(f"  {domain:<30} {ref_c:>5} "
              f"{ref_m['n10']/ref_c:>10.4f} {ref_m['recall10']/ref_c*100:>11.2f}%")
        row = {'domain': domain}
        for mk in ABLATION_KEYS:
            m_ = dm.get(mk, _init_metric()); c_ = m_['count'] or 1
            row[f'{mk}_ndcg@10']   = round(m_['n10']      / c_,       6)
            row[f'{mk}_recall@10'] = round(m_['recall10'] / c_ * 100, 4)
        domain_summary_rows.append(row)

    pd.DataFrame(summary_rows).to_csv(
        os.path.join(WORKING_DIR, "MASTER_ablation_summary.csv"), index=False)
    pd.DataFrame(domain_summary_rows).to_csv(
        os.path.join(WORKING_DIR, "MASTER_domain_summary.csv"), index=False)

    # Latency CSV
    all_latency_rows = []
    for mk, tracker in latency_trackers.items():
        all_latency_rows.extend(tracker.to_dict())
    if all_latency_rows:
        pd.DataFrame(all_latency_rows).to_csv(
            os.path.join(WORKING_DIR, "MASTER_latency_scoring_only.csv"), index=False)

    if all_batch_stats:
        pd.DataFrame(all_batch_stats).to_csv(
            os.path.join(WORKING_DIR, "MASTER_batch_stats.csv"), index=False)
    print("\n✅ All summaries saved")
    for row in all_batch_stats: print(f"  {row}")

except Exception as e:
    print(f"❌ Final aggregation error: {e}")
    import traceback; traceback.print_exc()


# ==============================================================================
# LATENCY REPORTS
# ==============================================================================
print("\n" + "=" * 80)
print("LATENCY REPORTS (scoring only, post-pooling/pruning)")
print("=" * 80)
for method_key in ['traditional', 'trad_weighted']:
    latency_trackers[method_key].report()
print("\n--- Ablation sample: L16_normpre, all ratios ---")
for sc in ['trad', 'weighted']:
    for r in TOPK_RATIOS:
        key = f"L16_normpre_r{int(r*100)}_{sc}"
        if key in latency_trackers:
            latency_trackers[key].report()


# ==============================================================================
# BƯỚC 5: VISUALIZE
# ==============================================================================
print("\n>>> BƯỚC 5: Visualize")
try:
    import matplotlib; matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker

    COLORS_N    = {4:'#E07B39', 8:'#3B72C4', 16:'#1D9E75', 28:'#9B59B6'}
    COLORS_MODE = {'pre':'#C23B22', 'post':'#4682B4'}

    def _mv(mk, field, pct=False):
        if mk not in all_metrics: return float('nan')
        m = all_metrics[mk]; cnt = m['count'] or 1
        return (m[field]/cnt)*100 if pct else m[field]/cnt

    # Fig 1: nDCG@10 & Recall@10 vs ratio per (n_layers, norm_mode)
    n_rows = len(N_LAST_LAYERS_LIST) * len(NORMALIZE_MODES)
    fig1, axes1 = plt.subplots(n_rows, 2, figsize=(14, 4 * n_rows), squeeze=False)
    fig1.suptitle("ColQwen2 — nDCG@10 & Recall@10 vs Ratio [SVD+ClusterPool v13]", fontsize=12)
    row_idx = 0
    for n_layers in N_LAST_LAYERS_LIST:
        for mode in NORMALIZE_MODES:
            for col_idx, (field, ylabel, pct) in enumerate([
                ('n10', 'nDCG@10', False),
                ('recall10', 'Recall@10 (%)', True),
            ]):
                ax = axes1[row_idx][col_idx]
                ax.set_title(f"L{n_layers}_{mode} — {ylabel}", fontsize=9)
                for sc, ls in [('trad','--'), ('weighted','-')]:
                    vals = [_mv(f"L{n_layers}_norm{mode}_r{int(r*100)}_{sc}", field, pct)
                            for r in TOPK_RATIOS]
                    ax.plot(TOPK_RATIOS, vals, color=COLORS_N[n_layers], linestyle=ls,
                            marker='o', ms=5, lw=1.8, label=sc)
                # baseline
                bv = _mv('traditional', field, pct)
                ax.axhline(bv, color='gray', ls=':', lw=1.2, label='trad_full')
                ax.set_xlabel('Keep ratio', fontsize=8)
                ax.set_ylabel(ylabel if col_idx == 0 else '', fontsize=8)
                ax.legend(fontsize=7, framealpha=0.4)
                ax.grid(alpha=0.25); ax.spines[['top','right']].set_visible(False)
            row_idx += 1
    plt.tight_layout()
    p1 = os.path.join(WORKING_DIR, "plot1_quality_vs_ratio.png")
    plt.savefig(p1, dpi=130, bbox_inches='tight'); plt.close()
    print(f"✅ Plot 1 → {p1}")

    # Fig 2: nDCG@10 vs n_layers at fixed ratio=0.5
    fig2, (ax2a, ax2b) = plt.subplots(1, 2, figsize=(12, 5))
    fig2.suptitle("ColQwen2 — Effect of N_LAST_LAYERS at ratio=0.5", fontsize=11)
    for mode in NORMALIZE_MODES:
        for sc in ['trad', 'weighted']:
            label  = f"{mode}_{sc}"
            color  = COLORS_MODE[mode]
            ls     = '-' if sc == 'trad' else '--'
            vals_n = [_mv(f"L{n}_norm{mode}_r50_{sc}", 'n10') for n in N_LAST_LAYERS_LIST]
            vals_r = [_mv(f"L{n}_norm{mode}_r50_{sc}", 'recall10', pct=True) for n in N_LAST_LAYERS_LIST]
            ax2a.plot(N_LAST_LAYERS_LIST, vals_n, color=color, linestyle=ls,
                      marker='o', ms=6, lw=1.8, label=label)
            ax2b.plot(N_LAST_LAYERS_LIST, vals_r, color=color, linestyle=ls,
                      marker='o', ms=6, lw=1.8, label=label)
    for ax, title in [(ax2a, 'nDCG@10'), (ax2b, 'Recall@10 (%)')]:
        ax.set_title(title, fontsize=10); ax.set_xlabel('N_LAST_LAYERS', fontsize=9)
        ax.set_xticks(N_LAST_LAYERS_LIST); ax.grid(alpha=0.25)
        ax.spines[['top','right']].set_visible(False)
        ax.legend(fontsize=8, framealpha=0.4)
    plt.tight_layout()
    p2 = os.path.join(WORKING_DIR, "plot2_quality_vs_nlayers.png")
    plt.savefig(p2, dpi=130, bbox_inches='tight'); plt.close()
    print(f"✅ Plot 2 → {p2}")

    # Fig 3: Latency vs nDCG@10 scatter
    fig3, ax3 = plt.subplots(figsize=(10, 6))
    ax3.set_title("ColQwen2 — Latency vs nDCG@10", fontsize=10)
    ax3.set_xlabel("Avg Latency (ms/query)", fontsize=9)
    ax3.set_ylabel("nDCG@10", fontsize=9)
    ax3.grid(alpha=0.25); ax3.spines[['top','right']].set_visible(False)
    for n_layers in N_LAST_LAYERS_LIST:
        for mode in NORMALIZE_MODES:
            for sc in ['trad', 'weighted']:
                for r in TOPK_RATIOS:
                    mk  = f"L{n_layers}_norm{mode}_r{int(r*100)}_{sc}"
                    nd  = _mv(mk, 'n10')
                    lat = _lat_mean_from_tracker(mk)
                    if np.isnan(nd) or np.isnan(lat): continue
                    mk_sc = 'o' if sc == 'trad' else 's'
                    ax3.scatter(lat, nd, color=COLORS_N[n_layers], marker=mk_sc, s=50, alpha=0.7)
    for bk, bc, bm in [('traditional','black','*'),('trad_weighted','red','P')]:
        bnd = _mv(bk, 'n10'); blat = _lat_mean_from_tracker(bk)
        if not np.isnan(bnd) and not np.isnan(blat):
            ax3.scatter(blat, bnd, color=bc, marker=bm, s=180, zorder=5, label=bk)
    ax3.legend(fontsize=8, framealpha=0.4)
    plt.tight_layout()
    p3 = os.path.join(WORKING_DIR, "plot3_latency_vs_ndcg.png")
    plt.savefig(p3, dpi=130, bbox_inches='tight'); plt.close()
    print(f"✅ Plot 3 → {p3}")

    # Fig 4: Heatmap nDCG@10 — n_layers × ratio (norm=pre, scoring=weighted)
    hm_data = np.array([[_mv(f"L{n}_normpre_r{int(r*100)}_weighted", 'n10')
                          for r in TOPK_RATIOS]
                         for n in N_LAST_LAYERS_LIST])
    fig4, ax4 = plt.subplots(figsize=(10, 4))
    im = ax4.imshow(hm_data, aspect='auto', cmap='YlGn')
    ax4.set_xticks(range(len(TOPK_RATIOS)))
    ax4.set_xticklabels([f"{int(r*100)}%" for r in TOPK_RATIOS], fontsize=8)
    ax4.set_yticks(range(len(N_LAST_LAYERS_LIST)))
    ax4.set_yticklabels([f"L{n}" for n in N_LAST_LAYERS_LIST], fontsize=9)
    ax4.set_title("ColQwen2 — nDCG@10 heatmap [norm=pre, scoring=weighted]", fontsize=10)
    plt.colorbar(im, ax=ax4)
    vmax = np.nanmax(hm_data)
    for ii in range(len(N_LAST_LAYERS_LIST)):
        for jj in range(len(TOPK_RATIOS)):
            val = hm_data[ii, jj]
            ax4.text(jj, ii, f"{val:.4f}", ha='center', va='center', fontsize=7.5,
                     color='white' if val > vmax * 0.75 else 'black')
    plt.tight_layout()
    p4 = os.path.join(WORKING_DIR, "plot4_heatmap_ndcg_pre_weighted.png")
    plt.savefig(p4, dpi=130, bbox_inches='tight'); plt.close()
    print(f"✅ Plot 4 → {p4}")

    # Fig 5: Delta nDCG@10 (weighted − trad) per ratio at L16_normpre
    deltas_n = []
    deltas_r = []
    for r in TOPK_RATIOS:
        kt = f"L16_normpre_r{int(r*100)}_trad"
        kw = f"L16_normpre_r{int(r*100)}_weighted"
        deltas_n.append(_mv(kw,'n10') - _mv(kt,'n10') if kt in all_metrics and kw in all_metrics else 0.0)
        deltas_r.append(_mv(kw,'recall10',True) - _mv(kt,'recall10',True) if kt in all_metrics and kw in all_metrics else 0.0)

    fig5, (ax5a, ax5b) = plt.subplots(1, 2, figsize=(12, 4))
    for ax5, deltas, ylabel in [(ax5a, deltas_n, 'Δ nDCG@10'), (ax5b, deltas_r, 'Δ Recall@10 (%)')]:
        colors = ['#1D9E75' if d >= 0 else '#E07B39' for d in deltas]
        ax5.bar([str(r) for r in TOPK_RATIOS], deltas, color=colors, edgecolor='white', lw=0.5)
        ax5.axhline(0, color='black', lw=0.8)
        ax5.set_title(f"{ylabel}: weighted − trad [L16_normpre]", fontsize=9)
        ax5.set_xlabel('Keep ratio', fontsize=8); ax5.set_ylabel(ylabel, fontsize=8)
        ax5.grid(axis='y', alpha=0.25); ax5.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    p5 = os.path.join(WORKING_DIR, "plot5_delta_weighted_vs_trad.png")
    plt.savefig(p5, dpi=130, bbox_inches='tight'); plt.close()
    print(f"✅ Plot 5 → {p5}")

    # Fig 6: Per-domain
    if all_domain_metrics:
        all_domains_sorted = sorted(all_domain_metrics.keys()); n_dom = len(all_domains_sorted)
        x = np.arange(n_dom)
        best_mk = max(
            (mk for mk in ABLATION_KEYS if mk in all_metrics),
            key=lambda mk: _mv(mk, 'n10'),
            default='traditional'
        )
        fig6, (ax6a, ax6b) = plt.subplots(2, 1, figsize=(max(8, n_dom * 1.1), 9))
        for ax6, field, pct, ylabel in [
            (ax6a, 'n10',      False, 'nDCG@10'),
            (ax6b, 'recall10', True,  'Recall@10 (%)'),
        ]:
            bv, rv = [], []
            for domain in all_domains_sorted:
                dm   = all_domain_metrics[domain]
                mb   = dm.get(best_mk,      _init_metric()); cb = mb['count'] or 1
                mr   = dm.get('traditional', _init_metric()); cr = mr['count'] or 1
                bv.append(mb[field] / cb * (100 if pct else 1))
                rv.append(mr[field] / cr * (100 if pct else 1))
            w = 0.35
            ax6.bar(x - w/2, rv, w, label='traditional',           color='#3B72C4', alpha=0.85)
            ax6.bar(x + w/2, bv, w, label=f'best ({best_mk[:20]})', color='#1D9E75', alpha=0.85)
            ax6.set_ylabel(ylabel, fontsize=9); ax6.set_xticks(x)
            ax6.set_xticklabels(all_domains_sorted, rotation=30, ha='right', fontsize=8)
            ax6.legend(fontsize=7, framealpha=0.4)
            ax6.grid(axis='y', alpha=0.25); ax6.spines[['top','right']].set_visible(False)
        fig6.suptitle("ColQwen2 — Per-domain: traditional vs best", fontsize=10)
        plt.tight_layout()
        p6 = os.path.join(WORKING_DIR, "plot6_domain_trad_vs_best.png")
        plt.savefig(p6, dpi=130, bbox_inches='tight'); plt.close()
        print(f"✅ Plot 6 → {p6}")

except Exception as e:
    print(f"⚠️  Plot error (non-fatal): {e}")
    import traceback; traceback.print_exc()

print("\n>>> BƯỚC 4+5 v13-ColQwen2 COMPLETE — SVD+ClusterPool+AreaRecall DONE")
'''

'\nprint(">>> BƯỚC 3 v13-ColQwen2 [Full Ablation — SVD+ClusterPool+AreaRecall] STARTING")\n\nimport torch\nimport torch.nn.functional as F\nfrom transformers.models.qwen2_vl import Qwen2VLModel\n\n# ==============================================================================\n# CONFIG\n# ==============================================================================\nIMPORTANCE_VARIANT   = \'svd_sink_softplus_clusterPool_v13_ablation_colqwen2\'\nTEMPERATURE          = 0.5\nMIN_IMPORTANCE_CLAMP = 1e-6\nSVD_RANK_REMOVE      = 1\nTOPK_RATIOS          = [round(i * 0.1, 1) for i in range(1, 10)]   # 0.1 … 0.9\n\n# Ablation axes\nN_LAST_LAYERS_LIST = [4, 8, 16, 28]\nNORMALIZE_MODES    = [\'pre\', \'post\']\n\nMAX_N_LAYERS = max(N_LAST_LAYERS_LIST)\n\n# PKL path override (giữ nguyên từ bản gốc ColQwen2)\nBATCH_RANGE_PKL_OVERRIDE = {\n    (0, 25): "/kaggle/input/datasets/nguyenducdung1107/vvvvvvv/colqwen2_fused_index (2).pkl",\n}\n\ndef _make_ablation_keys():\n    keys = [\'traditional\', \'t

2.HIERTICAL POOLING

In [13]:
# ==============================================================================
# BƯỚC 3 v12-HierOnly – Hierarchical Pooling (Ward's method)
# Method: gộp N token query thành K = max(1, round(N * ratio)) cụm
#         bằng Ward agglomerative clustering (cosine distance),
#         mean pool các vector trong mỗi cụm → K vectors đại diện,
#         MaxSim K vectors đó với doc → sum scores.
# Ablation: thử các topk_ratio khác nhau.
# Baseline: traditional MaxSim (sum toàn bộ token).
#
# Recall@K chuẩn: hits@K / |GT|  (per query, mean over queries)
# ==============================================================================

print(">>> BƯỚC 3 v12-HierOnly [FIXED]: Hierarchical Pooling (Ward's method)")

import torch
import torch.nn.functional as F
import numpy as np
import json, os, pickle, gc, glob
import pandas as pd
from tqdm.notebook import tqdm

# ==============================================================================
# CONFIG
# ==============================================================================
TOPK_RATIOS      = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
WORKING_DIR      = "/kaggle/working"
ANNOTATIONS_PATH = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_annotations.jsonl"
COLSMOL_DIR      = "/kaggle/input/datasets/nguyenducdung1107/colsmol500m-layoutmmdoc/colsmol500m-pkl"
QUERY_BATCH_SIZE = 50

BATCH_RANGE_PKL_OVERRIDE = {
    (0, 25): "/kaggle/input/datasets/nguyenducdung1107/vvvvvvv/colqwen2_fused_index (2).pkl",
}

# ==============================================================================
# UTILS
# ==============================================================================
def build_content_mask(inputs, processor):
    attn_mask = inputs["attention_mask"]
    input_ids = inputs.get("input_ids", None)
    if input_ids is None:
        return attn_mask.float()
    tok = getattr(processor, 'tokenizer', processor)
    special_ids = set()
    for attr in ['pad_token_id', 'bos_token_id', 'eos_token_id',
                 'unk_token_id', 'sep_token_id', 'cls_token_id']:
        tid = getattr(tok, attr, None)
        if tid is not None:
            special_ids.add(int(tid))
    if hasattr(tok, 'added_tokens_encoder'):
        for _, tid in tok.added_tokens_encoder.items():
            special_ids.add(int(tid))
    if not special_ids:
        return attn_mask.float()
    special_tensor = torch.tensor(list(special_ids), device=input_ids.device)
    is_special = (input_ids.unsqueeze(-1) == special_tensor).any(dim=-1)
    return attn_mask.float() * (~is_special).float()


# ==============================================================================
# DOC MATRIX BUILDER — handle dict và Tensor
# ==============================================================================
def _coerce_to_tensor(item):
    if isinstance(item, torch.Tensor):
        return item
    if isinstance(item, dict):
        for key in ('embedding', 'embeddings', 'vector', 'vectors', 'token_embeddings'):
            if key in item:
                v = item[key]
                if isinstance(v, torch.Tensor):
                    return v
        tensor_vals = [(k, v) for k, v in item.items() if isinstance(v, torch.Tensor)]
        if len(tensor_vals) == 1:
            return tensor_vals[0][1]
        if tensor_vals:
            return tensor_vals[0][1]
    raise ValueError(
        f"Cannot extract Tensor from fused_embeddings item of type {type(item)}. "
        f"Keys: {list(item.keys()) if isinstance(item, dict) else 'N/A'}"
    )


def build_doc_matrix(docs_emb, device):
    tensors = []
    for i, item in enumerate(docs_emb):
        try:
            t = _coerce_to_tensor(item)
        except ValueError as e:
            raise ValueError(f"docs_emb[{i}]: {e}") from e
        if t.dim() == 1:
            t = t.unsqueeze(0)
        elif t.dim() > 2:
            t = t.squeeze(0)
        tensors.append(t)

    n     = len(tensors)
    max_l = max(t.shape[0] for t in tensors)
    D     = tensors[0].shape[1]
    doc_matrix = torch.zeros(n, max_l, D, device=device)
    doc_mask   = torch.zeros(n, max_l,    device=device, dtype=torch.bool)
    for i, t in enumerate(tensors):
        L = t.shape[0]
        doc_matrix[i, :L] = F.normalize(t.float().to(device), dim=-1)
        doc_mask[i, :L]   = True
    return doc_matrix, doc_mask


@torch.no_grad()
def fast_maxsim(q_norm, doc_matrix, doc_mask):
    """q_norm: (Q, D) → returns (Q, n_docs)"""
    sim = torch.einsum('qd,nld->qnl', q_norm, doc_matrix)
    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
    return sim.max(dim=-1).values


# ==============================================================================
# WARD AGGLOMERATIVE CLUSTERING (cosine distance)
# ==============================================================================
def _ward_distance_matrix(cents, sizes):
    device  = cents.device
    sim     = torch.matmul(cents, cents.t()).clamp(-1.0, 1.0)
    sq_dist = 2.0 * (1.0 - sim)
    ni = sizes.float().unsqueeze(1)
    nj = sizes.float().unsqueeze(0)
    w  = (ni * nj) / (ni + nj)
    ward = w * sq_dist
    mask = torch.ones(cents.shape[0], cents.shape[0], dtype=torch.bool, device=device).tril()
    ward.masked_fill_(mask, float('inf'))
    return ward


def ward_pool(vecs, n_clusters):
    N = vecs.shape[0]
    if N <= n_clusters:
        return vecs.clone()

    device = vecs.device
    sums   = vecs.clone().float()
    sizes  = torch.ones(N, device=device, dtype=torch.float32)
    cents  = F.normalize(sums, dim=-1)
    active = torch.ones(N, dtype=torch.bool, device=device)
    C      = N

    while C > n_clusters:
        active_idx = active.nonzero(as_tuple=True)[0]
        c_act = cents[active_idx]
        s_act = sizes[active_idx]

        ward     = _ward_distance_matrix(c_act, s_act)
        flat_idx = ward.argmin().item()
        n_act    = active_idx.shape[0]
        ai       = flat_idx // n_act
        aj       = flat_idx %  n_act
        gi       = active_idx[ai].item()
        gj       = active_idx[aj].item()

        sums[gi]   = sums[gi] + sums[gj]
        sizes[gi]  = sizes[gi] + sizes[gj]
        cents[gi]  = F.normalize(sums[gi].unsqueeze(0), dim=-1).squeeze(0)
        active[gj] = False
        C -= 1

    final_idx = active.nonzero(as_tuple=True)[0]
    return cents[final_idx]


def ward_pool_scores_all_ratios(q_norm, doc_matrix, doc_mask, topk_ratios):
    N       = q_norm.shape[0]
    results = {}
    sorted_ratios = sorted(topk_ratios, reverse=True)
    prev_vecs = q_norm.clone()
    prev_C    = N

    for ratio in sorted_ratios:
        target_C = max(1, round(N * ratio))
        if target_C >= prev_C:
            centroids = prev_vecs
        else:
            centroids = ward_pool(prev_vecs, target_C)
            prev_vecs = centroids
            prev_C    = centroids.shape[0]

        M_c            = fast_maxsim(centroids, doc_matrix, doc_mask)
        results[ratio] = M_c.sum(0)

    return results


# ==============================================================================
# METRICS
# Recall@K = |topK ∩ GT| / |GT|  (per query, range [0,1])
# Hit@K    = 1 nếu có ít nhất 1 GT trong top-K (binary)
# nDCG@K   = DCG@K / IDCG@K
# ==============================================================================
def compute_ndcg(ranked, gt_set, k):
    dcg  = sum(1.0 / np.log2(r + 2) for r, i in enumerate(ranked[:k]) if i in gt_set)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt_set), k)))
    return dcg / idcg if idcg > 0 else 0.0

def first_hit(top_k, gt_set):
    for r, i in enumerate(top_k):
        if i in gt_set: return r + 1
    return -1

def hit_metrics(top_k_list, gt_set):
    """
    top_k_list : list[int] — ranked doc indices từ torch.topk
    gt_set     : set[int]  — ground-truth doc indices

    Recall@K = |topK ∩ GT| / |GT|   per query
    Hit@K    = 1 nếu có ít nhất 1 GT trong top-K
    """
    gt_size = max(len(gt_set), 1)
    h = first_hit(top_k_list, gt_set)

    hits1  = sum(1 for idx in top_k_list[:1]  if idx in gt_set)
    hits5  = sum(1 for idx in top_k_list[:5]  if idx in gt_set)
    hits10 = sum(1 for idx in top_k_list[:10] if idx in gt_set)

    return {
        'r1':       int(h != -1 and h <= 1),
        'r5':       int(h != -1 and h <= 5),
        'r10':      int(h != -1 and h <= 10),
        'recall1':  hits1  / gt_size,
        'recall5':  hits5  / gt_size,
        'recall10': hits10 / gt_size,
        'n1':       float(compute_ndcg(top_k_list, gt_set, 1)),
        'n5':       float(compute_ndcg(top_k_list, gt_set, 5)),
        'n10':      float(compute_ndcg(top_k_list, gt_set, 10)),
    }


# ==============================================================================
# METRIC STORE
# ==============================================================================
def _init_metric():
    return {'r1': 0, 'r5': 0, 'r10': 0,
            'n1': 0., 'n5': 0., 'n10': 0.,
            'recall1': 0., 'recall5': 0., 'recall10': 0.,
            'count': 0}

def _add_metric(dst, src):
    for f in ('r1', 'r5', 'r10'):
        dst[f] += int(src[f])
    for f in ('n1', 'n5', 'n10', 'recall1', 'recall5', 'recall10'):
        dst[f] += float(src[f])
    dst['count'] += 1

def _ensure(store, key):
    if key not in store:
        store[key] = _init_metric()
    return store[key]

def record(key, m, domain):
    _add_metric(_ensure(all_metrics, key), m)
    _add_metric(_ensure(all_domain_metrics[domain], key), m)


# ==============================================================================
# METHOD KEYS
# ==============================================================================
ABLATION_KEYS = ['traditional'] + [f"hier_r{int(r*100)}" for r in TOPK_RATIOS]

# ==============================================================================
# MASTER STORAGE
# ==============================================================================
all_query_results  = []
all_metrics        = {}
all_domain_metrics = {}
all_batch_stats    = []

print(f"topk_ratios : {TOPK_RATIOS}  →  K = max(1, round(N_tokens * ratio))")
print("=" * 70)

for batch_num, (START_IDX, END_IDX) in enumerate(BATCH_RANGES, 1):
    print(f"\n[{batch_num}/{len(BATCH_RANGES)}] Batch [{START_IDX}:{END_IDX}]")

    # ── Resolve index path ─────────────────────────────────────────────────────
    batch_key  = (START_IDX, END_IDX)
    index_path = BATCH_RANGE_PKL_OVERRIDE.get(batch_key, None)

    if index_path is not None:
        print(f"  ℹ️  Using PKL override: {index_path}")
    else:
        for _pkl in pkl_files:
            try:
                _nums = os.path.basename(_pkl).replace('.pkl', '').split(' ')[-1]
                _s, _e = map(int, _nums.split('-'))
                if _s == START_IDX and _e == END_IDX:
                    index_path = _pkl
                    break
            except Exception:
                continue
        if index_path is None:
            _candidate = os.path.join(COLSMOL_DIR, f"{START_IDX}-{END_IDX}.pkl")
            if os.path.exists(_candidate):
                index_path = _candidate

    if index_path is None or not os.path.exists(index_path):
        print(f"  ⚠️  Cannot find PKL for [{START_IDX}:{END_IDX}]")
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}", 'status': 'index_not_found'})
        continue

    try:
        with open(index_path, 'rb') as f:
            saved = pickle.load(f)
        if isinstance(saved, list):
            fused_embeddings = saved
        elif isinstance(saved, dict):
            fused_embeddings = saved.get('embeddings', [])
        else:
            fused_embeddings = saved
        print(f"  ✅ {len(fused_embeddings)} layouts")
    except Exception as e:
        print(f"  ❌ {e}"); continue

    # ── Load batch data ────────────────────────────────────────────────────────
    try:
        batch_doc_names    = intersection_docs[START_IDX:END_IDX]
        batch_target_files = [jsonl_map[d] for d in batch_doc_names]

        batch_df_orig = pd.read_parquet(PARQUET_PATH)
        batch_df_orig['join_doc_name'] = batch_df_orig['doc_name'].str.replace('.pdf', '', regex=False)
        batch_df_orig = batch_df_orig[batch_df_orig['join_doc_name'].isin(batch_doc_names)]

        dfs = []
        for f in tqdm(batch_target_files, desc="  Reading JSONLs", leave=False):
            try:
                temp = pd.read_json(f, lines=True)
                temp['join_doc_name'] = os.path.basename(f).replace('_layout.jsonl', '')
                if 'layout' in temp.columns:
                    temp = temp.rename(columns={'layout': 'layout_id'})
                cols = ['join_doc_name', 'layout_id', 'vlm_text', 'img_enhanced_path']
                if 'text_level' in temp.columns: cols.append('text_level')
                dfs.append(temp[[c for c in cols if c in temp.columns]])
            except Exception: pass

        if dfs:
            batch_df_enh = pd.concat(dfs, ignore_index=True).rename(
                columns={'vlm_text': 'vlm_text_enhanced', 'text_level': 'text_level_enhanced'}
            )
        else:
            batch_df_enh = pd.DataFrame()

        df = pd.merge(batch_df_orig, batch_df_enh, on=['join_doc_name', 'layout_id'], how='left')
        df = df.sort_values(['join_doc_name', 'page_id', 'layout_id'])

        is_header = df['type'].isin(['title', 'section_header', 'header']) | \
                    df.get('text_level_enhanced', pd.Series(dtype=object)).notna()
        df['temp_header']     = df['text'].where(is_header)
        df['current_section'] = df.groupby('join_doc_name')['temp_header'] \
                                   .ffill().fillna("General Content")

        enh_image_map = {}
        if os.path.exists(ENHANCED_IMG_DIR):
            for fn in glob.glob(os.path.join(ENHANCED_IMG_DIR, "*")):
                enh_image_map[os.path.basename(fn)] = fn

        if 'img_enhanced_path' in df.columns:
            df['img_data'] = df['img_enhanced_path'].dropna().map(
                lambda p: enh_image_map.get(os.path.basename(str(p))))
            df['img_type'] = df['img_data'].notna().map(lambda x: 'path' if x else None)
        else:
            df['img_data'] = None; df['img_type'] = None

        if 'image_binary' in df.columns:
            m = df['img_type'].isna() & df['image_binary'].notna()
            df.loc[m, 'img_type'] = 'binary'
            df.loc[m, 'img_data'] = df.loc[m, 'image_binary']

        def _pick_text(row):
            for col in ['vlm_text_enhanced', 'text', 'ocr_text', 'vlm_text']:
                v = row.get(col)
                if pd.notna(v) and len(str(v)) > 5: return str(v)
            return "Document layout."

        df['final_text'] = "Section: " + df['current_section'].fillna('') + \
                           " \n Content: " + df.apply(_pick_text, axis=1)
        layouts_df = df.dropna(subset=['img_type']).reset_index(drop=True)
        print(f"  ✅ {len(layouts_df)} layouts with image")

    except Exception as e:
        print(f"  ❌ Batch data error: {e}"); import traceback; traceback.print_exc(); continue

    # ── Build QA pairs ─────────────────────────────────────────────────────────
    def calculate_iou(b1, b2):
        b1, b2 = list(b1), list(b2)
        xi, yi = max(b1[0], b2[0]), max(b1[1], b2[1])
        xa, ya = min(b1[2], b2[2]), min(b1[3], b2[3])
        inter  = max(0, xa - xi) * max(0, ya - yi)
        union  = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
        return inter / union if union > 0 else 0.0

    qa_pairs   = []
    doc_lookup = {k: v for k, v in layouts_df.groupby('join_doc_name')}
    avail_docs = set(doc_lookup.keys())

    with open(ANNOTATIONS_PATH, 'r') as f:
        for line in f:
            try: doc_data = json.loads(line)
            except Exception: continue
            target_doc = doc_data['doc_name'].replace('.pdf', '')
            if target_doc not in avail_docs: continue
            doc_layouts = doc_lookup[target_doc].copy()
            col_page    = 'page_idx' if 'page_idx' in doc_layouts.columns else 'page_id'
            doc_layouts['safe_page'] = pd.to_numeric(
                doc_layouts[col_page], errors='coerce').fillna(-999).astype(int)
            domain = doc_data.get('domain', 'General')
            for q_item in doc_data.get('questions', []):
                gt = []
                for target in q_item.get('layout_mapping', []):
                    try:
                        tp    = int(target['page'])
                        cands = pd.concat([doc_layouts[doc_layouts['safe_page'] == tp],
                                           doc_layouts[doc_layouts['safe_page'] == tp - 1]])
                        for idx, row in cands.iterrows():
                            if calculate_iou(row['bbox'], target['bbox']) > 0.5:
                                gt.append(int(idx))
                    except Exception: continue
                if gt:
                    qa_pairs.append({
                        'question':   q_item['Q'],
                        'gt_indices': list(set(gt)),
                        'doc_name':   target_doc,
                        'domain':     domain,
                    })

    print(f"  ✅ {len(qa_pairs)} QA pairs")
    if not qa_pairs:
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}", 'n_qa': 0, 'status': 'no_qa'})
        continue

    # ── Evaluate ───────────────────────────────────────────────────────────────
    try:
        device   = "cuda" if torch.cuda.is_available() else "cpu"
        n_docs   = len(fused_embeddings)
        doc_matrix, doc_mask = build_doc_matrix(fused_embeddings, device)
        print(f"  ✅ Doc matrix: {doc_matrix.shape}")

        batch_query_rows = []
        batch_metrics    = {k: _init_metric() for k in ABLATION_KEYS}

        for qs in range(0, len(qa_pairs), QUERY_BATCH_SIZE):
            qe = min(qs + QUERY_BATCH_SIZE, len(qa_pairs))
            pbar = tqdm(enumerate(qa_pairs[qs:qe]), total=qe - qs,
                        desc=f"  Q[{qs}:{qe}]", leave=False)

            for qi, item in pbar:
                gq     = qs + qi
                gt_set = set(item['gt_indices'])
                domain = item['domain']
                if domain not in all_domain_metrics:
                    all_domain_metrics[domain] = {}

                with torch.no_grad():
                    q_inputs = processor.process_queries([item['question']]).to(device)
                    outputs  = model(**q_inputs)
                    if isinstance(outputs, torch.Tensor):
                        proj = outputs[0].float()
                    elif hasattr(outputs, 'last_hidden_state'):
                        proj = outputs.last_hidden_state[0].float()
                    else:
                        proj = outputs[0][0].float()
                    proj = proj / (proj.norm(dim=-1, keepdim=True) + 1e-8)

                content_mask_1d = build_content_mask(q_inputs, processor)[0].float()
                valid_idx = torch.where(content_mask_1d > 0)[0]
                if valid_idx.numel() == 0:
                    valid_idx = torch.where(q_inputs['attention_mask'][0] > 0)[0]

                q_norm = F.normalize(proj[valid_idx], dim=-1)

                # Baseline: traditional MaxSim
                M_trad      = fast_maxsim(q_norm, doc_matrix, doc_mask)
                trad_scores = M_trad.sum(0)
                trad_top10  = torch.topk(trad_scores, min(10, n_docs)).indices.cpu().tolist()
                m_trad      = hit_metrics(trad_top10, gt_set)
                _add_metric(batch_metrics['traditional'], m_trad)
                record('traditional', m_trad, domain)

                query_row = {
                    'batch':          f"{START_IDX}-{END_IDX}",
                    'query_id':       gq,
                    'doc_name':       item['doc_name'],
                    'domain':         domain,
                    'question':       item['question'],
                    'gt_count':       len(gt_set),
                    'trad_hit@1':     m_trad['r1'],
                    'trad_hit@5':     m_trad['r5'],
                    'trad_hit@10':    m_trad['r10'],
                    'trad_recall@1':  round(m_trad['recall1'],  6),
                    'trad_recall@5':  round(m_trad['recall5'],  6),
                    'trad_recall@10': round(m_trad['recall10'], 6),
                    'trad_ndcg@10':   round(m_trad['n10'],      4),
                }

                # Ward Hierarchical Pool — tất cả ratio trong 1 lần
                ratio_scores = ward_pool_scores_all_ratios(
                    q_norm, doc_matrix, doc_mask, TOPK_RATIOS
                )

                for r in TOPK_RATIOS:
                    key    = f"hier_r{int(r*100)}"
                    scores = ratio_scores[r]
                    top10  = torch.topk(scores, min(10, n_docs)).indices.cpu().tolist()
                    m      = hit_metrics(top10, gt_set)
                    _add_metric(batch_metrics[key], m)
                    record(key, m, domain)
                    query_row.update({
                        f'{key}_hit@1':     m['r1'],
                        f'{key}_hit@5':     m['r5'],
                        f'{key}_hit@10':    m['r10'],
                        f'{key}_recall@1':  round(m['recall1'],  6),
                        f'{key}_recall@5':  round(m['recall5'],  6),
                        f'{key}_recall@10': round(m['recall10'], 6),
                        f'{key}_ndcg@10':   round(m['n10'],      4),
                    })

                batch_query_rows.append(query_row)

            gc.collect(); torch.cuda.empty_cache()
            print(f"    ✓ Q[{qs}:{qe}] done")

        all_query_results.extend(batch_query_rows)

        t   = batch_metrics['traditional']
        cnt = t['count'] if t['count'] > 0 else 1
        print(f"  ✅ {len(batch_query_rows)} queries — "
              f"Baseline Hit@10={t['r10']/cnt*100:.1f}%  "
              f"Recall@10={t['recall10']/cnt*100:.1f}%  "
              f"nDCG@10={t['n10']/cnt:.4f}")

        all_batch_stats.append({
            'batch':          f"{START_IDX}-{END_IDX}",
            'n_layouts':      n_docs,
            'n_queries':      len(qa_pairs),
            'trad_hit@10':    round(t['r10']     / cnt * 100, 2),
            'trad_recall@10': round(t['recall10'] / cnt * 100, 2),
            'trad_ndcg@10':   round(t['n10']     / cnt,       4),
            'status':         'ok',
        })

    except Exception as e:
        print(f"  ❌ Eval error: {e}"); import traceback; traceback.print_exc()
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}",
                                'status': 'error', 'error': str(e)})

    # ── Save & cleanup ─────────────────────────────────────────────────────────
    try:
        if batch_query_rows:
            pd.DataFrame(batch_query_rows).to_csv(
                os.path.join(WORKING_DIR, f"batch_{START_IDX}_{END_IDX}_queries.csv"), index=False)
        print("  ✅ Batch CSV saved")
    except Exception as e:
        print(f"  ❌ Save error: {e}")

    del doc_matrix, doc_mask, fused_embeddings, layouts_df, qa_pairs, df
    del batch_df_orig, batch_df_enh
    gc.collect(); torch.cuda.empty_cache()
    print("  ✓ Cleaned\n")


# ==============================================================================
# FINAL SUMMARY
# ==============================================================================
print("=" * 80)
print("CONSOLIDATING")
print("=" * 80)

try:
    if all_query_results:
        df_q = pd.DataFrame(all_query_results)
        df_q.to_csv(os.path.join(WORKING_DIR, "MASTER_queries.csv"), index=False)
        print(f"✅ {len(df_q)} queries → MASTER_queries.csv")

    SEP = "-" * 75
    print(f"\n{'Method':<20} {'Hit@1':>7} {'Hit@5':>7} {'Hit@10':>7} "
          f"{'Rec@1':>8} {'Rec@5':>8} {'Rec@10':>9} {'nDCG@10':>9}")
    print(SEP)
    for key in ABLATION_KEYS:
        if key not in all_metrics: continue
        m = all_metrics[key]; cnt = m['count'] if m['count'] > 0 else 1
        print(f"{key:<20} "
              f"{m['r1']/cnt*100:6.2f}%  {m['r5']/cnt*100:6.2f}%  {m['r10']/cnt*100:6.2f}%  "
              f"{m['recall1']/cnt*100:7.2f}%  {m['recall5']/cnt*100:7.2f}%  "
              f"{m['recall10']/cnt*100:8.2f}%  {m['n10']/cnt:8.4f}")

    print("\n" + "=" * 80)
    print("PER-DOMAIN — traditional vs ward per ratio (nDCG@10)")
    print("=" * 80)
    ratio_keys = [f"hier_r{int(r*100)}" for r in TOPK_RATIOS]
    hdr = " | ".join(f"r{int(r*100):>3}" for r in TOPK_RATIOS)
    print(f"{'Domain':<22} | {'trad':>6} | {hdr}")
    print("-" * (22 + 3 + 8 + 3 + len(TOPK_RATIOS) * 8))
    for domain in sorted(all_domain_metrics):
        dm  = all_domain_metrics[domain]
        t_m = dm.get('traditional', _init_metric()); tc = t_m['count'] if t_m['count'] > 0 else 1
        vals = []
        for key in ratio_keys:
            m_ = dm.get(key, _init_metric()); c_ = m_['count'] if m_['count'] > 0 else 1
            vals.append(f"{m_['n10']/c_:6.4f}")
        print(f"{domain:<22} | {t_m['n10']/tc:6.4f} | {' | '.join(vals)}")

    summary_rows = []
    for key in ABLATION_KEYS:
        if key not in all_metrics: continue
        m = all_metrics[key]; cnt = m['count'] if m['count'] > 0 else 1
        summary_rows.append({
            'method':    key,
            'hit@1':     round(m['r1']      / cnt * 100, 4),
            'hit@5':     round(m['r5']      / cnt * 100, 4),
            'hit@10':    round(m['r10']     / cnt * 100, 4),
            'recall@1':  round(m['recall1'] / cnt * 100, 4),
            'recall@5':  round(m['recall5'] / cnt * 100, 4),
            'recall@10': round(m['recall10']/ cnt * 100, 4),
            'ndcg@1':    round(m['n1']      / cnt,       6),
            'ndcg@5':    round(m['n5']      / cnt,       6),
            'ndcg@10':   round(m['n10']     / cnt,       6),
        })
    pd.DataFrame(summary_rows).to_csv(
        os.path.join(WORKING_DIR, "MASTER_ablation_summary.csv"), index=False)

    domain_rows = []
    for domain in sorted(all_domain_metrics):
        dm  = all_domain_metrics[domain]
        row = {'domain': domain}
        for key in ABLATION_KEYS:
            m_   = dm.get(key, _init_metric()); cnt_ = m_['count'] if m_['count'] > 0 else 1
            row[f'{key}_ndcg@10']   = round(m_['n10']     / cnt_,       6)
            row[f'{key}_recall@10'] = round(m_['recall10'] / cnt_ * 100, 4)
            row[f'{key}_hit@10']    = round(m_['r10']     / cnt_ * 100, 4)
        domain_rows.append(row)
    pd.DataFrame(domain_rows).to_csv(
        os.path.join(WORKING_DIR, "MASTER_domain_summary.csv"), index=False)

    pd.DataFrame(all_batch_stats).to_csv(
        os.path.join(WORKING_DIR, "MASTER_batch_stats.csv"), index=False)

    print("\n✅ MASTER_ablation_summary.csv")
    print("✅ MASTER_domain_summary.csv")
    print("✅ MASTER_batch_stats.csv")

except Exception as e:
    print(f"❌ Summary error: {e}"); import traceback; traceback.print_exc()

# ==============================================================================
# VISUALIZE
# ==============================================================================
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker

    metrics_to_plot = [
        ('n10',      'nDCG@10',    False),
        ('recall10', 'Recall@10',  True),
        ('n5',       'nDCG@5',     False),
        ('recall1',  'Recall@1',   True),
    ]

    total_q = all_metrics.get('traditional', {}).get('count', 1)

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    axes = axes.flatten()

    for ax_idx, (mkey, title, pct_flag) in enumerate(metrics_to_plot):
        ax = axes[ax_idx]
        ax.set_title(title, fontsize=12, fontweight='normal', pad=8)
        ax.set_xlabel('Top-k ratio', fontsize=10)
        ax.set_ylabel('Recall (%)' if pct_flag else 'Score', fontsize=10)
        ax.grid(axis='y', alpha=0.25, linewidth=0.6)
        ax.grid(axis='x', alpha=0.15, linewidth=0.4)
        ax.spines[['top', 'right']].set_visible(False)

        if total_q > 0 and 'traditional' in all_metrics:
            base_val = all_metrics['traditional'][mkey] / total_q
            if pct_flag: base_val *= 100
            ax.axhline(base_val, color='#888780', linewidth=1.8,
                       linestyle='--', label='Baseline (full ColSMoL)', zorder=2)

        vals = []
        for r in TOPK_RATIOS:
            key = f"hier_r{int(r*100)}"
            if key in all_metrics and total_q > 0:
                v = all_metrics[key][mkey] / total_q
                if pct_flag: v *= 100
                vals.append(v)
            else:
                vals.append(None)

        valid = [(r, v) for r, v in zip(TOPK_RATIOS, vals) if v is not None]
        if valid:
            xs, ys = zip(*valid)
            ax.plot(xs, ys, color='#1DB954', linewidth=2.2,
                    marker='o', markersize=5,
                    label='Ward Hierarchical Pooling', zorder=3)

        ax.set_xticks(TOPK_RATIOS)
        ax.set_xticklabels([str(r) for r in TOPK_RATIOS], fontsize=8)
        fmt = (lambda v, _: f"{v:.1f}%") if pct_flag else (lambda v, _: f"{v:.3f}")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt))
        ax.legend(fontsize=8, framealpha=0.4, loc='lower right')

    plt.suptitle(
        "Token Pooling with Ward Hierarchical Clustering vs Baseline\n"
        "K clusters = round(N_tokens × top-k ratio)  |  Recall@K = hits/|GT| per query",
        fontsize=11, y=1.02
    )
    plt.tight_layout()
    plot_path = os.path.join(WORKING_DIR, "hierarchical_pooling_ward_comparison.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"✅ Plot saved → {plot_path}")

except Exception as e:
    print(f"⚠️  Plot error (non-fatal): {e}")

print("\n>>> BƯỚC 3 v12-HierOnly [FIXED] DONE")

>>> BƯỚC 3 v12-HierOnly [FIXED]: Hierarchical Pooling (Ward's method)
topk_ratios : [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]  →  K = max(1, round(N_tokens * ratio))

[1/13] Batch [0:25]
  ℹ️  Using PKL override: /kaggle/input/datasets/nguyenducdung1107/vvvvvvv/colqwen2_fused_index (2).pkl
  ✅ 6850 layouts


  Reading JSONLs:   0%|          | 0/25 [00:00<?, ?it/s]

  ✅ 6850 layouts with image
  ✅ 122 QA pairs
  ✅ Doc matrix: torch.Size([6850, 1471, 128])


  Q[0:50]:   0%|          | 0/50 [00:00<?, ?it/s]

    ✓ Q[0:50] done


...

  ✅ 8399 layouts with image
  ✅ 59 QA pairs
  ✅ Doc matrix: torch.Size([8399, 1499, 128])


  Q[0:50]:   0%|          | 0/50 [00:00<?, ?it/s]

    ✓ Q[0:50] done


  Q[50:59]:   0%|          | 0/9 [00:00<?, ?it/s]

    ✓ Q[50:59] done
  ✅ 59 queries — Baseline Hit@10=74.6%  Recall@10=62.0%  nDCG@10=0.5154
  ✅ Batch CSV saved
  ✓ Cleaned

CONSOLIDATING
✅ 1603 queries → MASTER_queries.csv

Method                 Hit@1   Hit@5  Hit@10    Rec@1    Rec@5    Rec@10   nDCG@10
---------------------------------------------------------------------------
traditional           41.80%   64.07%   70.31%    31.17%    52.21%     58.85%    0.4873
hier_r10              16.22%   31.94%   39.99%    11.76%    24.73%     32.07%    0.2290
hier_r20              26.76%   48.47%   56.21%    20.16%    39.02%     46.19%    0.3520
hier_r30              34.56%   56.89%   64.88%    26.08%    46.28%     53.28%    0.4251
hier_r40              38.49%   60.57%   67.94%    29.05%    48.96%     56.69%    0.4597
hier_r50              40.11%   63.13%   69.12%    30.45%    51.51%     57.84%    0.4761
hier_r60              40.86%   63.82%   69.49%    30.88%    51.95%     58.14%    0.4807
hier_r70              41.30%   63.94%   69.68%   

**3.Attention**

In [14]:
# ==============================================================================
# BƯỚC 3 v13-AttentionScore – Token Pruning theo Lassance et al. (2021)
# "A Study on Token Pruning for ColBERT" — arXiv:2112.06540
#
# Recall@K chuẩn: hits@K / |GT|  (per query, mean over queries)
# ==============================================================================

print(">>> BƯỚC 3 v13-AttentionScore [FIXED]: Token Pruning — correct Recall")

import torch
import torch.nn.functional as F
from transformers.models.qwen2_vl import Qwen2VLModel

# ==============================================================================
# CONFIG
# ==============================================================================
IMPORTANCE_VARIANT = 'attention_score_colbert_paper_v13_fixed'
TOPK_RATIOS        = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
N_LAST_LAYERS_LIST = [1, 2, 4, 8]
SCORING_MODES      = ['trad', 'weighted']

WORKING_DIR      = "/kaggle/working"
ANNOTATIONS_PATH = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_annotations.jsonl"
COLSMOL_DIR      = "/kaggle/input/datasets/nguyenducdung1107/colsmol500m-layoutmmdoc/colsmol500m-pkl"
QUERY_BATCH_SIZE = 50

BATCH_RANGE_PKL_OVERRIDE = {
    (0, 25): "/kaggle/input/datasets/nguyenducdung1107/vvvvvvv/colqwen2_fused_index (2).pkl",
}

_COLQWEN2_FORWARD_STRIP = {
    'output_hidden_states', 'return_dict', 'use_cache', 'output_attentions'
}
_QWEN2VL_ALLOWED_KEYS = {
    'input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw',
    'video_grid_thw', 'pixel_values_videos', 'position_ids',
    'past_key_values', 'inputs_embeds', 'use_cache',
    'output_attentions', 'output_hidden_states', 'return_dict',
}


# ==============================================================================
# UTILS
# ==============================================================================
def build_content_mask(inputs, processor):
    attn_mask = inputs["attention_mask"]
    input_ids = inputs.get("input_ids", None)
    if input_ids is None:
        return attn_mask.float()
    special_ids = set()
    tok = getattr(processor, 'tokenizer', processor)
    for attr in ['pad_token_id', 'bos_token_id', 'eos_token_id',
                 'unk_token_id', 'sep_token_id', 'cls_token_id']:
        tid = getattr(tok, attr, None)
        if tid is not None:
            special_ids.add(int(tid))
    if hasattr(tok, 'added_tokens_encoder'):
        for _, tid in tok.added_tokens_encoder.items():
            special_ids.add(int(tid))
    if not special_ids:
        return attn_mask.float()
    special_tensor = torch.tensor(list(special_ids), device=input_ids.device)
    is_special     = (input_ids.unsqueeze(-1) == special_tensor).any(dim=-1)
    return attn_mask.float() * (~is_special).float()


# ==============================================================================
# ATTENTION SCORE (Lassance et al. 2021, Eq. 2)
# i(t) = Σ_h Σ_j (A)_{h,t,j}
#   attn: (B, H, S_q, S_k)
#   Step 1: sum over key  → .sum(dim=-1) → (B, H, S_q)
#   Step 2: sum over head → .sum(dim=1)  → (B, S_q)
# ==============================================================================
def compute_attention_score_importance(attn_list, content_mask, n_layers_used,
                                       normalize_scores=False):
    device = content_mask.device
    B, S   = content_mask.shape

    if not attn_list:
        return content_mask.clone()

    importance = torch.zeros(B, S, device=device, dtype=torch.float32)

    for attn in attn_list:
        attn    = attn.float().to(device)           # (B, H, S_q, S_k)
        row_sum = attn.sum(dim=-1).sum(dim=1)        # (B, S_q)

        S_q = row_sum.shape[-1]
        if S_q > S:
            row_sum = row_sum[:, :S]
        elif S_q < S:
            pad     = torch.zeros(B, S - S_q, device=device)
            row_sum = torch.cat([row_sum, pad], dim=-1)

        importance = importance + row_sum

    importance = importance / max(n_layers_used, 1)
    importance = importance * content_mask

    if normalize_scores:
        imp_max    = importance.max(dim=-1, keepdim=True).values.clamp(min=1e-8)
        importance = importance / imp_max

    return importance


# ==============================================================================
# EXTRACTION
# ==============================================================================
def extract_embeddings_and_attn_importance(model, inputs, processor, n_last_layers=1):
    colqwen2_inputs = {k: v for k, v in inputs.items() if k not in _COLQWEN2_FORWARD_STRIP}
    inner_inputs    = {k: v for k, v in inputs.items()
                       if k in _QWEN2VL_ALLOWED_KEYS and k not in _COLQWEN2_FORWARD_STRIP}

    try:
        with torch.no_grad():
            proj = model(**colqwen2_inputs)
        proj = F.normalize(proj.float(), dim=-1)
        proj = proj * inputs["attention_mask"].unsqueeze(-1).float()
    except Exception as e:
        print(f"❌ ColQwen2 embedding extraction failed: {e}")
        B, S = inputs['attention_mask'].shape
        D    = getattr(model.config, 'projection_dim',
                       getattr(model.config, 'hidden_size', 128))
        proj = torch.zeros(B, S, D, device=inputs['attention_mask'].device)
        content_mask = build_content_mask(inputs, processor).to(proj.device)
        return proj, content_mask.clone(), False

    content_mask = build_content_mask(inputs, processor).to(proj.device)

    attn_list = []
    attn_ok   = False
    try:
        with torch.no_grad():
            inner_out = Qwen2VLModel.forward(
                model, **inner_inputs,
                output_attentions=True,
                output_hidden_states=False,
                use_cache=False,
                return_dict=True,
                attn_implementation="eager",
            )
        all_attentions = list(inner_out.attentions)
        attn_list = all_attentions[-n_last_layers:] if len(all_attentions) >= n_last_layers \
                    else all_attentions
        attn_ok   = len(attn_list) > 0
    except Exception as e:
        print(f"⚠️  Attention extraction failed (n={n_last_layers}): {e}")

    n_actual   = len(attn_list)
    importance = compute_attention_score_importance(attn_list, content_mask, n_actual)
    return proj, importance, attn_ok


# ==============================================================================
# DOC MATRIX
# ==============================================================================
def _coerce_to_tensor(item):
    if isinstance(item, torch.Tensor):
        return item
    if isinstance(item, dict):
        for key in ('embedding', 'embeddings', 'vector', 'vectors', 'token_embeddings'):
            if key in item:
                v = item[key]
                if isinstance(v, torch.Tensor):
                    return v
        tensor_vals = [(k, v) for k, v in item.items() if isinstance(v, torch.Tensor)]
        if len(tensor_vals) == 1:
            return tensor_vals[0][1]
        if tensor_vals:
            return tensor_vals[0][1]
    raise ValueError(f"Cannot extract Tensor from item of type {type(item)}.")


def build_doc_matrix(docs_emb, device):
    tensors = []
    for i, item in enumerate(docs_emb):
        try:
            t = _coerce_to_tensor(item)
        except ValueError as e:
            raise ValueError(f"docs_emb[{i}]: {e}") from e
        if t.dim() == 1:
            t = t.unsqueeze(0)
        elif t.dim() > 2:
            t = t.squeeze(0)
        tensors.append(t)
    n_docs  = len(tensors)
    max_len = max(t.shape[0] for t in tensors)
    D       = tensors[0].shape[1]
    doc_matrix = torch.zeros(n_docs, max_len, D, device=device)
    doc_mask   = torch.zeros(n_docs, max_len,    device=device, dtype=torch.bool)
    for i, t in enumerate(tensors):
        L = t.shape[0]
        doc_matrix[i, :L] = F.normalize(t.float().to(device), dim=-1)
        doc_mask[i, :L]   = True
    return doc_matrix, doc_mask


@torch.no_grad()
def fast_maxsim(q_norm, doc_matrix, doc_mask):
    """q_norm: (S_q, D) → returns (S_q, n_docs)"""
    sim = torch.einsum('qd,nld->qnl', q_norm, doc_matrix)
    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
    return sim.max(dim=-1).values


# ==============================================================================
# ATTENTION-SCORE PRUNING
# ==============================================================================
def attention_prune_scores(q_embed, method_idx, importance_1d,
                           doc_matrix, doc_mask, topk_ratios, scoring='trad'):
    M      = method_idx.numel()
    n_docs = doc_matrix.shape[0]

    imp_content = importance_1d[method_idx].float()
    q_vecs      = F.normalize(q_embed[method_idx].float(), dim=-1)
    M_full      = fast_maxsim(q_vecs, doc_matrix, doc_mask)    # (M, n_docs)
    sorted_idx  = torch.argsort(imp_content, descending=True)

    results = {}
    for ratio in topk_ratios:
        n_keep = max(1, round(M * ratio))
        kept   = sorted_idx[:n_keep]
        M_kept = M_full[kept]                                   # (K, n_docs)

        if scoring == 'trad':
            scores = M_kept.sum(dim=0)
        else:
            imp_kept   = imp_content[kept]
            imp_kept_n = imp_kept / imp_kept.sum().clamp(min=1e-8)
            scores     = (M_kept * imp_kept_n.unsqueeze(-1)).sum(dim=0)

        results[ratio] = scores
    return results


# ==============================================================================
# METRICS
# Recall@K = |topK ∩ GT| / |GT|  per query
# ==============================================================================
import numpy as np

def compute_ndcg(ranked_indices, gt_set, k):
    dcg  = sum(1.0 / np.log2(r + 2) for r, idx in enumerate(ranked_indices[:k]) if idx in gt_set)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt_set), k)))
    return dcg / idcg if idcg > 0 else 0.0

def first_hit(top_k, gt_set):
    for r, idx in enumerate(top_k):
        if idx in gt_set: return r + 1
    return -1

def pretty_token(t):
    t = str(t).replace("\n", "\\n").replace(" ", "_")
    return t if len(t) <= 24 else (t[:22] + "..")

def hit_metrics(top_k_list, gt_set):
    """
    Recall@K = |topK ∩ GT| / |GT|  per query
    Hit@K    = 1 nếu ≥1 GT trong top-K
    """
    gt_size = max(len(gt_set), 1)
    h = first_hit(top_k_list, gt_set)

    hits1  = sum(1 for idx in top_k_list[:1]  if idx in gt_set)
    hits5  = sum(1 for idx in top_k_list[:5]  if idx in gt_set)
    hits10 = sum(1 for idx in top_k_list[:10] if idx in gt_set)

    return {
        'r1':       int(h != -1 and h <= 1),
        'r5':       int(h != -1 and h <= 5),
        'r10':      int(h != -1 and h <= 10),
        'recall1':  hits1  / gt_size,
        'recall5':  hits5  / gt_size,
        'recall10': hits10 / gt_size,
        'n1':       float(compute_ndcg(top_k_list, gt_set, 1)),
        'n5':       float(compute_ndcg(top_k_list, gt_set, 5)),
        'n10':      float(compute_ndcg(top_k_list, gt_set, 10)),
    }

print("✅ Core functions defined")


# ==============================================================================
# METHOD KEYS
# ==============================================================================
def make_method_keys():
    keys = ['traditional', 'trad_weighted']
    for n in N_LAST_LAYERS_LIST:
        for scoring in SCORING_MODES:
            for r in TOPK_RATIOS:
                keys.append(f"attn_L{n}_{scoring}_r{int(r*100)}")
    return keys

METHOD_KEYS = make_method_keys()


# ==============================================================================
# METRIC STORE
# ==============================================================================
def _init_metric():
    return {'r1': 0, 'r5': 0, 'r10': 0,
            'n1': 0., 'n5': 0., 'n10': 0.,
            'recall1': 0., 'recall5': 0., 'recall10': 0.,
            'count': 0}

def _add_metric(dst, src):
    for f in ('r1', 'r5', 'r10'):
        dst[f] += int(src[f])
    for f in ('n1', 'n5', 'n10', 'recall1', 'recall5', 'recall10'):
        dst[f] += float(src[f])
    dst['count'] += 1

def _ensure(store, key):
    if key not in store: store[key] = _init_metric()
    return store[key]

def record(key, m, domain):
    _add_metric(_ensure(all_metrics, key), m)
    _add_metric(_ensure(all_domain_metrics[domain], key), m)


# ==============================================================================
# MASTER STORAGE
# ==============================================================================
all_query_results  = []
all_token_results  = []
all_metrics        = {}
all_domain_metrics = {}
all_batch_stats    = []

import json, os, pickle, gc, glob
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm

print(">>> BƯỚC 4: Process All Batches — Attention Score Pruning")
print(f"n_last_layers : {N_LAST_LAYERS_LIST}")
print(f"scoring modes : {SCORING_MODES}")
print(f"TOPK_RATIOS   : {TOPK_RATIOS}")
print("=" * 80)

for batch_num, (START_IDX, END_IDX) in enumerate(BATCH_RANGES, 1):
    print(f"\n[{batch_num}/{len(BATCH_RANGES)}] Batch [{START_IDX}:{END_IDX}]")

    # ── Resolve index path ────────────────────────────────────────────────────
    batch_key  = (START_IDX, END_IDX)
    index_path = BATCH_RANGE_PKL_OVERRIDE.get(batch_key, None)

    if index_path is not None:
        print(f"  ℹ️  Using PKL override: {index_path}")
    else:
        for _pkl in pkl_files:
            try:
                _nums = os.path.basename(_pkl).replace('.pkl', '').split(' ')[-1]
                _s, _e = map(int, _nums.split('-'))
                if _s == START_IDX and _e == END_IDX:
                    index_path = _pkl; break
            except Exception: continue
        if index_path is None:
            _candidate = os.path.join(COLSMOL_DIR, f"{START_IDX}-{END_IDX}.pkl")
            if os.path.exists(_candidate):
                index_path = _candidate

    if index_path is None or not os.path.exists(index_path):
        print(f"  ⚠️  Cannot find PKL for [{START_IDX}:{END_IDX}]")
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}", 'status': 'index_not_found'})
        continue

    try:
        with open(index_path, 'rb') as f:
            saved = pickle.load(f)
        fused_embeddings = saved.get('embeddings', []) if isinstance(saved, dict) else \
                           (saved if isinstance(saved, list) else saved)
        print(f"  ✅ {len(fused_embeddings)} layouts")
    except Exception as e:
        print(f"  ❌ {e}"); continue

    # ── Load batch data ────────────────────────────────────────────────────────
    print("  Loading batch data...")
    try:
        batch_doc_names    = intersection_docs[START_IDX:END_IDX]
        batch_target_files = [jsonl_map[d] for d in batch_doc_names]

        batch_df_orig = pd.read_parquet(PARQUET_PATH)
        batch_df_orig['join_doc_name'] = batch_df_orig['doc_name'].str.replace('.pdf', '', regex=False)
        batch_df_orig = batch_df_orig[batch_df_orig['join_doc_name'].isin(batch_doc_names)]

        dfs = []
        for f in tqdm(batch_target_files, desc="    Reading JSONLs", leave=False):
            try:
                temp = pd.read_json(f, lines=True)
                temp['join_doc_name'] = os.path.basename(f).replace('_layout.jsonl', '')
                if 'layout' in temp.columns:
                    temp = temp.rename(columns={'layout': 'layout_id'})
                cols = ['join_doc_name', 'layout_id', 'vlm_text', 'img_enhanced_path']
                if 'text_level' in temp.columns: cols.append('text_level')
                dfs.append(temp[[c for c in cols if c in temp.columns]])
            except Exception: pass

        batch_df_enh = pd.concat(dfs, ignore_index=True).rename(
            columns={'vlm_text': 'vlm_text_enhanced', 'text_level': 'text_level_enhanced'}
        ) if dfs else pd.DataFrame()

        batch_df_final = pd.merge(
            batch_df_orig, batch_df_enh, on=['join_doc_name', 'layout_id'], how='left'
        ).sort_values(['join_doc_name', 'page_id', 'layout_id'])

        is_header = batch_df_final['type'].isin(['title', 'section_header', 'header']) | \
                    batch_df_final.get('text_level_enhanced', pd.Series(dtype=object)).notna()
        batch_df_final['temp_header']     = batch_df_final['text'].where(is_header)
        batch_df_final['current_section'] = (
            batch_df_final.groupby('join_doc_name')['temp_header'].ffill().fillna("General Content")
        )

        enh_image_map = {}
        if os.path.exists(ENHANCED_IMG_DIR):
            for fn in glob.glob(os.path.join(ENHANCED_IMG_DIR, "*")):
                enh_image_map[os.path.basename(fn)] = fn

        df = batch_df_final.copy()
        if 'img_enhanced_path' in df.columns:
            df['img_data'] = df['img_enhanced_path'].dropna().map(
                lambda p: enh_image_map.get(os.path.basename(str(p))))
            df['img_type'] = df['img_data'].notna().map(lambda x: 'path' if x else None)
        else:
            df['img_data'] = None; df['img_type'] = None

        if 'image_binary' in df.columns:
            need = df['img_type'].isna() & df['image_binary'].notna()
            df.loc[need, 'img_type'] = 'binary'
            df.loc[need, 'img_data'] = df.loc[need, 'image_binary']

        def _pick_text(row):
            for col in ['vlm_text_enhanced', 'text', 'ocr_text', 'vlm_text']:
                v = row.get(col)
                if pd.notna(v) and len(str(v)) > 5: return str(v)
            return "Document layout."

        df['final_text'] = "Section: " + df['current_section'].fillna('') + \
                           " \n Content: " + df.apply(_pick_text, axis=1)
        batch_sample_layouts_df = df.dropna(subset=['img_type']).reset_index(drop=True)
        print(f"  ✅ {len(batch_sample_layouts_df)} layouts")

    except Exception as e:
        print(f"  ❌ Batch data error: {e}"); import traceback; traceback.print_exc(); continue

    # ── Build QA pairs ─────────────────────────────────────────────────────────
    def calculate_iou(box1, box2):
        b1 = list(box1); b2 = list(box2)
        x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
        x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
        inter  = max(0, x2-x1) * max(0, y2-y1)
        union  = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
        return inter / union if union > 0 else 0.0

    batch_qa_pairs   = []
    col_page_qa      = 'page_idx' if 'page_idx' in batch_sample_layouts_df.columns else 'page_id'
    batch_doc_lookup = {k: v for k, v in batch_sample_layouts_df.groupby('join_doc_name')}
    batch_avail_docs = set(batch_doc_lookup.keys())

    with open(ANNOTATIONS_PATH, 'r') as f:
        for line in f:
            try: doc_data = json.loads(line)
            except Exception: continue
            target_doc = doc_data['doc_name'].replace('.pdf', '')
            if target_doc not in batch_avail_docs: continue
            doc_df = batch_doc_lookup[target_doc].copy()
            doc_df['safe_page'] = pd.to_numeric(
                doc_df[col_page_qa], errors='coerce').fillna(-999).astype(int)
            domain = doc_data.get('domain', 'General')
            for q_item in doc_data.get('questions', []):
                gt = []
                for target in q_item.get('layout_mapping', []):
                    try:
                        tp    = int(target['page'])
                        cands = pd.concat([doc_df[doc_df['safe_page'] == tp],
                                           doc_df[doc_df['safe_page'] == tp - 1]])
                        for idx, row in cands.iterrows():
                            if calculate_iou(row['bbox'], target['bbox']) > 0.5:
                                gt.append(int(idx))
                    except Exception: continue
                if gt:
                    batch_qa_pairs.append({
                        'question':   q_item['Q'],
                        'gt_indices': list(set(gt)),
                        'doc_name':   target_doc,
                        'domain':     domain,
                    })

    print(f"  ✅ {len(batch_qa_pairs)} QA pairs")
    if not batch_qa_pairs:
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}",
                                'n_layouts': len(fused_embeddings), 'n_qa': 0, 'status': 'no_qa'})
        continue

    # ── Evaluate ───────────────────────────────────────────────────────────────
    print(f"  Evaluating {len(batch_qa_pairs)} queries ...")
    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        n_docs = len(fused_embeddings)
        total  = len(batch_qa_pairs)

        doc_matrix, doc_mask = build_doc_matrix(fused_embeddings, device)
        print(f"  ✅ Doc matrix: {doc_matrix.shape}")

        batch_query_rows = []
        batch_token_rows = []

        for qb_start in range(0, total, QUERY_BATCH_SIZE):
            qb_end   = min(qb_start + QUERY_BATCH_SIZE, total)
            pbar = tqdm(enumerate(batch_qa_pairs[qb_start:qb_end]),
                        total=qb_end - qb_start,
                        desc=f"    Q[{qb_start}:{qb_end}]", leave=False)

            for local_q, item in pbar:
                global_q = qb_start + local_q
                gt_set   = set(item['gt_indices'])
                domain   = item['domain']
                if domain not in all_domain_metrics:
                    all_domain_metrics[domain] = {}

                with torch.no_grad():
                    q_inputs = processor.process_queries([item['question']]).to(device)

                attn_mask_1d    = q_inputs['attention_mask'][0].float()
                content_mask_1d = build_content_mask(q_inputs, processor)[0].float()
                trad_idx   = torch.where(attn_mask_1d > 0)[0]
                method_idx = torch.where(content_mask_1d > 0)[0]
                if trad_idx.numel() == 0: continue
                if method_idx.numel() == 0: method_idx = trad_idx

                # Encode once per n_layers config
                embed_cache = {}
                for n_layers in N_LAST_LAYERS_LIST:
                    proj, imp, _ = extract_embeddings_and_attn_importance(
                        model, q_inputs, processor, n_last_layers=n_layers
                    )
                    embed_cache[n_layers] = (proj[0].float(), imp[0].float())

                # Baseline: traditional (no pruning, uniform sum)
                q_embed_base = embed_cache[N_LAST_LAYERS_LIST[0]][0]
                q_trad_vecs  = F.normalize(q_embed_base[trad_idx].float(), dim=-1)
                M_trad       = fast_maxsim(q_trad_vecs, doc_matrix, doc_mask)

                trad_scores = M_trad.sum(dim=0)
                trad_top10  = torch.topk(trad_scores, k=min(10, n_docs)).indices.cpu().tolist()
                trad_m      = hit_metrics(trad_top10, gt_set)
                record('traditional', trad_m, domain)

                # Baseline weighted (no pruning, importance-weighted)
                imp_base      = embed_cache[N_LAST_LAYERS_LIST[0]][1]
                imp_trad      = imp_base[trad_idx]
                imp_trad_n    = imp_trad / imp_trad.sum().clamp(min=1e-8)
                trad_w_scores = (M_trad * imp_trad_n.unsqueeze(-1)).sum(dim=0)
                trad_w_top10  = torch.topk(trad_w_scores, k=min(10, n_docs)).indices.cpu().tolist()
                trad_w_m      = hit_metrics(trad_w_top10, gt_set)
                record('trad_weighted', trad_w_m, domain)

                query_row = {
                    'batch':            f"{START_IDX}-{END_IDX}",
                    'query_id':         global_q,
                    'doc_name':         item['doc_name'],
                    'domain':           domain,
                    'question':         item['question'],
                    'gt_count':         len(gt_set),
                    'trad_hit@1':       trad_m['r1'],
                    'trad_hit@5':       trad_m['r5'],
                    'trad_hit@10':      trad_m['r10'],
                    'trad_recall@1':    round(trad_m['recall1'],  6),
                    'trad_recall@5':    round(trad_m['recall5'],  6),
                    'trad_recall@10':   round(trad_m['recall10'], 6),
                    'trad_ndcg@10':     round(trad_m['n10'],      4),
                    'tradW_hit@1':      trad_w_m['r1'],
                    'tradW_hit@10':     trad_w_m['r10'],
                    'tradW_recall@10':  round(trad_w_m['recall10'], 6),
                    'tradW_ndcg@10':    round(trad_w_m['n10'],      4),
                }

                # Attention pruning per (n_layers, scoring, ratio)
                for n_layers in N_LAST_LAYERS_LIST:
                    q_embed, q_imp = embed_cache[n_layers]
                    for scoring in SCORING_MODES:
                        ratio_scores = attention_prune_scores(
                            q_embed, method_idx, q_imp,
                            doc_matrix, doc_mask,
                            topk_ratios=TOPK_RATIOS,
                            scoring=scoring,
                        )
                        for r in TOPK_RATIOS:
                            key    = f"attn_L{n_layers}_{scoring}_r{int(r*100)}"
                            scores = ratio_scores[r]
                            top10  = torch.topk(scores, k=min(10, n_docs)).indices.cpu().tolist()
                            m      = hit_metrics(top10, gt_set)
                            record(key, m, domain)
                            query_row.update({
                                f'{key}_hit@1':     m['r1'],
                                f'{key}_hit@5':     m['r5'],
                                f'{key}_hit@10':    m['r10'],
                                f'{key}_recall@1':  round(m['recall1'],  6),
                                f'{key}_recall@5':  round(m['recall5'],  6),
                                f'{key}_recall@10': round(m['recall10'], 6),
                                f'{key}_ndcg@10':   round(m['n10'],      4),
                            })

                batch_query_rows.append(query_row)

                # Token-level analysis (ref = n=1, paper-exact)
                q_imp_ref  = embed_cache[1][1]
                q_emb_ref  = embed_cache[1][0]
                q_ref_vecs = F.normalize(q_emb_ref[method_idx].float(), dim=-1)
                M_ref      = fast_maxsim(q_ref_vecs, doc_matrix, doc_mask)
                method_np  = method_idx.cpu().numpy()
                imp_np     = q_imp_ref[method_idx].cpu().numpy()
                input_ids  = q_inputs['input_ids'][0].cpu().tolist()
                tok_strs   = processor.tokenizer.convert_ids_to_tokens(input_ids)
                best_scores= M_ref.max(dim=1).values.cpu().numpy()
                for local_t, global_t in enumerate(method_np):
                    tok_raw = tok_strs[int(global_t)] if int(global_t) < len(tok_strs) else "<UNK>"
                    batch_token_rows.append({
                        'batch':           f"{START_IDX}-{END_IDX}",
                        'query_id':        global_q,
                        'token_idx':       int(global_t),
                        'token':           pretty_token(tok_raw),
                        'attn_importance': round(float(imp_np[local_t]), 8),
                        'maxsim_best':     round(float(best_scores[local_t]), 8),
                    })

            gc.collect(); torch.cuda.empty_cache()
            print(f"      ✓ Q[{qb_start}:{qb_end}] done")

        all_query_results.extend(batch_query_rows)
        all_token_results.extend(batch_token_rows)

        t_m  = all_metrics.get('traditional', _init_metric())
        t_cnt = t_m['count'] if t_m['count'] > 0 else 1
        print(f"  ✅ {len(batch_query_rows)} Q | "
              f"Baseline Hit@10={t_m['r10']/t_cnt*100:.1f}%  "
              f"Recall@10={t_m['recall10']/t_cnt*100:.1f}%  "
              f"nDCG@10={t_m['n10']/t_cnt:.4f}")

        all_batch_stats.append({
            'batch':          f"{START_IDX}-{END_IDX}",
            'n_layouts':      n_docs,
            'n_queries':      total,
            'trad_hit@10':    round(t_m['r10']     / t_cnt * 100, 2),
            'trad_recall@10': round(t_m['recall10'] / t_cnt * 100, 2),
            'trad_ndcg@10':   round(t_m['n10']     / t_cnt,       4),
            'status':         'ok',
        })

    except Exception as e:
        print(f"  ❌ Eval error: {e}"); import traceback; traceback.print_exc()
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}",
                                'status': 'error', 'error': str(e)})

    # ── Save & cleanup ─────────────────────────────────────────────────────────
    try:
        if batch_query_rows:
            pd.DataFrame(batch_query_rows).to_csv(
                os.path.join(WORKING_DIR, f"batch_{START_IDX}_{END_IDX}_queries.csv"), index=False)
        if batch_token_rows:
            pd.DataFrame(batch_token_rows).to_csv(
                os.path.join(WORKING_DIR, f"batch_{START_IDX}_{END_IDX}_tokens.csv"), index=False)
        print("  ✅ Batch files saved")
    except Exception as e:
        print(f"  ❌ Save error: {e}")

    del doc_matrix, doc_mask, fused_embeddings, batch_sample_layouts_df, batch_qa_pairs
    del batch_df_orig, batch_df_enh, batch_df_final
    gc.collect(); torch.cuda.empty_cache()
    print("  ✓ Cleaned\n")


# ==============================================================================
# FINAL SUMMARY
# ==============================================================================
print("=" * 90)
print("CONSOLIDATING RESULTS")
print("=" * 90)

try:
    if all_query_results:
        df_all_q = pd.DataFrame(all_query_results)
        df_all_q.to_csv(os.path.join(WORKING_DIR, "MASTER_all_batches_queries.csv"), index=False)
        print(f"✅ {len(df_all_q)} queries saved")

    if all_token_results:
        df_all_t = pd.DataFrame(all_token_results)
        df_all_t.to_csv(os.path.join(WORKING_DIR, "MASTER_all_batches_tokens.csv"), index=False)
        print(f"✅ {len(df_all_t)} tokens saved")

    SEP = "-" * 90
    print(f"\n{'Method':<45} {'Hit@1':>7} {'Hit@10':>7} "
          f"{'Rec@1':>8} {'Rec@10':>9} {'nDCG@10':>9}")
    print(SEP)
    for method in METHOD_KEYS:
        if method not in all_metrics: continue
        m   = all_metrics[method]; cnt = m['count'] if m['count'] > 0 else 1
        print(f"{method:<45} "
              f"{m['r1']/cnt*100:6.2f}%  {m['r10']/cnt*100:6.2f}%  "
              f"{m['recall1']/cnt*100:7.2f}%  {m['recall10']/cnt*100:8.2f}%  "
              f"{m['n10']/cnt:8.4f}")

    summary_rows = []
    for method in METHOD_KEYS:
        if method not in all_metrics: continue
        m   = all_metrics[method]; cnt = m['count'] if m['count'] > 0 else 1
        summary_rows.append({
            'method':    method,
            'hit@1':     round(m['r1']      / cnt * 100, 4),
            'hit@5':     round(m['r5']      / cnt * 100, 4),
            'hit@10':    round(m['r10']     / cnt * 100, 4),
            'recall@1':  round(m['recall1'] / cnt * 100, 4),
            'recall@5':  round(m['recall5'] / cnt * 100, 4),
            'recall@10': round(m['recall10']/ cnt * 100, 4),
            'ndcg@1':    round(m['n1']      / cnt,       6),
            'ndcg@5':    round(m['n5']      / cnt,       6),
            'ndcg@10':   round(m['n10']     / cnt,       6),
        })
    pd.DataFrame(summary_rows).to_csv(
        os.path.join(WORKING_DIR, "MASTER_attn_pruning_summary.csv"), index=False)

    if all_batch_stats:
        pd.DataFrame(all_batch_stats).to_csv(
            os.path.join(WORKING_DIR, "MASTER_batch_stats.csv"), index=False)

    print("\n✅ MASTER_attn_pruning_summary.csv")
    print("✅ MASTER_batch_stats.csv")

except Exception as e:
    print(f"❌ Final error: {e}"); import traceback; traceback.print_exc()


# ==============================================================================
# VISUALIZE
# ==============================================================================
print("\n>>> BƯỚC 5: Visualize")

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker

    METRICS_PLOT   = [
        ('n10',      'nDCG@10',    False),
        ('recall10', 'Recall@10',  True),
        ('n5',       'nDCG@5',     False),
        ('recall1',  'Recall@1',   True),
    ]
    COLORS_N       = {1: '#1D9E75', 2: '#E07B39', 4: '#3B72C4', 8: '#9B59B6'}
    COLORS_SCORING = {'trad': '#1D9E75', 'weighted': '#E07B39'}
    COLOR_BASE     = '#888780'

    def _mval(key, mkey, pct):
        m = all_metrics.get(key)
        if m is None or m['count'] == 0: return None
        v = m[mkey] / m['count']
        return v * 100 if pct else v

    # Figure 1: per n_last_layers, scoring=trad
    fig1, axes1 = plt.subplots(1, 2, figsize=(13, 5))
    for ai, (mkey, title, pct_flag) in enumerate(METRICS_PLOT[:2]):
        ax = axes1[ai]
        ax.set_title(f"Attention Score Pruning — {title}\n(scoring=trad)", fontsize=11)
        ax.set_xlabel('Top-k ratio', fontsize=10)
        ax.set_ylabel('Recall (%)' if pct_flag else 'Score', fontsize=10)
        ax.grid(axis='y', alpha=0.25); ax.spines[['top', 'right']].set_visible(False)
        base_v = _mval('traditional', mkey, pct_flag)
        if base_v is not None:
            ax.axhline(base_v, color=COLOR_BASE, linewidth=1.8, linestyle='--',
                       label='Baseline', zorder=2)
        for n_layers in N_LAST_LAYERS_LIST:
            vals  = [_mval(f"attn_L{n_layers}_trad_r{int(r*100)}", mkey, pct_flag) for r in TOPK_RATIOS]
            valid = [(r, v) for r, v in zip(TOPK_RATIOS, vals) if v is not None]
            if not valid: continue
            xs, ys = zip(*valid)
            ax.plot(xs, ys, color=COLORS_N[n_layers], linewidth=2.0 if n_layers > 1 else 2.8,
                    marker='o', markersize=5,
                    label=f"L={n_layers}" + (" ★" if n_layers == 1 else ""))
        ax.set_xticks(TOPK_RATIOS); ax.set_xticklabels([str(r) for r in TOPK_RATIOS], fontsize=8)
        fmt = (lambda v, _: f"{v:.1f}%") if pct_flag else (lambda v, _: f"{v:.3f}")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt))
        ax.legend(fontsize=8, framealpha=0.4, loc='lower right')
    plt.suptitle("Attention Score Pruning — effect of n_last_layers\n"
                 "Recall@K = hits@K / |GT| per query", fontsize=11, y=1.02)
    plt.tight_layout()
    p1 = os.path.join(WORKING_DIR, "attn_pruning_main_nlayers.png")
    plt.savefig(p1, dpi=150, bbox_inches='tight'); plt.close()
    print(f"✅ Plot 1 → {p1}")

    # Figure 2: scoring mode comparison (n=1)
    fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5))
    for ai, (mkey, title, pct_flag) in enumerate(METRICS_PLOT[:2]):
        ax = axes2[ai]
        ax.set_title(f"Scoring Mode — {title}\n(n_last_layers=1)", fontsize=11)
        ax.set_xlabel('Top-k ratio', fontsize=10)
        ax.set_ylabel('Recall (%)' if pct_flag else 'Score', fontsize=10)
        ax.grid(axis='y', alpha=0.25); ax.spines[['top', 'right']].set_visible(False)
        base_v = _mval('traditional', mkey, pct_flag)
        if base_v is not None:
            ax.axhline(base_v, color=COLOR_BASE, linewidth=1.8, linestyle='--', label='Baseline')
        for scoring in SCORING_MODES:
            vals  = [_mval(f"attn_L1_{scoring}_r{int(r*100)}", mkey, pct_flag) for r in TOPK_RATIOS]
            valid = [(r, v) for r, v in zip(TOPK_RATIOS, vals) if v is not None]
            if not valid: continue
            xs, ys = zip(*valid)
            label  = 'Traditional Sum' if scoring == 'trad' else 'Importance-Weighted'
            ax.plot(xs, ys, color=COLORS_SCORING[scoring], linewidth=2.2,
                    marker='o', markersize=5, label=label)
        ax.set_xticks(TOPK_RATIOS); ax.set_xticklabels([str(r) for r in TOPK_RATIOS], fontsize=8)
        fmt = (lambda v, _: f"{v:.1f}%") if pct_flag else (lambda v, _: f"{v:.3f}")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt))
        ax.legend(fontsize=9, framealpha=0.4)
    plt.suptitle("Attention Score Pruning: trad vs weighted scoring (n=1)\n"
                 "Recall@K = hits@K / |GT| per query", fontsize=11, y=1.02)
    plt.tight_layout()
    p2 = os.path.join(WORKING_DIR, "attn_pruning_scoring_comparison.png")
    plt.savefig(p2, dpi=150, bbox_inches='tight'); plt.close()
    print(f"✅ Plot 2 → {p2}")

    # Figure 3: 4-panel overview
    fig3, axes3 = plt.subplots(2, 2, figsize=(13, 9))
    axes3 = axes3.flatten()
    for ai, (mkey, title, pct_flag) in enumerate(METRICS_PLOT):
        ax = axes3[ai]
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('Top-k ratio', fontsize=10)
        ax.set_ylabel('Recall (%)' if pct_flag else 'Score', fontsize=10)
        ax.grid(axis='y', alpha=0.25); ax.spines[['top', 'right']].set_visible(False)
        base_v = _mval('traditional', mkey, pct_flag)
        if base_v is not None:
            ax.axhline(base_v, color=COLOR_BASE, linewidth=1.8, linestyle='--', label='Baseline')
        vals  = [_mval(f"attn_L1_trad_r{int(r*100)}", mkey, pct_flag) for r in TOPK_RATIOS]
        valid = [(r, v) for r, v in zip(TOPK_RATIOS, vals) if v is not None]
        if valid:
            xs, ys = zip(*valid)
            ax.plot(xs, ys, color='#1DB954', linewidth=2.2, marker='o', markersize=5,
                    label='Attention Score Pruning (n=1)')
        ax.set_xticks(TOPK_RATIOS); ax.set_xticklabels([str(r) for r in TOPK_RATIOS], fontsize=8)
        fmt = (lambda v, _: f"{v:.1f}%") if pct_flag else (lambda v, _: f"{v:.3f}")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt))
        ax.legend(fontsize=8, framealpha=0.4)
    plt.suptitle("Token Pruning with Attention Score vs Baseline (n=1)\n"
                 "Recall@K = hits@K / |GT| per query", fontsize=11, y=1.02)
    plt.tight_layout()
    p3 = os.path.join(WORKING_DIR, "attn_pruning_4panel_overview.png")
    plt.savefig(p3, dpi=150, bbox_inches='tight'); plt.close()
    print(f"✅ Plot 3 → {p3}")

except Exception as e:
    print(f"⚠️  Plot error: {e}"); import traceback; traceback.print_exc()

print("\n>>> BƯỚC 3 v13-AttentionScore [FIXED] DONE")

>>> BƯỚC 3 v13-AttentionScore [FIXED]: Token Pruning — correct Recall
✅ Core functions defined
>>> BƯỚC 4: Process All Batches — Attention Score Pruning
n_last_layers : [1, 2, 4, 8]
scoring modes : ['trad', 'weighted']
TOPK_RATIOS   : [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

[1/13] Batch [0:25]
  ℹ️  Using PKL override: /kaggle/input/datasets/nguyenducdung1107/vvvvvvv/colqwen2_fused_index (2).pkl
  ✅ 6850 layouts
  Loading batch data...


    Reading JSONLs:   0%|          | 0/25 [00:00<?, ?it/s]

  ✅ 6850 layouts
  ✅ 122 QA pairs
  Evaluating 122 queries ...
  ✅ Doc matrix: torch.Size([6850, 1471, 128])


    Q[0:50]:   0%|          | 0/50 [00:00<?, ?it/s]

      ✓ Q[0:50] done


...

  ✅ 8399 layouts
  ✅ 59 QA pairs
  Evaluating 59 queries ...
  ✅ Doc matrix: torch.Size([8399, 1499, 128])


    Q[0:50]:   0%|          | 0/50 [00:00<?, ?it/s]

      ✓ Q[0:50] done


    Q[50:59]:   0%|          | 0/9 [00:00<?, ?it/s]

      ✓ Q[50:59] done
  ✅ 59 Q | Baseline Hit@10=71.8%  Recall@10=59.7%  nDCG@10=0.4962
  ✅ Batch files saved
  ✓ Cleaned

CONSOLIDATING RESULTS
✅ 1603 queries saved
✅ 31179 tokens saved

Method                                          Hit@1  Hit@10    Rec@1    Rec@10   nDCG@10
------------------------------------------------------------------------------------------
traditional                                    42.55%   71.80%    31.84%     59.66%    0.4962
trad_weighted                                  41.80%   70.31%    31.17%     58.84%    0.4872
attn_L1_trad_r10                               23.83%   49.22%    17.17%     39.43%    0.3018
attn_L1_trad_r20                               32.63%   60.32%    24.31%     49.09%    0.3930
attn_L1_trad_r30                               35.87%   64.38%    26.92%     53.25%    0.4288
attn_L1_trad_r40                               36.99%   66.94%    27.63%     55.52%    0.4476
attn_L1_trad_r50                               38.74%   68.00%    

**4.random**

In [15]:
# ==============================================================================
# BƯỚC 3 v14-RandomPruning – Random Token Pruning (Ablation/Baseline)
#
# Recall@K chuẩn: hits@K / |GT|  (per query, mean over queries)
# ==============================================================================

print(">>> BƯỚC 3 v14-RandomPruning [FIXED]: Random Token Pruning")

import torch
import torch.nn.functional as F

# ==============================================================================
# CONFIG
# ==============================================================================
IMPORTANCE_VARIANT = 'random_pruning_topk_v14_fixed'
TOPK_RATIOS        = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
N_RANDOM_SEEDS     = 1

WORKING_DIR      = "/kaggle/working"
ANNOTATIONS_PATH = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_annotations.jsonl"
COLSMOL_DIR      = "/kaggle/input/datasets/nguyenducdung1107/colsmol500m-layoutmmdoc/colsmol500m-pkl"
QUERY_BATCH_SIZE = 50

BATCH_RANGE_PKL_OVERRIDE = {
    (0, 25): "/kaggle/input/datasets/nguyenducdung1107/vvvvvvv/colqwen2_fused_index (2).pkl",
}


# ==============================================================================
# UTILS
# ==============================================================================
def build_content_mask(inputs, processor):
    attn_mask = inputs["attention_mask"]
    input_ids = inputs.get("input_ids", None)
    if input_ids is None:
        return attn_mask.float()
    special_ids = set()
    tok = getattr(processor, 'tokenizer', processor)
    for attr in ['pad_token_id', 'bos_token_id', 'eos_token_id',
                 'unk_token_id', 'sep_token_id', 'cls_token_id']:
        tid = getattr(tok, attr, None)
        if tid is not None:
            special_ids.add(int(tid))
    if hasattr(tok, 'added_tokens_encoder'):
        for _, tid in tok.added_tokens_encoder.items():
            special_ids.add(int(tid))
    if not special_ids:
        return attn_mask.float()
    special_tensor = torch.tensor(list(special_ids), device=input_ids.device)
    is_special     = (input_ids.unsqueeze(-1) == special_tensor).any(dim=-1)
    return attn_mask.float() * (~is_special).float()


# ==============================================================================
# DOC MATRIX — handle dict và Tensor
# ==============================================================================
def _coerce_to_tensor(item):
    if isinstance(item, torch.Tensor):
        return item
    if isinstance(item, dict):
        for key in ('embedding', 'embeddings', 'vector', 'vectors', 'token_embeddings'):
            if key in item:
                v = item[key]
                if isinstance(v, torch.Tensor):
                    return v
        tensor_vals = [(k, v) for k, v in item.items() if isinstance(v, torch.Tensor)]
        if len(tensor_vals) == 1:
            return tensor_vals[0][1]
        if tensor_vals:
            return tensor_vals[0][1]
    raise ValueError(f"Cannot extract Tensor from item of type {type(item)}.")


def precompute_docs(docs_list, device):
    tensors = []
    for i, item in enumerate(docs_list):
        try:
            t = _coerce_to_tensor(item)
        except ValueError as e:
            raise ValueError(f"docs_list[{i}]: {e}") from e
        if t.dim() == 1:
            t = t.unsqueeze(0)
        elif t.dim() > 2:
            t = t.squeeze(0)
        tensors.append(t)
    n_docs  = len(tensors)
    max_len = max(t.shape[0] for t in tensors)
    D       = tensors[0].shape[1]
    doc_pad  = torch.zeros(n_docs, max_len, D, device=device, dtype=torch.float32)
    doc_mask = torch.zeros(n_docs, max_len,    device=device, dtype=torch.bool)
    for i, t in enumerate(tensors):
        L = t.shape[0]
        doc_pad[i, :L]  = F.normalize(t.float().to(device), dim=-1)
        doc_mask[i, :L] = True
    return doc_pad, doc_mask


# ==============================================================================
# MAXSIM SCORING
# ==============================================================================
def token_doc_maxsim_matrix_precomp(q_norm, doc_pad, doc_mask):
    sim = torch.einsum('md,nld->mnl', q_norm, doc_pad)
    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
    return sim.max(dim=-1).values.sum(dim=0)


# ==============================================================================
# RANDOM PRUNING
# ==============================================================================
def random_prune_topk(q_embed, content_mask, doc_pad, doc_mask,
                      topk_ratios, n_seeds=N_RANDOM_SEEDS):
    content_idx = torch.where(content_mask > 0)[0]
    M           = content_idx.numel()
    n_docs      = doc_pad.shape[0]
    device      = q_embed.device

    if M == 0:
        return {r: torch.zeros(n_docs, device=device) for r in topk_ratios}

    content_vecs = F.normalize(q_embed[content_idx].float(), dim=-1)

    results = {}
    for ratio in topk_ratios:
        k = max(1, min(round(M * ratio), M))
        if k == M:
            results[ratio] = token_doc_maxsim_matrix_precomp(content_vecs, doc_pad, doc_mask)
            continue
        accumulated = torch.zeros(n_docs, device=device, dtype=torch.float32)
        for seed in range(n_seeds):
            perm        = torch.randperm(M, device=device)[:k]
            pruned      = content_vecs[perm]
            accumulated += token_doc_maxsim_matrix_precomp(pruned, doc_pad, doc_mask)
        results[ratio] = accumulated / n_seeds
    return results


# ==============================================================================
# EXTRACTION
# ==============================================================================
def extract_embeddings_v14_random(model, inputs, processor):
    _STRIP = {'output_hidden_states', 'return_dict', 'use_cache', 'output_attentions'}
    colqwen2_inputs = {k: v for k, v in inputs.items() if k not in _STRIP}
    try:
        with torch.no_grad():
            outputs = model(**colqwen2_inputs)
        if isinstance(outputs, torch.Tensor):
            proj = outputs
        elif hasattr(outputs, 'last_hidden_state'):
            proj = outputs.last_hidden_state
        else:
            proj = outputs[0]
        proj = proj / (proj.norm(dim=-1, keepdim=True) + 1e-8)
        proj = proj * inputs["attention_mask"].unsqueeze(-1).float()
        return proj
    except Exception as e:
        print(f"❌ Extraction error: {e}")
        B, S = inputs['input_ids'].shape
        D    = getattr(model.config, 'projection_dim',
                       getattr(model.config, 'hidden_size', 128))
        return torch.zeros(B, S, D, device=inputs['input_ids'].device)


# ==============================================================================
# METRICS
# Recall@K = |topK ∩ GT| / |GT|  per query
# ==============================================================================
import numpy as np

def compute_ndcg(ranked_indices, gt_set, k):
    dcg  = sum(1.0 / np.log2(r + 2) for r, idx in enumerate(ranked_indices[:k]) if idx in gt_set)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt_set), k)))
    return dcg / idcg if idcg > 0 else 0.0

def first_hit(top_k, gt_set):
    for r, idx in enumerate(top_k):
        if idx in gt_set: return r + 1
    return -1

def hit_metrics(top_k_list, gt_set):
    """
    Recall@K = |topK ∩ GT| / |GT|  per query
    Hit@K    = 1 nếu ≥1 GT trong top-K
    """
    gt_size = max(len(gt_set), 1)
    h = first_hit(top_k_list, gt_set)

    hits1  = sum(1 for idx in top_k_list[:1]  if idx in gt_set)
    hits5  = sum(1 for idx in top_k_list[:5]  if idx in gt_set)
    hits10 = sum(1 for idx in top_k_list[:10] if idx in gt_set)

    return {
        'r1':       int(h != -1 and h <= 1),
        'r5':       int(h != -1 and h <= 5),
        'r10':      int(h != -1 and h <= 10),
        'recall1':  hits1  / gt_size,
        'recall5':  hits5  / gt_size,
        'recall10': hits10 / gt_size,
        'n1':       float(compute_ndcg(top_k_list, gt_set, 1)),
        'n5':       float(compute_ndcg(top_k_list, gt_set, 5)),
        'n10':      float(compute_ndcg(top_k_list, gt_set, 10)),
    }

print("✅ Utility functions loaded")


# ==============================================================================
# METHOD KEYS
# ==============================================================================
METHOD_RAND = ['traditional'] + [f"rand_r{int(r*100)}" for r in TOPK_RATIOS]


# ==============================================================================
# LATENCY TRACKER
# ==============================================================================
class LatencyTracker:
    def __init__(self):
        self.ratio_latency = {}

    def add_ratio(self, ratio, score_ms):
        ratio = float(ratio)
        if ratio not in self.ratio_latency:
            self.ratio_latency[ratio] = []
        self.ratio_latency[ratio].append(float(score_ms))

    def report(self):
        rows = []
        for ratio in sorted(self.ratio_latency.keys()):
            values = np.array(self.ratio_latency[ratio], dtype=np.float32)
            rows.append({
                'ratio':     ratio,
                'avg_ms':    float(values.mean())            if len(values) else float('nan'),
                'p50_ms':    float(np.percentile(values, 50)) if len(values) else float('nan'),
                'p95_ms':    float(np.percentile(values, 95)) if len(values) else float('nan'),
                'n_queries': int(len(values)),
            })
        return pd.DataFrame(rows)


random_latency_tracker = LatencyTracker()


# ==============================================================================
# METRIC STORE
# ==============================================================================
def _init_metric():
    return {'r1': 0, 'r5': 0, 'r10': 0,
            'n1': 0., 'n5': 0., 'n10': 0.,
            'recall1': 0., 'recall5': 0., 'recall10': 0.,
            'count': 0}

def _add_metric(dst, src):
    for f in ('r1', 'r5', 'r10'):
        dst[f] += int(src[f])
    for f in ('n1', 'n5', 'n10', 'recall1', 'recall5', 'recall10'):
        dst[f] += float(src[f])
    dst['count'] += 1

def _ensure(store, key):
    if key not in store: store[key] = _init_metric()
    return store[key]

def record(key, m, domain):
    _add_metric(_ensure(all_metrics, key), m)
    _add_metric(_ensure(all_domain_metrics[domain], key), m)


# ==============================================================================
# MASTER STORAGE
# ==============================================================================
all_query_results  = []
all_metrics        = {}
all_domain_metrics = {}
all_batch_stats    = []

import json, os, pickle, gc, glob, time
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm

print("\n>>> BƯỚC 4: Process All Batches (Random Pruning)")
print(f"Processing {len(BATCH_RANGES)} batches | seeds={N_RANDOM_SEEDS}")
print("=" * 80)

for batch_num, (START_IDX, END_IDX) in enumerate(BATCH_RANGES, 1):
    print(f"\n[{batch_num}/{len(BATCH_RANGES)}] Batch [{START_IDX}:{END_IDX}]")

    # ── Resolve index path ────────────────────────────────────────────────────
    batch_key  = (START_IDX, END_IDX)
    index_path = BATCH_RANGE_PKL_OVERRIDE.get(batch_key, None)
    if index_path is not None:
        print(f"  ℹ️  PKL override: {index_path}")
    else:
        for _pkl in pkl_files:
            try:
                _nums = os.path.basename(_pkl).replace('.pkl', '').split(' ')[-1]
                _s, _e = map(int, _nums.split('-'))
                if _s == START_IDX and _e == END_IDX:
                    index_path = _pkl; break
            except Exception: continue
        if index_path is None:
            _candidate = os.path.join(COLSMOL_DIR, f"{START_IDX}-{END_IDX}.pkl")
            if os.path.exists(_candidate):
                index_path = _candidate

    if index_path is None or not os.path.exists(index_path):
        print(f"  ⚠️  Cannot find PKL for [{START_IDX}:{END_IDX}]")
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}", 'status': 'index_not_found'})
        continue

    try:
        with open(index_path, 'rb') as f:
            saved = pickle.load(f)
        fused_embeddings = saved.get('embeddings', []) if isinstance(saved, dict) else \
                           (saved if isinstance(saved, list) else saved)
        print(f"  ✅ {len(fused_embeddings)} layouts")
    except Exception as e:
        print(f"  ❌ {e}"); continue

    # ── Load batch data ────────────────────────────────────────────────────────
    print("  Loading batch data...")
    try:
        batch_doc_names    = intersection_docs[START_IDX:END_IDX]
        batch_target_files = [jsonl_map[d] for d in batch_doc_names]

        batch_df_orig = pd.read_parquet(PARQUET_PATH)
        batch_df_orig['join_doc_name'] = batch_df_orig['doc_name'].str.replace('.pdf', '', regex=False)
        batch_df_orig = batch_df_orig[batch_df_orig['join_doc_name'].isin(batch_doc_names)]

        dfs = []
        for f in tqdm(batch_target_files, desc="    Reading JSONLs", leave=False):
            try:
                temp = pd.read_json(f, lines=True)
                temp['join_doc_name'] = os.path.basename(f).replace('_layout.jsonl', '')
                if 'layout' in temp.columns:
                    temp = temp.rename(columns={'layout': 'layout_id'})
                cols = ['join_doc_name', 'layout_id', 'vlm_text', 'img_enhanced_path']
                if 'text_level' in temp.columns: cols.append('text_level')
                dfs.append(temp[[c for c in cols if c in temp.columns]])
            except Exception: pass

        batch_df_enh = pd.concat(dfs, ignore_index=True).rename(
            columns={'vlm_text': 'vlm_text_enhanced', 'text_level': 'text_level_enhanced'}
        ) if dfs else pd.DataFrame()

        batch_df_final = pd.merge(
            batch_df_orig, batch_df_enh, on=['join_doc_name', 'layout_id'], how='left'
        ).sort_values(['join_doc_name', 'page_id', 'layout_id'])

        is_header = batch_df_final['type'].isin(['title', 'section_header', 'header']) | \
                    batch_df_final.get('text_level_enhanced', pd.Series(dtype=object)).notna()
        batch_df_final['temp_header']     = batch_df_final['text'].where(is_header)
        batch_df_final['current_section'] = (
            batch_df_final.groupby('join_doc_name')['temp_header'].ffill().fillna("General Content")
        )

        enh_image_map = {}
        if os.path.exists(ENHANCED_IMG_DIR):
            for fn in glob.glob(os.path.join(ENHANCED_IMG_DIR, "*")):
                enh_image_map[os.path.basename(fn)] = fn

        df = batch_df_final.copy()
        if 'img_enhanced_path' in df.columns:
            df['img_data'] = df['img_enhanced_path'].dropna().map(
                lambda p: enh_image_map.get(os.path.basename(str(p))))
            df['img_type'] = df['img_data'].notna().map(lambda x: 'path' if x else None)
        else:
            df['img_data'] = None; df['img_type'] = None

        if 'image_binary' in df.columns:
            need = df['img_type'].isna() & df['image_binary'].notna()
            df.loc[need, 'img_type'] = 'binary'
            df.loc[need, 'img_data'] = df.loc[need, 'image_binary']

        def _pick_text(row):
            for col in ['vlm_text_enhanced', 'text', 'ocr_text', 'vlm_text']:
                v = row.get(col)
                if pd.notna(v) and len(str(v)) > 5: return str(v)
            return "Document layout."

        df['final_text'] = "Section: " + df['current_section'].fillna('') + \
                           " \n Content: " + df.apply(_pick_text, axis=1)
        batch_sample_layouts_df = df.dropna(subset=['img_type']).reset_index(drop=True)
        print(f"  ✅ {len(batch_sample_layouts_df)} layouts")

    except Exception as e:
        print(f"  ❌ Batch data error: {e}"); import traceback; traceback.print_exc(); continue

    # ── Build QA pairs ─────────────────────────────────────────────────────────
    def calculate_iou(box1, box2):
        b1 = list(box1); b2 = list(box2)
        x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
        x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
        inter  = max(0, x2-x1) * max(0, y2-y1)
        union  = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
        return inter / union if union > 0 else 0.0

    batch_qa_pairs   = []
    col_page_qa      = 'page_idx' if 'page_idx' in batch_sample_layouts_df.columns else 'page_id'
    batch_doc_lookup = {k: v for k, v in batch_sample_layouts_df.groupby('join_doc_name')}
    batch_avail_docs = set(batch_doc_lookup.keys())

    with open(ANNOTATIONS_PATH, 'r') as f:
        for line in f:
            try: doc_data = json.loads(line)
            except Exception: continue
            target_doc = doc_data['doc_name'].replace('.pdf', '')
            if target_doc not in batch_avail_docs: continue
            doc_df = batch_doc_lookup[target_doc].copy()
            doc_df['safe_page'] = pd.to_numeric(
                doc_df[col_page_qa], errors='coerce').fillna(-999).astype(int)
            domain = doc_data.get('domain', 'General')
            for q_item in doc_data.get('questions', []):
                gt = []
                for target in q_item.get('layout_mapping', []):
                    try:
                        tp    = int(target['page'])
                        cands = pd.concat([doc_df[doc_df['safe_page'] == tp],
                                           doc_df[doc_df['safe_page'] == tp - 1]])
                        for idx, row in cands.iterrows():
                            if calculate_iou(row['bbox'], target['bbox']) > 0.5:
                                gt.append(int(idx))
                    except Exception: continue
                if gt:
                    batch_qa_pairs.append({
                        'question':   q_item['Q'],
                        'gt_indices': list(set(gt)),
                        'doc_name':   target_doc,
                        'domain':     domain,
                    })

    print(f"  ✅ {len(batch_qa_pairs)} QA pairs")
    if not batch_qa_pairs:
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}",
                                'n_layouts': len(fused_embeddings), 'n_qa': 0, 'status': 'no_qa'})
        continue

    # ── Evaluate ───────────────────────────────────────────────────────────────
    print(f"  Evaluating {len(batch_qa_pairs)} queries...")
    try:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        n_docs = len(fused_embeddings)
        total  = len(batch_qa_pairs)

        doc_pad, doc_mask = precompute_docs(fused_embeddings, device)
        print(f"  ✅ doc_pad: {doc_pad.shape}")

        batch_query_rows = []

        for qb_start in range(0, total, QUERY_BATCH_SIZE):
            qb_end = min(qb_start + QUERY_BATCH_SIZE, total)
            pbar   = tqdm(enumerate(batch_qa_pairs[qb_start:qb_end]),
                          total=qb_end - qb_start,
                          desc=f"    Q[{qb_start}:{qb_end}]", leave=False)

            for local_q, item in pbar:
                global_q = qb_start + local_q
                gt_set   = set(item['gt_indices'])
                domain   = item['domain']
                if domain not in all_domain_metrics:
                    all_domain_metrics[domain] = {}

                q_inputs = processor.process_queries([item['question']]).to(device)
                q_embed  = extract_embeddings_v14_random(model, q_inputs, processor).float()[0]
                content_mask = build_content_mask(q_inputs, processor)[0].float()

                # Baseline: full MaxSim
                t0 = time.perf_counter()
                content_idx = torch.where(content_mask > 0)[0]
                q_trad_norm = F.normalize(q_embed[content_idx].float(), dim=-1)
                with torch.no_grad():
                    trad_scores = token_doc_maxsim_matrix_precomp(q_trad_norm, doc_pad, doc_mask)
                if device == 'cuda': torch.cuda.synchronize()
                trad_lat_ms = (time.perf_counter() - t0) * 1000.0

                trad_top10 = torch.topk(trad_scores, k=min(10, n_docs)).indices.cpu().tolist()
                trad_m     = hit_metrics(trad_top10, gt_set)
                record('traditional', trad_m, domain)
                random_latency_tracker.add_ratio(1.0, trad_lat_ms)

                query_row = {
                    'batch':          f"{START_IDX}-{END_IDX}",
                    'query_id':       global_q,
                    'doc_name':       item['doc_name'],
                    'domain':         domain,
                    'question':       item['question'],
                    'gt_count':       len(gt_set),
                    'trad_hit@1':     trad_m['r1'],
                    'trad_hit@5':     trad_m['r5'],
                    'trad_hit@10':    trad_m['r10'],
                    'trad_recall@1':  round(trad_m['recall1'],  6),
                    'trad_recall@5':  round(trad_m['recall5'],  6),
                    'trad_recall@10': round(trad_m['recall10'], 6),
                    'trad_ndcg@10':   round(trad_m['n10'],      4),
                    'trad_lat_ms':    round(trad_lat_ms,        4),
                    'n_random_seeds': N_RANDOM_SEEDS,
                }

                # Random pruning
                t0 = time.perf_counter()
                ratio_scores = random_prune_topk(
                    q_embed=q_embed, content_mask=content_mask,
                    doc_pad=doc_pad, doc_mask=doc_mask,
                    topk_ratios=TOPK_RATIOS, n_seeds=N_RANDOM_SEEDS,
                )
                if device == 'cuda': torch.cuda.synchronize()
                per_ratio_lat = (time.perf_counter() - t0) / len(TOPK_RATIOS) * 1000.0

                for r in TOPK_RATIOS:
                    key    = f"rand_r{int(r*100)}"
                    scores = ratio_scores[r]
                    top10  = torch.topk(scores, k=min(10, n_docs)).indices.cpu().tolist()
                    m      = hit_metrics(top10, gt_set)
                    record(key, m, domain)
                    random_latency_tracker.add_ratio(r, per_ratio_lat)
                    query_row.update({
                        f'{key}_hit@1':     m['r1'],
                        f'{key}_hit@5':     m['r5'],
                        f'{key}_hit@10':    m['r10'],
                        f'{key}_recall@1':  round(m['recall1'],  6),
                        f'{key}_recall@5':  round(m['recall5'],  6),
                        f'{key}_recall@10': round(m['recall10'], 6),
                        f'{key}_ndcg@10':   round(m['n10'],      4),
                        f'{key}_lat_ms':    round(per_ratio_lat, 4),
                    })

                batch_query_rows.append(query_row)

            gc.collect(); torch.cuda.empty_cache()
            print(f"      ✓ Q[{qb_start}:{qb_end}] done")

        del doc_pad, doc_mask
        gc.collect(); torch.cuda.empty_cache()
        all_query_results.extend(batch_query_rows)

        t_m  = all_metrics.get('traditional', _init_metric())
        t_cnt = t_m['count'] if t_m['count'] > 0 else 1
        print(f"  ✅ {len(batch_query_rows)} Q | "
              f"Baseline Hit@10={t_m['r10']/t_cnt*100:.1f}%  "
              f"Recall@10={t_m['recall10']/t_cnt*100:.1f}%  "
              f"nDCG@10={t_m['n10']/t_cnt:.4f}")

        all_batch_stats.append({
            'batch':          f"{START_IDX}-{END_IDX}",
            'n_layouts':      n_docs,
            'n_queries':      total,
            'trad_hit@10':    round(t_m['r10']     / t_cnt * 100, 2),
            'trad_recall@10': round(t_m['recall10'] / t_cnt * 100, 2),
            'trad_ndcg@10':   round(t_m['n10']     / t_cnt,       4),
            'status':         'ok',
        })

    except Exception as e:
        print(f"  ❌ Eval error: {e}"); import traceback; traceback.print_exc()
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}",
                                'status': 'error', 'error': str(e)})

    # ── Save & cleanup ─────────────────────────────────────────────────────────
    try:
        if batch_query_rows:
            pd.DataFrame(batch_query_rows).to_csv(
                os.path.join(WORKING_DIR, f"batch_{START_IDX}_{END_IDX}_queries.csv"), index=False)
        print("  ✅ Batch files saved")
    except Exception as e:
        print(f"  ❌ Save error: {e}")

    del fused_embeddings, batch_sample_layouts_df, batch_qa_pairs
    del batch_df_orig, batch_df_enh, batch_df_final
    gc.collect(); torch.cuda.empty_cache()
    print("  ✓ Cleaned\n")


# ==============================================================================
# FINAL SUMMARY
# ==============================================================================
print("=" * 80)
print("CONSOLIDATING RESULTS")
print("=" * 80)

try:
    if all_query_results:
        df_all_q = pd.DataFrame(all_query_results)
        df_all_q.to_csv(os.path.join(WORKING_DIR, "MASTER_all_batches_queries.csv"), index=False)
        print(f"✅ {len(df_all_q)} queries saved")

    SEP = "-" * 80
    print(f"\n{'Method':<18} {'Hit@1':>7} {'Hit@5':>7} {'Hit@10':>7} "
          f"{'Rec@1':>8} {'Rec@5':>8} {'Rec@10':>9} {'nDCG@10':>9}")
    print(SEP)
    for method in METHOD_RAND:
        if method not in all_metrics: continue
        m   = all_metrics[method]; cnt = m['count'] if m['count'] > 0 else 1
        print(f"{method:<18} "
              f"{m['r1']/cnt*100:6.2f}%  {m['r5']/cnt*100:6.2f}%  {m['r10']/cnt*100:6.2f}%  "
              f"{m['recall1']/cnt*100:7.2f}%  {m['recall5']/cnt*100:7.2f}%  "
              f"{m['recall10']/cnt*100:8.2f}%  {m['n10']/cnt:8.4f}")

    lat_report = random_latency_tracker.report()
    if not lat_report.empty:
        print("\nSCORING-ONLY LATENCY")
        print(lat_report.to_string(index=False))
        lat_report.to_csv(
            os.path.join(WORKING_DIR, "MASTER_random_latency_scoring_only.csv"), index=False)

    summary_rows = []
    for method in METHOD_RAND:
        if method not in all_metrics: continue
        m   = all_metrics[method]; cnt = m['count'] if m['count'] > 0 else 1
        summary_rows.append({
            'method':    method,
            'hit@1':     round(m['r1']      / cnt * 100, 4),
            'hit@5':     round(m['r5']      / cnt * 100, 4),
            'hit@10':    round(m['r10']     / cnt * 100, 4),
            'recall@1':  round(m['recall1'] / cnt * 100, 4),
            'recall@5':  round(m['recall5'] / cnt * 100, 4),
            'recall@10': round(m['recall10']/ cnt * 100, 4),
            'ndcg@1':    round(m['n1']      / cnt,       6),
            'ndcg@5':    round(m['n5']      / cnt,       6),
            'ndcg@10':   round(m['n10']     / cnt,       6),
        })
    pd.DataFrame(summary_rows).to_csv(
        os.path.join(WORKING_DIR, "MASTER_ablation_summary.csv"), index=False)

    if all_batch_stats:
        pd.DataFrame(all_batch_stats).to_csv(
            os.path.join(WORKING_DIR, "MASTER_batch_stats.csv"), index=False)

    print("\n✅ MASTER_ablation_summary.csv, MASTER_batch_stats.csv saved")

except Exception as e:
    print(f"❌ Final error: {e}"); import traceback; traceback.print_exc()


# ==============================================================================
# VISUALIZE
# ==============================================================================
print("\n>>> BƯỚC 5: Visualize")

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker

    metrics_to_plot = [
        ('n10',      'nDCG@10',    False),
        ('recall10', 'Recall@10',  True),
        ('n5',       'nDCG@5',     False),
        ('recall1',  'Recall@1',   True),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    axes = axes.flatten()

    for ax_idx, (mkey, title, pct_flag) in enumerate(metrics_to_plot):
        ax = axes[ax_idx]
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('Top-k ratio (fraction of tokens kept)', fontsize=10)
        ax.set_ylabel('Recall (%)' if pct_flag else 'Score', fontsize=10)
        ax.grid(axis='y', alpha=0.25); ax.grid(axis='x', alpha=0.15)
        ax.spines[['top', 'right']].set_visible(False)

        if 'traditional' in all_metrics:
            m   = all_metrics['traditional']; cnt = m['count'] if m['count'] > 0 else 1
            bv  = m[mkey] / cnt * (100 if pct_flag else 1)
            ax.axhline(bv, color='#888780', linewidth=1.8, linestyle='--',
                       label='Baseline (full ColSMoL)', zorder=2)

        vals = []
        for r in TOPK_RATIOS:
            key = f"rand_r{int(r*100)}"
            if key in all_metrics:
                m   = all_metrics[key]; cnt = m['count'] if m['count'] > 0 else 1
                v   = m[mkey] / cnt * (100 if pct_flag else 1)
                vals.append(v)
            else:
                vals.append(None)

        valid = [(r, v) for r, v in zip(TOPK_RATIOS, vals) if v is not None]
        if valid:
            xs, ys = zip(*valid)
            ax.plot(xs, ys, color='#9B59B6', linewidth=2.2, marker='o', markersize=5,
                    label=f'Random Pruning (avg {N_RANDOM_SEEDS} seeds)', zorder=3)

        ax.set_xticks(TOPK_RATIOS)
        ax.set_xticklabels([str(r) for r in TOPK_RATIOS], fontsize=8)
        fmt = (lambda v, _: f"{v:.1f}%") if pct_flag else (lambda v, _: f"{v:.3f}")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt))
        ax.legend(fontsize=8, framealpha=0.4, loc='lower right')

    plt.suptitle(
        f"Random Token Pruning vs Baseline  (seeds={N_RANDOM_SEEDS})\n"
        "Recall@K = |topK ∩ GT| / |GT|  per query",
        fontsize=11, y=1.02
    )
    plt.tight_layout()
    plot_path = os.path.join(WORKING_DIR, "random_pruning_topk_comparison.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"✅ Plot saved → {plot_path}")

except Exception as e:
    print(f"⚠️  Plot error: {e}")

print("\n>>> BƯỚC 3 v14-RandomPruning [FIXED] DONE")

>>> BƯỚC 3 v14-RandomPruning [FIXED]: Random Token Pruning
✅ Utility functions loaded

>>> BƯỚC 4: Process All Batches (Random Pruning)
Processing 13 batches | seeds=1

[1/13] Batch [0:25]
  ℹ️  PKL override: /kaggle/input/datasets/nguyenducdung1107/vvvvvvv/colqwen2_fused_index (2).pkl
  ✅ 6850 layouts
  Loading batch data...


    Reading JSONLs:   0%|          | 0/25 [00:00<?, ?it/s]

  ✅ 6850 layouts
  ✅ 122 QA pairs
  Evaluating 122 queries...
  ✅ doc_pad: torch.Size([6850, 1471, 128])


    Q[0:50]:   0%|          | 0/50 [00:00<?, ?it/s]

      ✓ Q[0:50] done


...

  ✅ 8399 layouts
  ✅ 59 QA pairs
  Evaluating 59 queries...
  ✅ doc_pad: torch.Size([8399, 1499, 128])


    Q[0:50]:   0%|          | 0/50 [00:00<?, ?it/s]

      ✓ Q[0:50] done


    Q[50:59]:   0%|          | 0/9 [00:00<?, ?it/s]

      ✓ Q[50:59] done
  ✅ 59 Q | Baseline Hit@10=70.3%  Recall@10=58.8%  nDCG@10=0.4871
  ✅ Batch files saved
  ✓ Cleaned

CONSOLIDATING RESULTS
✅ 1603 queries saved

Method               Hit@1   Hit@5  Hit@10    Rec@1    Rec@5    Rec@10   nDCG@10
--------------------------------------------------------------------------------
traditional         41.73%   64.07%   70.31%    31.13%    52.21%     58.85%    0.4871
rand_r10            23.27%   41.48%   48.03%    17.21%    32.46%     38.82%    0.2989
rand_r20            31.13%   52.53%   59.95%    23.51%    41.93%     49.09%    0.3883
rand_r30            34.25%   57.21%   63.07%    25.57%    46.26%     52.42%    0.4205
rand_r40            38.49%   60.45%   66.87%    29.26%    48.90%     55.55%    0.4539
rand_r50            39.80%   61.51%   67.44%    30.17%    49.60%     55.57%    0.4618
rand_r60            40.36%   62.69%   68.12%    30.20%    50.55%     56.39%    0.4687
rand_r70            41.61%   63.01%   68.81%    30.97%    51.22%     

**5.Kmeans**

In [16]:
# ==============================================================================
# BƯỚC 3 v12-KMeans – Spherical KMeans Token Pooling
# Method: gộp N token query thành K = max(1, int(N * topk_ratio)) cụm
#         bằng spherical KMeans (cosine similarity),
#         mean pool → K vectors, MaxSim → sum scores.
#
# Recall@K chuẩn: hits@K / |GT|  (per query, mean over queries)
# ==============================================================================

print(">>> BƯỚC 3 v12-KMeans [FIXED]")

import torch
import torch.nn.functional as F
import numpy as np
import time
import json, os, pickle, gc, glob
import pandas as pd
from tqdm.notebook import tqdm

# ==============================================================================
# CONFIG
# ==============================================================================
TOPK_RATIOS  = [round(i * 0.1, 1) for i in range(1, 10)]  # 0.1 … 0.9
KMEANS_ITERS = 10

WORKING_DIR      = "/kaggle/working"
ANNOTATIONS_PATH = "/kaggle/input/datasets/namthi/mmdocir-eval-data/MMDocIR_annotations.jsonl"
COLSMOL_DIR      = "/kaggle/input/datasets/nguyenducdung1107/colsmol500m-layoutmmdoc/colsmol500m-pkl"
QUERY_BATCH_SIZE = 50

BATCH_RANGE_PKL_OVERRIDE = {
    (0, 25): "/kaggle/input/datasets/nguyenducdung1107/vvvvvvv/colqwen2_fused_index (2).pkl",
}


# ==============================================================================
# UTILS
# ==============================================================================
def build_content_mask(inputs, processor):
    attn_mask = inputs["attention_mask"]
    input_ids = inputs.get("input_ids", None)
    if input_ids is None:
        return attn_mask.float()
    tok = getattr(processor, 'tokenizer', processor)
    special_ids = set()
    for attr in ['pad_token_id', 'bos_token_id', 'eos_token_id',
                 'unk_token_id', 'sep_token_id', 'cls_token_id']:
        tid = getattr(tok, attr, None)
        if tid is not None:
            special_ids.add(int(tid))
    if hasattr(tok, 'added_tokens_encoder'):
        for _, tid in tok.added_tokens_encoder.items():
            special_ids.add(int(tid))
    if not special_ids:
        return attn_mask.float()
    special_tensor = torch.tensor(list(special_ids), device=input_ids.device)
    is_special = (input_ids.unsqueeze(-1) == special_tensor).any(dim=-1)
    return attn_mask.float() * (~is_special).float()


# ==============================================================================
# SPHERICAL KMEANS
# ==============================================================================
def spherical_kmeans(X, K, n_iters=KMEANS_ITERS):
    N, D = X.shape
    K    = min(K, N)
    if K >= N:
        return X.clone()
    perm      = torch.randperm(N, device=X.device)
    centroids = X[perm[:K]].clone()
    for _ in range(n_iters):
        sim         = torch.mm(X, centroids.t())
        cluster_ids = sim.argmax(dim=1)
        new_centroids = torch.zeros_like(centroids)
        for c in range(K):
            members = (cluster_ids == c).nonzero(as_tuple=True)[0]
            new_centroids[c] = X[members].mean(dim=0) if members.numel() > 0 else centroids[c]
        centroids = F.normalize(new_centroids, dim=-1)
    return centroids


# ==============================================================================
# DOC MATRIX — handle dict và Tensor
# ==============================================================================
def _coerce_to_tensor(item):
    if isinstance(item, torch.Tensor):
        return item
    if isinstance(item, dict):
        for key in ('embedding', 'embeddings', 'vector', 'vectors', 'token_embeddings'):
            if key in item:
                v = item[key]
                if isinstance(v, torch.Tensor):
                    return v
        tensor_vals = [(k, v) for k, v in item.items() if isinstance(v, torch.Tensor)]
        if len(tensor_vals) == 1:
            return tensor_vals[0][1]
        if tensor_vals:
            return tensor_vals[0][1]
    raise ValueError(f"Cannot extract Tensor from item of type {type(item)}.")


def build_doc_matrix(docs_emb, device):
    tensors = []
    for i, item in enumerate(docs_emb):
        try:
            t = _coerce_to_tensor(item)
        except ValueError as e:
            raise ValueError(f"docs_emb[{i}]: {e}") from e
        if t.dim() == 1:
            t = t.unsqueeze(0)
        elif t.dim() > 2:
            t = t.squeeze(0)
        tensors.append(t)
    n     = len(tensors)
    max_l = max(t.shape[0] for t in tensors)
    D     = tensors[0].shape[1]
    doc_matrix = torch.zeros(n, max_l, D, device=device)
    doc_mask   = torch.zeros(n, max_l,    device=device, dtype=torch.bool)
    for i, t in enumerate(tensors):
        L = t.shape[0]
        doc_matrix[i, :L] = F.normalize(t.float().to(device), dim=-1)
        doc_mask[i, :L]   = True
    return doc_matrix, doc_mask


@torch.no_grad()
def fast_maxsim(q_norm, doc_matrix, doc_mask):
    """q_norm: (Q, D) → returns (Q, n_docs)"""
    sim = torch.einsum('qd,nld->qnl', q_norm, doc_matrix)
    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
    return sim.max(dim=-1).values


# ==============================================================================
# METRICS
# Recall@K = |topK ∩ GT| / |GT|  per query
# ==============================================================================
def compute_ndcg(ranked, gt_set, k):
    dcg  = sum(1.0 / np.log2(r + 2) for r, i in enumerate(ranked[:k]) if i in gt_set)
    idcg = sum(1.0 / np.log2(r + 2) for r in range(min(len(gt_set), k)))
    return dcg / idcg if idcg > 0 else 0.0

def first_hit(top_k, gt_set):
    for r, i in enumerate(top_k):
        if i in gt_set: return r + 1
    return -1

def hit_metrics(top_k_list, gt_set):
    """
    Recall@K = |topK ∩ GT| / |GT|  per query
    Hit@K    = 1 nếu ≥1 GT trong top-K
    """
    gt_size = max(len(gt_set), 1)
    h = first_hit(top_k_list, gt_set)

    hits1  = sum(1 for idx in top_k_list[:1]  if idx in gt_set)
    hits5  = sum(1 for idx in top_k_list[:5]  if idx in gt_set)
    hits10 = sum(1 for idx in top_k_list[:10] if idx in gt_set)

    return {
        'r1':       int(h != -1 and h <= 1),
        'r5':       int(h != -1 and h <= 5),
        'r10':      int(h != -1 and h <= 10),
        'recall1':  hits1  / gt_size,
        'recall5':  hits5  / gt_size,
        'recall10': hits10 / gt_size,
        'n1':       float(compute_ndcg(top_k_list, gt_set, 1)),
        'n5':       float(compute_ndcg(top_k_list, gt_set, 5)),
        'n10':      float(compute_ndcg(top_k_list, gt_set, 10)),
    }


# ==============================================================================
# METRIC STORE
# ==============================================================================
def _init_metric():
    return {'r1': 0, 'r5': 0, 'r10': 0,
            'n1': 0., 'n5': 0., 'n10': 0.,
            'recall1': 0., 'recall5': 0., 'recall10': 0.,
            'count': 0}

def _add_metric(dst, src):
    for f in ('r1', 'r5', 'r10'):
        dst[f] += int(src[f])
    for f in ('n1', 'n5', 'n10', 'recall1', 'recall5', 'recall10'):
        dst[f] += float(src[f])
    dst['count'] += 1

def _ensure(store, key):
    if key not in store: store[key] = _init_metric()
    return store[key]

def record(key, m, domain):
    _add_metric(_ensure(all_metrics, key), m)
    _add_metric(_ensure(all_domain_metrics[domain], key), m)


# ==============================================================================
# METHOD KEYS
# ==============================================================================
ABLATION_KEYS = ['traditional'] + [f"kmeans_r{int(r*100)}" for r in TOPK_RATIOS]


# ==============================================================================
# LATENCY TRACKER
# ==============================================================================
class LatencyTracker:
    def __init__(self):
        self.ratio_latency = {}

    def add_ratio(self, ratio, score_ms):
        ratio = float(ratio)
        if ratio not in self.ratio_latency:
            self.ratio_latency[ratio] = []
        self.ratio_latency[ratio].append(float(score_ms))

    def report(self):
        rows = []
        for ratio in sorted(self.ratio_latency.keys()):
            values = np.array(self.ratio_latency[ratio], dtype=np.float32)
            rows.append({
                'ratio':     ratio,
                'avg_ms':    float(values.mean())             if len(values) else float('nan'),
                'p50_ms':    float(np.percentile(values, 50)) if len(values) else float('nan'),
                'p95_ms':    float(np.percentile(values, 95)) if len(values) else float('nan'),
                'n_queries': int(len(values)),
            })
        return pd.DataFrame(rows)


kmeans_latency_tracker = LatencyTracker()


# ==============================================================================
# MASTER STORAGE
# ==============================================================================
all_query_results  = []
all_metrics        = {}
all_domain_metrics = {}
all_batch_stats    = []

print(f"topk_ratios : {TOPK_RATIOS}  →  K = max(1, int(N_tokens * ratio))")
print(f"kmeans_iters: {KMEANS_ITERS}")
print("=" * 70)

for batch_num, (START_IDX, END_IDX) in enumerate(BATCH_RANGES, 1):
    print(f"\n[{batch_num}/{len(BATCH_RANGES)}] Batch [{START_IDX}:{END_IDX}]")

    # ── Resolve index path ────────────────────────────────────────────────────
    batch_key  = (START_IDX, END_IDX)
    index_path = BATCH_RANGE_PKL_OVERRIDE.get(batch_key, None)
    if index_path is not None:
        print(f"  ℹ️  PKL override: {index_path}")
    else:
        for _pkl in pkl_files:
            try:
                _nums = os.path.basename(_pkl).replace('.pkl', '').split(' ')[-1]
                _s, _e = map(int, _nums.split('-'))
                if _s == START_IDX and _e == END_IDX:
                    index_path = _pkl; break
            except Exception: continue
        if index_path is None:
            _candidate = os.path.join(COLSMOL_DIR, f"{START_IDX}-{END_IDX}.pkl")
            if os.path.exists(_candidate):
                index_path = _candidate

    if index_path is None or not os.path.exists(index_path):
        print(f"  ⚠️  Cannot find PKL for [{START_IDX}:{END_IDX}]")
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}", 'status': 'index_not_found'})
        continue

    try:
        with open(index_path, 'rb') as f:
            saved = pickle.load(f)
        fused_embeddings = saved.get('embeddings', []) if isinstance(saved, dict) else \
                           (saved if isinstance(saved, list) else saved)
        print(f"  ✅ {len(fused_embeddings)} layouts")
    except Exception as e:
        print(f"  ❌ {e}"); continue

    # ── Load batch data ────────────────────────────────────────────────────────
    try:
        batch_doc_names    = intersection_docs[START_IDX:END_IDX]
        batch_target_files = [jsonl_map[d] for d in batch_doc_names]

        batch_df_orig = pd.read_parquet(PARQUET_PATH)
        batch_df_orig['join_doc_name'] = batch_df_orig['doc_name'].str.replace('.pdf', '', regex=False)
        batch_df_orig = batch_df_orig[batch_df_orig['join_doc_name'].isin(batch_doc_names)]

        dfs = []
        for f in tqdm(batch_target_files, desc="  Reading JSONLs", leave=False):
            try:
                temp = pd.read_json(f, lines=True)
                temp['join_doc_name'] = os.path.basename(f).replace('_layout.jsonl', '')
                if 'layout' in temp.columns:
                    temp = temp.rename(columns={'layout': 'layout_id'})
                cols = ['join_doc_name', 'layout_id', 'vlm_text', 'img_enhanced_path']
                if 'text_level' in temp.columns: cols.append('text_level')
                dfs.append(temp[[c for c in cols if c in temp.columns]])
            except Exception: pass

        batch_df_enh = pd.concat(dfs, ignore_index=True).rename(
            columns={'vlm_text': 'vlm_text_enhanced', 'text_level': 'text_level_enhanced'}
        ) if dfs else pd.DataFrame()

        df = pd.merge(batch_df_orig, batch_df_enh, on=['join_doc_name', 'layout_id'], how='left')
        df = df.sort_values(['join_doc_name', 'page_id', 'layout_id'])

        is_header = df['type'].isin(['title', 'section_header', 'header']) | \
                    df.get('text_level_enhanced', pd.Series(dtype=object)).notna()
        df['temp_header']     = df['text'].where(is_header)
        df['current_section'] = df.groupby('join_doc_name')['temp_header'] \
                                   .ffill().fillna("General Content")

        enh_image_map = {}
        if os.path.exists(ENHANCED_IMG_DIR):
            for fn in glob.glob(os.path.join(ENHANCED_IMG_DIR, "*")):
                enh_image_map[os.path.basename(fn)] = fn

        if 'img_enhanced_path' in df.columns:
            df['img_data'] = df['img_enhanced_path'].dropna().map(
                lambda p: enh_image_map.get(os.path.basename(str(p))))
            df['img_type'] = df['img_data'].notna().map(lambda x: 'path' if x else None)
        else:
            df['img_data'] = None; df['img_type'] = None

        if 'image_binary' in df.columns:
            m = df['img_type'].isna() & df['image_binary'].notna()
            df.loc[m, 'img_type'] = 'binary'
            df.loc[m, 'img_data'] = df.loc[m, 'image_binary']

        def _pick_text(row):
            for col in ['vlm_text_enhanced', 'text', 'ocr_text', 'vlm_text']:
                v = row.get(col)
                if pd.notna(v) and len(str(v)) > 5: return str(v)
            return "Document layout."

        df['final_text'] = "Section: " + df['current_section'].fillna('') + \
                           " \n Content: " + df.apply(_pick_text, axis=1)
        layouts_df = df.dropna(subset=['img_type']).reset_index(drop=True)
        print(f"  ✅ {len(layouts_df)} layouts")

    except Exception as e:
        print(f"  ❌ Batch data error: {e}"); import traceback; traceback.print_exc(); continue

    # ── Build QA pairs ─────────────────────────────────────────────────────────
    def calculate_iou(b1, b2):
        b1, b2 = list(b1), list(b2)
        xi, yi = max(b1[0], b2[0]), max(b1[1], b2[1])
        xa, ya = min(b1[2], b2[2]), min(b1[3], b2[3])
        inter  = max(0, xa - xi) * max(0, ya - yi)
        union  = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
        return inter / union if union > 0 else 0.0

    qa_pairs   = []
    doc_lookup = {k: v for k, v in layouts_df.groupby('join_doc_name')}
    avail_docs = set(doc_lookup.keys())

    with open(ANNOTATIONS_PATH, 'r') as f:
        for line in f:
            try: doc_data = json.loads(line)
            except Exception: continue
            target_doc = doc_data['doc_name'].replace('.pdf', '')
            if target_doc not in avail_docs: continue
            doc_layouts = doc_lookup[target_doc].copy()
            col_page    = 'page_idx' if 'page_idx' in doc_layouts.columns else 'page_id'
            doc_layouts['safe_page'] = pd.to_numeric(
                doc_layouts[col_page], errors='coerce').fillna(-999).astype(int)
            domain = doc_data.get('domain', 'General')
            for q_item in doc_data.get('questions', []):
                gt = []
                for target in q_item.get('layout_mapping', []):
                    try:
                        tp    = int(target['page'])
                        cands = pd.concat([doc_layouts[doc_layouts['safe_page'] == tp],
                                           doc_layouts[doc_layouts['safe_page'] == tp - 1]])
                        for idx, row in cands.iterrows():
                            if calculate_iou(row['bbox'], target['bbox']) > 0.5:
                                gt.append(int(idx))
                    except Exception: continue
                if gt:
                    qa_pairs.append({
                        'question':   q_item['Q'],
                        'gt_indices': list(set(gt)),
                        'doc_name':   target_doc,
                        'domain':     domain,
                    })

    print(f"  ✅ {len(qa_pairs)} QA pairs")
    if not qa_pairs:
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}", 'n_qa': 0, 'status': 'no_qa'})
        continue

    # ── Evaluate ───────────────────────────────────────────────────────────────
    try:
        device   = "cuda" if torch.cuda.is_available() else "cpu"
        n_docs   = len(fused_embeddings)
        doc_matrix, doc_mask = build_doc_matrix(fused_embeddings, device)
        print(f"  ✅ Doc matrix: {doc_matrix.shape}")

        batch_query_rows = []

        for qs in range(0, len(qa_pairs), QUERY_BATCH_SIZE):
            qe = min(qs + QUERY_BATCH_SIZE, len(qa_pairs))
            pbar = tqdm(enumerate(qa_pairs[qs:qe]), total=qe - qs,
                        desc=f"  Q[{qs}:{qe}]", leave=False)

            for qi, item in pbar:
                gq     = qs + qi
                gt_set = set(item['gt_indices'])
                domain = item['domain']
                if domain not in all_domain_metrics:
                    all_domain_metrics[domain] = {}

                with torch.no_grad():
                    q_inputs = processor.process_queries([item['question']]).to(device)
                    outputs  = model(**q_inputs)
                    if isinstance(outputs, torch.Tensor):
                        proj = outputs[0].float()
                    elif hasattr(outputs, 'last_hidden_state'):
                        proj = outputs.last_hidden_state[0].float()
                    else:
                        proj = outputs[0][0].float()
                    proj = proj / (proj.norm(dim=-1, keepdim=True) + 1e-8)

                content_mask_1d = build_content_mask(q_inputs, processor)[0].float()
                valid_idx = torch.where(content_mask_1d > 0)[0]
                if valid_idx.numel() == 0:
                    valid_idx = torch.where(q_inputs['attention_mask'][0] > 0)[0]

                q_norm = F.normalize(proj[valid_idx], dim=-1)
                N_tok  = q_norm.shape[0]

                # Baseline: traditional MaxSim
                t0 = time.perf_counter()
                M_trad      = fast_maxsim(q_norm, doc_matrix, doc_mask)
                trad_scores = M_trad.sum(0)
                if device == 'cuda': torch.cuda.synchronize()
                trad_lat_ms = (time.perf_counter() - t0) * 1000.0
                trad_top10  = torch.topk(trad_scores, min(10, n_docs)).indices.cpu().tolist()
                m_trad      = hit_metrics(trad_top10, gt_set)
                record('traditional', m_trad, domain)
                kmeans_latency_tracker.add_ratio(1.0, trad_lat_ms)

                query_row = {
                    'batch':          f"{START_IDX}-{END_IDX}",
                    'query_id':       gq,
                    'doc_name':       item['doc_name'],
                    'domain':         domain,
                    'question':       item['question'],
                    'gt_count':       len(gt_set),
                    'trad_hit@1':     m_trad['r1'],
                    'trad_hit@5':     m_trad['r5'],
                    'trad_hit@10':    m_trad['r10'],
                    'trad_recall@1':  round(m_trad['recall1'],  6),
                    'trad_recall@5':  round(m_trad['recall5'],  6),
                    'trad_recall@10': round(m_trad['recall10'], 6),
                    'trad_ndcg@10':   round(m_trad['n10'],      4),
                    'trad_lat_ms':    round(trad_lat_ms,        4),
                }

                # KMeans pool
                for r in TOPK_RATIOS:
                    K   = max(1, int(N_tok * r))
                    key = f"kmeans_r{int(r*100)}"

                    t0 = time.perf_counter()
                    centroids = spherical_kmeans(q_norm, K)
                    M_k       = fast_maxsim(centroids, doc_matrix, doc_mask)
                    scores    = M_k.sum(0)
                    if device == 'cuda': torch.cuda.synchronize()
                    lat_ms = (time.perf_counter() - t0) * 1000.0

                    top10 = torch.topk(scores, min(10, n_docs)).indices.cpu().tolist()
                    m     = hit_metrics(top10, gt_set)
                    record(key, m, domain)
                    kmeans_latency_tracker.add_ratio(r, lat_ms)

                    query_row.update({
                        f'{key}_hit@1':     m['r1'],
                        f'{key}_hit@5':     m['r5'],
                        f'{key}_hit@10':    m['r10'],
                        f'{key}_recall@1':  round(m['recall1'],  6),
                        f'{key}_recall@5':  round(m['recall5'],  6),
                        f'{key}_recall@10': round(m['recall10'], 6),
                        f'{key}_ndcg@10':   round(m['n10'],      4),
                        f'{key}_lat_ms':    round(lat_ms,        4),
                    })

                batch_query_rows.append(query_row)

            gc.collect(); torch.cuda.empty_cache()
            print(f"    ✓ Q[{qs}:{qe}] done")

        all_query_results.extend(batch_query_rows)

        t_m   = all_metrics.get('traditional', _init_metric())
        t_cnt = t_m['count'] if t_m['count'] > 0 else 1
        print(f"  ✅ {len(batch_query_rows)} queries — "
              f"Baseline Hit@10={t_m['r10']/t_cnt*100:.1f}%  "
              f"Recall@10={t_m['recall10']/t_cnt*100:.1f}%  "
              f"nDCG@10={t_m['n10']/t_cnt:.4f}")

        all_batch_stats.append({
            'batch':          f"{START_IDX}-{END_IDX}",
            'n_layouts':      n_docs,
            'n_queries':      len(qa_pairs),
            'trad_hit@10':    round(t_m['r10']     / t_cnt * 100, 2),
            'trad_recall@10': round(t_m['recall10'] / t_cnt * 100, 2),
            'trad_ndcg@10':   round(t_m['n10']     / t_cnt,       4),
            'status':         'ok',
        })

    except Exception as e:
        print(f"  ❌ Eval error: {e}"); import traceback; traceback.print_exc()
        all_batch_stats.append({'batch': f"{START_IDX}-{END_IDX}",
                                'status': 'error', 'error': str(e)})

    # ── Save & cleanup ─────────────────────────────────────────────────────────
    try:
        if batch_query_rows:
            pd.DataFrame(batch_query_rows).to_csv(
                os.path.join(WORKING_DIR, f"batch_{START_IDX}_{END_IDX}_queries.csv"), index=False)
        print("  ✅ Batch CSV saved")
    except Exception as e:
        print(f"  ❌ Save error: {e}")

    del doc_matrix, doc_mask, fused_embeddings, layouts_df, qa_pairs, df
    del batch_df_orig, batch_df_enh
    gc.collect(); torch.cuda.empty_cache()
    print("  ✓ Cleaned\n")


# ==============================================================================
# FINAL SUMMARY
# ==============================================================================
print("=" * 80)
print("CONSOLIDATING")
print("=" * 80)

try:
    if all_query_results:
        df_q = pd.DataFrame(all_query_results)
        df_q.to_csv(os.path.join(WORKING_DIR, "MASTER_queries.csv"), index=False)
        print(f"✅ {len(df_q)} queries → MASTER_queries.csv")

    SEP = "-" * 85
    print(f"\n{'Method':<25} {'Hit@1':>7} {'Hit@5':>7} {'Hit@10':>7} "
          f"{'Rec@1':>8} {'Rec@5':>8} {'Rec@10':>9} {'nDCG@10':>9}")
    print(SEP)
    for key in ABLATION_KEYS:
        if key not in all_metrics: continue
        m = all_metrics[key]; cnt = m['count'] if m['count'] > 0 else 1
        print(f"{key:<25} "
              f"{m['r1']/cnt*100:6.2f}%  {m['r5']/cnt*100:6.2f}%  {m['r10']/cnt*100:6.2f}%  "
              f"{m['recall1']/cnt*100:7.2f}%  {m['recall5']/cnt*100:7.2f}%  "
              f"{m['recall10']/cnt*100:8.2f}%  {m['n10']/cnt:8.4f}")

    print("\n" + "=" * 80)
    print("PER-DOMAIN — traditional vs kmeans per ratio (nDCG@10)")
    ratio_keys = [f"kmeans_r{int(r*100)}" for r in TOPK_RATIOS]
    hdr = " | ".join(f"r{int(r*100):>3}" for r in TOPK_RATIOS)
    print(f"{'Domain':<22} | {'trad':>6} | {hdr}")
    print("-" * (22 + 3 + 8 + 3 + len(TOPK_RATIOS) * 8))
    for domain in sorted(all_domain_metrics):
        dm  = all_domain_metrics[domain]
        t_m = dm.get('traditional', _init_metric()); tc = t_m['count'] if t_m['count'] > 0 else 1
        vals = []
        for key in ratio_keys:
            m_ = dm.get(key, _init_metric()); c_ = m_['count'] if m_['count'] > 0 else 1
            vals.append(f"{m_['n10']/c_:6.4f}")
        print(f"{domain:<22} | {t_m['n10']/tc:6.4f} | {' | '.join(vals)}")

    summary_rows = []
    for key in ABLATION_KEYS:
        if key not in all_metrics: continue
        m = all_metrics[key]; cnt = m['count'] if m['count'] > 0 else 1
        summary_rows.append({
            'method':    key,
            'hit@1':     round(m['r1']      / cnt * 100, 4),
            'hit@5':     round(m['r5']      / cnt * 100, 4),
            'hit@10':    round(m['r10']     / cnt * 100, 4),
            'recall@1':  round(m['recall1'] / cnt * 100, 4),
            'recall@5':  round(m['recall5'] / cnt * 100, 4),
            'recall@10': round(m['recall10']/ cnt * 100, 4),
            'ndcg@1':    round(m['n1']      / cnt,       6),
            'ndcg@5':    round(m['n5']      / cnt,       6),
            'ndcg@10':   round(m['n10']     / cnt,       6),
        })
    pd.DataFrame(summary_rows).to_csv(
        os.path.join(WORKING_DIR, "MASTER_ablation_summary.csv"), index=False)

    domain_rows = []
    for domain in sorted(all_domain_metrics):
        dm  = all_domain_metrics[domain]
        row = {'domain': domain}
        for key in ABLATION_KEYS:
            m_   = dm.get(key, _init_metric()); cnt_ = m_['count'] if m_['count'] > 0 else 1
            row[f'{key}_ndcg@10']   = round(m_['n10']     / cnt_,       6)
            row[f'{key}_recall@10'] = round(m_['recall10'] / cnt_ * 100, 4)
            row[f'{key}_hit@10']    = round(m_['r10']     / cnt_ * 100, 4)
        domain_rows.append(row)
    pd.DataFrame(domain_rows).to_csv(
        os.path.join(WORKING_DIR, "MASTER_domain_summary.csv"), index=False)

    lat_report = kmeans_latency_tracker.report()
    if not lat_report.empty:
        print("\nSCORING-ONLY LATENCY (KMeans)")
        print(lat_report.to_string(index=False))
        lat_report.to_csv(
            os.path.join(WORKING_DIR, "MASTER_kmeans_latency_scoring_only.csv"), index=False)

    pd.DataFrame(all_batch_stats).to_csv(
        os.path.join(WORKING_DIR, "MASTER_batch_stats.csv"), index=False)

    print("\n✅ MASTER_ablation_summary.csv")
    print("✅ MASTER_domain_summary.csv")
    print("✅ MASTER_batch_stats.csv")

except Exception as e:
    print(f"❌ Summary error: {e}"); import traceback; traceback.print_exc()


# ==============================================================================
# VISUALIZE
# ==============================================================================
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker

    metrics_to_plot = [
        ('n10',      'nDCG@10',    False),
        ('recall10', 'Recall@10',  True),
        ('n5',       'nDCG@5',     False),
        ('recall1',  'Recall@1',   True),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    axes = axes.flatten()

    for ax_idx, (mkey, title, pct_flag) in enumerate(metrics_to_plot):
        ax = axes[ax_idx]
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('Top-k ratio', fontsize=10)
        ax.set_ylabel('Recall (%)' if pct_flag else 'Score', fontsize=10)
        ax.grid(axis='y', alpha=0.25); ax.grid(axis='x', alpha=0.15)
        ax.spines[['top', 'right']].set_visible(False)

        if 'traditional' in all_metrics:
            m   = all_metrics['traditional']; cnt = m['count'] if m['count'] > 0 else 1
            bv  = m[mkey] / cnt * (100 if pct_flag else 1)
            ax.axhline(bv, color='#888780', linewidth=1.8, linestyle='--',
                       label='Baseline (full ColSMoL)', zorder=2)

        vals = []
        for r in TOPK_RATIOS:
            key = f"kmeans_r{int(r*100)}"
            if key in all_metrics:
                m   = all_metrics[key]; cnt = m['count'] if m['count'] > 0 else 1
                vals.append(m[mkey] / cnt * (100 if pct_flag else 1))
            else:
                vals.append(None)

        valid = [(r, v) for r, v in zip(TOPK_RATIOS, vals) if v is not None]
        if valid:
            xs, ys = zip(*valid)
            ax.plot(xs, ys, color='#E07B39', linewidth=2.2, marker='o', markersize=5,
                    label='Spherical KMeans Pooling', zorder=3)

        ax.set_xticks(TOPK_RATIOS)
        ax.set_xticklabels([str(r) for r in TOPK_RATIOS], fontsize=8)
        fmt = (lambda v, _: f"{v:.1f}%") if pct_flag else (lambda v, _: f"{v:.3f}")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt))
        ax.legend(fontsize=8, framealpha=0.4, loc='lower right')

    plt.suptitle(
        "Token Pooling with Spherical KMeans vs Baseline\n"
        f"K = max(1, int(N × ratio))  |  iters={KMEANS_ITERS}  |  "
        "Recall@K = hits@K / |GT| per query",
        fontsize=11, y=1.02
    )
    plt.tight_layout()
    plot_path = os.path.join(WORKING_DIR, "kmeans_pooling_comparison.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"✅ Plot saved → {plot_path}")

except Exception as e:
    print(f"⚠️  Plot error: {e}")

print("\n>>> BƯỚC 3 v12-KMeans [FIXED] DONE")

>>> BƯỚC 3 v12-KMeans [FIXED]
topk_ratios : [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]  →  K = max(1, int(N_tokens * ratio))
kmeans_iters: 10

[1/13] Batch [0:25]
  ℹ️  PKL override: /kaggle/input/datasets/nguyenducdung1107/vvvvvvv/colqwen2_fused_index (2).pkl
  ✅ 6850 layouts


  Reading JSONLs:   0%|          | 0/25 [00:00<?, ?it/s]

  ✅ 6850 layouts
  ✅ 122 QA pairs
  ✅ Doc matrix: torch.Size([6850, 1471, 128])


  Q[0:50]:   0%|          | 0/50 [00:00<?, ?it/s]

    ✓ Q[0:50] done


...

  ✅ 8399 layouts
  ✅ 59 QA pairs
  ✅ Doc matrix: torch.Size([8399, 1499, 128])


  Q[0:50]:   0%|          | 0/50 [00:00<?, ?it/s]

    ✓ Q[0:50] done


  Q[50:59]:   0%|          | 0/9 [00:00<?, ?it/s]

    ✓ Q[50:59] done
  ✅ 59 queries — Baseline Hit@10=70.3%  Recall@10=58.8%  nDCG@10=0.4873
  ✅ Batch CSV saved
  ✓ Cleaned

CONSOLIDATING
✅ 1603 queries → MASTER_queries.csv

Method                      Hit@1   Hit@5  Hit@10    Rec@1    Rec@5    Rec@10   nDCG@10
-------------------------------------------------------------------------------------
traditional                41.80%   64.07%   70.31%    31.17%    52.21%     58.85%    0.4873
kmeans_r10                 29.13%   51.90%   60.95%    21.44%    41.78%     49.65%    0.3791
kmeans_r20                 35.25%   58.76%   66.00%    26.75%    47.02%     54.47%    0.4342
kmeans_r30                 38.43%   60.14%   67.75%    28.64%    48.47%     56.11%    0.4548
kmeans_r40                 40.05%   61.07%   68.43%    30.24%    49.47%     56.61%    0.4660
kmeans_r50                 40.99%   61.82%   69.18%    31.17%    49.92%     57.32%    0.4747
kmeans_r60                 41.24%   62.57%   69.06%    31.08%    50.83%     57.49%    0.4785